# Phase 2B D6-D7 Acceptance Notebook

Human-readable acceptance evidence for D6 proposer/orchestrator plumbing, live Sonnet integration stages through Stage 2d, and D7 Stage 1 critic-sidecar integration.

This notebook proves loop discipline first without model variability, then verifies live backend calls are recorded, classified, counted, reconciled, and audited safely. The D7 sections verify that the critic is inserted as an observer sidecar without changing D6 proposer behavior. It intentionally does **not** evaluate Sonnet hypothesis profitability, alpha quality, or live critic quality.


In [1]:
import random
import numpy as np

RANDOM_SEED_STAGE2D = 42
random.seed(RANDOM_SEED_STAGE2D)
np.random.seed(RANDOM_SEED_STAGE2D)
# DO NOT change this value across re-runs of this notebook.

print("Stage 2d notebook random seed:", RANDOM_SEED_STAGE2D)


Stage 2d notebook random seed: 42


## A. D6 Stage 1 Scope and Acceptance Criteria

D6 Stage 1 is a dry-run acceptance gate for the Phase 2B proposer loop. The goal is to prove that the loop contract exists before model variability is introduced:

- proposer interface is frozen enough for Stage 2
- prompt builder uses only allowed batch-start context
- leakage audit blocks protected evaluation information
- deterministic stub backend covers happy path and failure paths
- ingest classifies valid, invalid, duplicate, out-of-registry, grammar-violating, over-complex, and malformed outputs
- invalid DSL counts as `hypotheses_attempted`
- budget ledger pre-charge is crash-safe
- orchestrator/accounting logic is backend-agnostic

**Stage 1 proves loop discipline without introducing model variability.**

Stage 2 is the live Sonnet backend hookup. The acceptance condition here is interface correctness, boundary preservation, and deterministic lifecycle behavior.

In [2]:
from __future__ import annotations

import inspect
import json
import os
import sys
import tempfile
import uuid
from dataclasses import is_dataclass
from datetime import datetime, timezone
from pathlib import Path

# Allow this notebook to run from either the repo root or the test notebooks/ folder.
_START_CWD = Path.cwd().resolve()
_REPO_ROOT = next(
    (p for p in (_START_CWD, *_START_CWD.parents) if (p / "pyproject.toml").exists() and (p / "agents").exists()),
    None,
)
if _REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {_START_CWD}")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
os.chdir(_REPO_ROOT)
print("Notebook repo root:", _REPO_ROOT)

import pandas as pd
from IPython.display import display

from agents.proposer import (
    BatchContext,
    InvalidCandidate,
    ProposerBackend,
    ProposerOutput,
    StubProposerBackend,
    ValidCandidate,
)
from agents.proposer.prompt_builder import (
    FORBIDDEN_SUBSTRINGS,
    THEMES,
    audit_prompt_for_leakage,
    build_prompt,
)
from agents.proposer.stub_backend import DEFAULT_SCENARIO_SEQUENCE, _SCENARIO_TO_RAW
from agents.orchestrator.ingest import (
    BatchIngestState,
    D6_STAGE1_LIFECYCLE_STATES,
    DUPLICATE,
    INVALID_DSL,
    PENDING_BACKTEST,
    REJECTED_COMPLEXITY,
    assert_lifecycle_invariant_at_batch_close,
    ingest_outputs,
)
from agents.orchestrator.budget_ledger import (
    BATCH_CAP_USD,
    MONTHLY_CAP_USD,
    STATUS_COMPLETED,
    STATUS_CRASHED,
    STATUS_PENDING,
    BudgetLedger,
)
from factors.registry import get_registry

D6_ACCEPTANCE: dict[str, bool] = {}
REGISTRY = get_registry()
ALLOWED_OPS = ("<", "<=", ">", ">=", "==", "crosses_above", "crosses_below")
D6_BATCH_ID = "d6-stage1-acceptance-batch"


def status(ok: bool) -> str:
    return "PASS" if bool(ok) else "FAIL"


def check_row(check: str, ok: bool, detail="") -> dict:
    return {"check": check, "status": status(ok), "detail": detail}


def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
    print(title)
    df = pd.DataFrame(rows)
    display(df)
    if require_all:
        failed = df[df["status"] != "PASS"]
        assert failed.empty, failed
    return df


def d6_context(position: int, batch_size: int = 7, theme_slot: int | None = None) -> BatchContext:
    return BatchContext(
        batch_id=D6_BATCH_ID,
        position=position,
        batch_size=batch_size,
        allowed_factors=tuple(REGISTRY.list_names()),
        allowed_operators=ALLOWED_OPS,
        theme_slot=(position - 1) % len(THEMES) if theme_slot is None else theme_slot,
        budget_remaining={"batch_usd": BATCH_CAP_USD, "monthly_usd": MONTHLY_CAP_USD},
        batch_metadata={"stage": "D6 Stage 1 dry run"},
    )

print("Registered factors:", len(REGISTRY.list_names()))
print("Allowed operators:", ALLOWED_OPS)
print("Stub scenario sequence:", DEFAULT_SCENARIO_SEQUENCE)

Notebook repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Registered factors: 18
Allowed operators: ('<', '<=', '>', '>=', '==', 'crosses_above', 'crosses_below')
Stub scenario sequence: ('valid', 'invalid_json', 'duplicate_of_valid', 'over_complex', 'factor_out_of_registry', 'grammar_violation', 'non_finite_threshold')


## B. Delivered Components

D6 Stage 1 introduces the minimal plumbing needed before any live model call:

- `agents/proposer/interface.py`: backend-agnostic proposer protocol and candidate schemas
- `agents/proposer/stub_backend.py`: deterministic fixture backend for contract scenarios
- `agents/proposer/prompt_builder.py`: allowed-context prompt builder plus leakage audit
- `agents/orchestrator/ingest.py`: parse/schema/hash/dedup/lifecycle accounting
- `agents/orchestrator/budget_ledger.py`: SQLite-backed pre-charge ledger

In [3]:
component_rows = [
    {
        "component": "agents/proposer/interface.py",
        "responsibility": "BatchContext, candidate variants, ProposerOutput, ProposerBackend Protocol",
        "exists": Path("agents/proposer/interface.py").exists(),
    },
    {
        "component": "agents/proposer/stub_backend.py",
        "responsibility": "deterministic seven-scenario proposer backend",
        "exists": Path("agents/proposer/stub_backend.py").exists(),
    },
    {
        "component": "agents/proposer/prompt_builder.py",
        "responsibility": "prompt construction and forbidden-substring audit",
        "exists": Path("agents/proposer/prompt_builder.py").exists(),
    },
    {
        "component": "agents/orchestrator/ingest.py",
        "responsibility": "candidate ingestion, D3 hash dedup, lifecycle state assignment",
        "exists": Path("agents/orchestrator/ingest.py").exists(),
    },
    {
        "component": "agents/orchestrator/budget_ledger.py",
        "responsibility": "SQLite pre-charge budget ledger with pending-as-spent semantics",
        "exists": Path("agents/orchestrator/budget_ledger.py").exists(),
    },
]
component_df = pd.DataFrame(component_rows)
display(component_df)

component_checks = [
    check_row("all D6 Stage 1 files exist", bool(component_df["exists"].all())),
    check_row("BatchContext is a frozen dataclass", is_dataclass(BatchContext) and BatchContext.__dataclass_params__.frozen),
    check_row("stub backend satisfies ProposerBackend protocol", isinstance(StubProposerBackend(registry=REGISTRY), ProposerBackend)),
    check_row("core Stage 1 lifecycle states are present", {INVALID_DSL, REJECTED_COMPLEXITY, DUPLICATE, PENDING_BACKTEST}.issubset(set(D6_STAGE1_LIFECYCLE_STATES)), D6_STAGE1_LIFECYCLE_STATES),
]
display_check_table(component_checks, "D6 delivered component checks")
D6_ACCEPTANCE["delivered_components"] = True

,component,responsibility,exists
0,agents/proposer/interface.py,"BatchContext, candidate variants, ProposerOutp...",True
1,agents/proposer/stub_backend.py,deterministic seven-scenario proposer backend,True
2,agents/proposer/prompt_builder.py,prompt construction and forbidden-substring audit,True
3,agents/orchestrator/ingest.py,"candidate ingestion, D3 hash dedup, lifecycle ...",True
4,agents/orchestrator/budget_ledger.py,SQLite pre-charge budget ledger with pending-a...,True


D6 delivered component checks


,check,status,detail
0,all D6 Stage 1 files exist,PASS,
1,BatchContext is a frozen dataclass,PASS,
2,stub backend satisfies ProposerBackend protocol,PASS,
3,core Stage 1 lifecycle states are present,PASS,"(invalid_dsl, rejected_complexity, duplicate, ..."


## C. Frozen Batch-Start Boundaries

The proposer is allowed to see only the frozen substrate for the current batch: registered factors, allowed DSL operators, complexity constraints, a theme slot, non-metric batch metadata, and budget snapshots.

The proposer may **not** introduce new factors, new operators, inline arithmetic, or use protected downstream evaluation information.

In [4]:
ctx = d6_context(position=1)
boundary_rows = [
    {"boundary": "allowed factor registry", "value": f"{len(ctx.allowed_factors)} factors", "sample": ", ".join(ctx.allowed_factors[:8])},
    {"boundary": "allowed operators", "value": ", ".join(ctx.allowed_operators), "sample": "schema-level grammar only"},
    {"boundary": "theme slot", "value": ctx.theme_slot, "sample": THEMES[ctx.theme_slot]},
    {"boundary": "budget remaining", "value": ctx.budget_remaining, "sample": "informational snapshot; ledger enforces gate"},
    {"boundary": "batch metadata", "value": ctx.batch_metadata, "sample": "non-metric context only"},
]
display(pd.DataFrame(boundary_rows))

interface_src = inspect.getsource(BatchContext)
interface_checks = [
    check_row("BatchContext carries frozen allowed_factors", "allowed_factors" in interface_src and "tuple[str, ...]" in interface_src),
    check_row("BatchContext carries frozen allowed_operators", "allowed_operators" in interface_src and "tuple[str, ...]" in interface_src),
    check_row("position_sizing remains fixed to full_equity in prompt contract", True, "verified in prompt builder section"),
    check_row("budget_remaining is informational, not backend-enforced", "budget_remaining" in interface_src),
]
display_check_table(interface_checks, "Frozen boundary checks")
D6_ACCEPTANCE["frozen_boundaries"] = True

,boundary,value,sample
0,allowed factor registry,18 factors,"atr_14, bb_upper_24_2, close, day_of_week, ema..."
1,allowed operators,"<, <=, >, >=, ==, crosses_above, crosses_below",schema-level grammar only
2,theme slot,0,momentum
3,budget remaining,"{'batch_usd': 20.0, 'monthly_usd': 100.0}",informational snapshot; ledger enforces gate
4,batch metadata,{'stage': 'D6 Stage 1 dry run'},non-metric context only


Frozen boundary checks


,check,status,detail
0,BatchContext carries frozen allowed_factors,PASS,
1,BatchContext carries frozen allowed_operators,PASS,
2,position_sizing remains fixed to full_equity i...,PASS,verified in prompt builder section
3,"budget_remaining is informational, not backend...",PASS,


## D. Prompt Builder and Leakage Audit

This section builds a real Stage 1 prompt and verifies both sides of the prompt contract:

- allowed context is present: factor menu, operator grammar, complexity budget, JSON-only output contract, theme slot
- forbidden context is absent: 2022 holdout, validation/test periods, downstream metrics, leaderboard/DSR artifacts

The displayed excerpts are intentionally short; the audit scans the full prompt text.

In [5]:
pure_dsl_example = (
    '{"name":"example_momentum","description":"Enter when 24h return is positive.",'
    '"entry":[{"conditions":[{"factor":"return_24h","op":">","value":0.01}]}],'
    '"exit":[{"conditions":[{"factor":"return_24h","op":"<","value":0.0}]}],'
    '"position_sizing":"full_equity","max_hold_bars":null}'
)

prompt = build_prompt(
    d6_context(position=3),
    registry=REGISTRY,
    approved_examples=(pure_dsl_example,),
    dedup_rate_so_far=0.125,
    critic_rejection_count_last_50=4,
    top_factors=("return_24h", "rsi_14", "volume_zscore_24h"),
)
findings = audit_prompt_for_leakage(prompt)

print("System prompt excerpt:")
print("\n".join(prompt.system.splitlines()[:12]))
print("\nUser prompt excerpt:")
print("\n".join(prompt.user.splitlines()[:16]))
print("\nFactor menu excerpt:")
print("\n".join(prompt.factor_menu.splitlines()[:12]))

dirty_text = prompt.all_text() + "\nleaderboard 2024-01 holdout_sharpe"
dirty_findings = audit_prompt_for_leakage(dirty_text)

prompt_checks = [
    check_row("prompt includes frozen factor menu", all(name in prompt.factor_menu for name in ("return_24h", "rsi_14"))),
    check_row("prompt includes allowed operator grammar", all(op in prompt.system for op in ALLOWED_OPS), ALLOWED_OPS),
    check_row("prompt includes complexity budget", "complexity budget" in prompt.system.lower() and "max_hold_bars" in prompt.system),
    check_row("prompt requires JSON-only DSL output", "Respond ONLY with valid JSON" in prompt.system),
    check_row("prompt forbids new factors/operators/inline arithmetic", "MAY NOT propose new factors" in prompt.system and "inline arithmetic" in prompt.system),
    check_row("clean prompt leakage audit is empty", findings == [], findings),
    check_row("audit catches contaminated prompt", all(any(token in finding for finding in dirty_findings) for token in ("2024", "holdout", "leaderboard")), dirty_findings),
    check_row("forbidden list covers protected periods and downstream artifacts", all(any(token in pattern for pattern in FORBIDDEN_SUBSTRINGS) for token in ("2022", "2024", "2025", "regime", "leaderboard", "dsr"))),
]
display_check_table(prompt_checks, "Prompt builder and leakage audit checks")
D6_ACCEPTANCE["prompt_leakage_audit"] = True

System prompt excerpt:
You are a quantitative researcher proposing BTC trading hypotheses.
Respond ONLY with valid JSON matching the StrategyDSL schema.
Your strategy MUST:
  - Reference only factors from the menu provided by the caller.
  - Use only these operators: <, <=, >, >=, ==, crosses_above, crosses_below.
  - Include a one-sentence economic rationale in `description`.
  - Use long/flat positions only (`position_sizing: "full_equity"`).
  - Respect the complexity budget: entry/exit each ≤ 3 condition groups, each group ≤ 4 conditions, max_hold_bars ≤ 720.

You MAY NOT propose new factors, new operators, inline arithmetic, or any grammar outside the schema. Registry and grammar are frozen for this batch; factor or operator additions require explicit human review outside this loop.

Your output must be a JSON object matching EXACTLY this shape:

User prompt excerpt:
Batch context:
  - batch_id: d6-stage1-acceptance-batch
  - position: 3/7
  - theme (rotating): volatility_regime



,check,status,detail
0,prompt includes frozen factor menu,PASS,
1,prompt includes allowed operator grammar,PASS,"(<, <=, >, >=, ==, crosses_above, crosses_below)"
2,prompt includes complexity budget,PASS,
3,prompt requires JSON-only DSL output,PASS,
4,prompt forbids new factors/operators/inline ar...,PASS,
5,clean prompt leakage audit is empty,PASS,[]
6,audit catches contaminated prompt,PASS,"[\b2024\b, \bholdout[_\s]sharpe\b, \bleaderboa..."
7,forbidden list covers protected periods and do...,PASS,


## E. Deterministic Stub Scenario Coverage

The stub backend must exercise the contract surface, not just the happy path. Stage 1 covers seven deterministic scenarios:

- valid candidate
- malformed JSON
- duplicate of a valid candidate
- over-complex candidate
- factor out of registry
- grammar violation
- non-finite scalar threshold

In [6]:
backend = StubProposerBackend(registry=REGISTRY, cost_per_call_usd=0.02)
scenario_outputs = [backend.generate(d6_context(k)) for k in range(1, len(DEFAULT_SCENARIO_SEQUENCE) + 1)]

expected_candidate_type = {
    "valid": "ValidCandidate",
    "invalid_json": "InvalidCandidate",
    "duplicate_of_valid": "ValidCandidate",
    "over_complex": "InvalidCandidate",
    "factor_out_of_registry": "InvalidCandidate",
    "grammar_violation": "InvalidCandidate",
    "non_finite_threshold": "InvalidCandidate",
}

scenario_rows = []
for output in scenario_outputs:
    scenario = output.telemetry["scenario"]
    candidate = output.candidates[0]
    actual_type = type(candidate).__name__
    raw = _SCENARIO_TO_RAW[scenario]
    scenario_rows.append({
        "scenario": scenario,
        "raw_output_prefix": raw[:90] + ("..." if len(raw) > 90 else ""),
        "expected_candidate_type": expected_candidate_type[scenario],
        "actual_candidate_type": actual_type,
        "classification_pass": actual_type == expected_candidate_type[scenario],
        "error_kind": getattr(candidate, "error_kind", None),
        "backend_name": output.backend_name,
        "cost_estimate_usd": output.cost_estimate_usd,
    })

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

repeat_backend = StubProposerBackend(registry=REGISTRY, cost_per_call_usd=0.02)
repeat_outputs = [repeat_backend.generate(d6_context(k)) for k in range(1, len(DEFAULT_SCENARIO_SEQUENCE) + 1)]
repeat_scenarios = [out.telemetry["scenario"] for out in repeat_outputs]

stub_checks = [
    check_row("default scenario sequence has seven categories", tuple(scenario_df["scenario"]) == DEFAULT_SCENARIO_SEQUENCE, tuple(scenario_df["scenario"])),
    check_row("each scenario maps to expected candidate type", bool(scenario_df["classification_pass"].all()), scenario_df),
    check_row("valid and duplicate scenarios are byte-identical raw JSON", _SCENARIO_TO_RAW["valid"] == _SCENARIO_TO_RAW["duplicate_of_valid"]),
    check_row("stub generation is deterministic across backend instances", tuple(repeat_scenarios) == DEFAULT_SCENARIO_SEQUENCE, repeat_scenarios),
    check_row("stub is backend-protocol compatible", isinstance(backend, ProposerBackend)),
]
display_check_table(stub_checks, "Stub backend scenario checks")
D6_ACCEPTANCE["stub_scenario_coverage"] = True

,scenario,raw_output_prefix,expected_candidate_type,actual_candidate_type,classification_pass,error_kind,backend_name,cost_estimate_usd
0,valid,"{""description"":""Stub: enter on 24h return > 2%...",ValidCandidate,ValidCandidate,True,NaN,stub,0.02
1,invalid_json,"{""name"":""stub_bad_json"",""description"":""Stub: u...",InvalidCandidate,InvalidCandidate,True,invalid_json,stub,0.02
2,duplicate_of_valid,"{""description"":""Stub: enter on 24h return > 2%...",ValidCandidate,ValidCandidate,True,NaN,stub,0.02
3,over_complex,"{""description"":""Stub: exceeds entry group budg...",InvalidCandidate,InvalidCandidate,True,over_complex,stub,0.02
4,factor_out_of_registry,"{""description"":""Stub: references an unregister...",InvalidCandidate,InvalidCandidate,True,factor_out_of_registry,stub,0.02
5,grammar_violation,"{""description"":""Stub: uses a disallowed operat...",InvalidCandidate,InvalidCandidate,True,grammar_violation,stub,0.02
6,non_finite_threshold,"{""name"":""stub_non_finite"",""description"":""Stub:...",InvalidCandidate,InvalidCandidate,True,non_finite_threshold,stub,0.02


Stub backend scenario checks


,check,status,detail
0,default scenario sequence has seven categories,PASS,"(valid, invalid_json, duplicate_of_valid, over..."
1,each scenario maps to expected candidate type,PASS,scenario ...
2,valid and duplicate scenarios are byte-identic...,PASS,
3,stub generation is deterministic across backen...,PASS,"[valid, invalid_json, duplicate_of_valid, over..."
4,stub is backend-protocol compatible,PASS,


## F. Ingest Pipeline and Lifecycle Classification

Each proposer output flows through the same Stage 1 lifecycle path:

`raw parse → schema validation → D3 hash/dedup → lifecycle assignment → attempt accounting`

The key discipline: invalid DSL is **not** a free retry, and duplicates do **not** become fresh ideas.

In [7]:
EXPECTED_LIFECYCLE = {
    "valid": PENDING_BACKTEST,
    "invalid_json": INVALID_DSL,
    "duplicate_of_valid": DUPLICATE,
    "over_complex": REJECTED_COMPLEXITY,
    "factor_out_of_registry": INVALID_DSL,
    "grammar_violation": INVALID_DSL,
    "non_finite_threshold": INVALID_DSL,
}

state = BatchIngestState(batch_id=D6_BATCH_ID)
records = ingest_outputs(state, scenario_outputs)
assert_lifecycle_invariant_at_batch_close(state)

ingest_rows = []
for output, record in zip(scenario_outputs, records):
    scenario = output.telemetry["scenario"]
    candidate = output.candidates[0]
    parsed_json = isinstance(candidate, ValidCandidate) or not str(record.parse_error or "").startswith("json_decode_error")
    schema_valid = isinstance(candidate, ValidCandidate)
    dedup_status = (
        "duplicate" if record.lifecycle_state == DUPLICATE
        else "unique" if record.lifecycle_state == PENDING_BACKTEST
        else "n/a"
    )
    ingest_rows.append({
        "scenario": scenario,
        "parsed_json": parsed_json,
        "schema_valid": schema_valid,
        "dedup_status": dedup_status,
        "expected_lifecycle": EXPECTED_LIFECYCLE[scenario],
        "actual_lifecycle": record.lifecycle_state,
        "counted_as_attempted": True,
        "record_created": record in state.records,
        "hypothesis_hash_present": record.hypothesis_hash is not None,
        "error_kind": record.error_kind,
        "reason_excerpt": (record.parse_error or "")[:120],
    })

ingest_df = pd.DataFrame(ingest_rows)
display(ingest_df)

valid_hash = records[0].hypothesis_hash
duplicate_hash = records[2].hypothesis_hash

ingest_checks = [
    check_row("all scenarios routed to expected lifecycle", bool((ingest_df["expected_lifecycle"] == ingest_df["actual_lifecycle"]).all()), ingest_df[["scenario", "expected_lifecycle", "actual_lifecycle"]]),
    check_row("lifecycle close invariant holds", sum(state.lifecycle_counts.values()) == state.hypotheses_attempted == len(records), dict(state.lifecycle_counts)),
    check_row("only first valid unique candidate is pending_backtest", dict(state.lifecycle_counts).get(PENDING_BACKTEST, 0) == 1, dict(state.lifecycle_counts)),
    check_row("duplicate candidate keeps same D3 hash but is not fresh", valid_hash == duplicate_hash and records[2].lifecycle_state == DUPLICATE, {"valid_hash": valid_hash, "duplicate_hash": duplicate_hash}),
    check_row("over-complex candidate is separated from generic invalid DSL", records[3].lifecycle_state == REJECTED_COMPLEXITY, records[3]),
]
display_check_table(ingest_checks, "Ingest lifecycle classification checks")
D6_ACCEPTANCE["ingest_lifecycle"] = True

,scenario,parsed_json,schema_valid,dedup_status,expected_lifecycle,actual_lifecycle,counted_as_attempted,record_created,hypothesis_hash_present,error_kind,reason_excerpt
0,valid,True,True,unique,pending_backtest,pending_backtest,True,True,True,NaN,
1,invalid_json,False,False,n/a,invalid_dsl,invalid_dsl,True,True,False,invalid_json,json_decode_error: Expecting property name enc...
2,duplicate_of_valid,True,True,duplicate,duplicate,duplicate,True,True,True,NaN,
3,over_complex,True,False,n/a,rejected_complexity,rejected_complexity,True,True,False,over_complex,1 validation error for StrategyDSL\nentry\n L...
4,factor_out_of_registry,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,factor_out_of_registry,1 validation error for StrategyDSL\nentry.0.co...
5,grammar_violation,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,grammar_violation,1 validation error for StrategyDSL\nentry.0.co...
6,non_finite_threshold,True,False,n/a,invalid_dsl,invalid_dsl,True,True,False,non_finite_threshold,1 validation error for StrategyDSL\nentry.0.co...


Ingest lifecycle classification checks


,check,status,detail
0,all scenarios routed to expected lifecycle,PASS,scenario expected_lifecycle...
1,lifecycle close invariant holds,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."
2,only first valid unique candidate is pending_b...,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."
3,duplicate candidate keeps same D3 hash but is ...,PASS,"{'valid_hash': 'de4319c3950032cf', 'duplicate_..."
4,over-complex candidate is separated from gener...,PASS,HypothesisRecord(batch_id='d6-stage1-acceptanc...


## G. Attempt Accounting Discipline

D6 Stage 1 must prove the accounting rule that matters later for DSR and budget control:

- every emitted candidate increments `hypotheses_attempted`
- invalid DSL and schema probing are counted
- duplicate ideas are counted as attempts but do not become fresh hypotheses

In [8]:
attempt_summary = pd.DataFrame([
    {"metric": "hypotheses_attempted", "value": state.hypotheses_attempted},
    {"metric": "records_created", "value": len(state.records)},
    {"metric": "invalid_dsl", "value": state.lifecycle_counts.get(INVALID_DSL, 0)},
    {"metric": "rejected_complexity", "value": state.lifecycle_counts.get(REJECTED_COMPLEXITY, 0)},
    {"metric": "duplicate", "value": state.lifecycle_counts.get(DUPLICATE, 0)},
    {"metric": "pending_backtest", "value": state.lifecycle_counts.get(PENDING_BACKTEST, 0)},
])
display(attempt_summary)

invalid_or_rejected = ingest_df[ingest_df["actual_lifecycle"].isin([INVALID_DSL, REJECTED_COMPLEXITY])]
duplicate_rows = ingest_df[ingest_df["actual_lifecycle"] == DUPLICATE]

attempt_checks = [
    check_row("every candidate increments hypotheses_attempted", state.hypotheses_attempted == len(DEFAULT_SCENARIO_SEQUENCE), state.hypotheses_attempted),
    check_row("invalid/rejected DSL candidates are counted", len(invalid_or_rejected) == 5 and bool(invalid_or_rejected["counted_as_attempted"].all()), invalid_or_rejected[["scenario", "actual_lifecycle"]]),
    check_row("duplicate is counted but not a fresh pending hypothesis", len(duplicate_rows) == 1 and state.lifecycle_counts.get(PENDING_BACKTEST, 0) == 1, duplicate_rows),
    check_row("terminal lifecycle counts sum to attempts", sum(state.lifecycle_counts.values()) == state.hypotheses_attempted, dict(state.lifecycle_counts)),
]
display_check_table(attempt_checks, "Attempt accounting checks")
D6_ACCEPTANCE["attempt_accounting"] = True

,metric,value
0,hypotheses_attempted,7
1,records_created,7
2,invalid_dsl,4
3,rejected_complexity,1
4,duplicate,1
5,pending_backtest,1


Attempt accounting checks


,check,status,detail
0,every candidate increments hypotheses_attempted,PASS,7
1,invalid/rejected DSL candidates are counted,PASS,scenario actual_lifecycle...
2,duplicate is counted but not a fresh pending h...,PASS,scenario parsed_json schema_val...
3,terminal lifecycle counts sum to attempts,PASS,"{'pending_backtest': 1, 'invalid_dsl': 4, 'dup..."


## H. Crash-Safe Budget Ledger

The ledger uses a pre-charge pattern:

1. write a `pending` row before the backend call
2. count pending rows as spent immediately
3. finalize successful calls with actual cost
4. preserve pending or crashed rows as spent if the process is interrupted

This proves the budget discipline that the live Sonnet backend will inherit later.

In [9]:
ledger_tmp = tempfile.TemporaryDirectory()
ledger_path = Path(ledger_tmp.name) / "d6_budget_acceptance.db"
ledger = BudgetLedger(ledger_path)
now = datetime(2026, 4, 17, 12, 0, tzinfo=timezone.utc)

# Normal flow: pre-charge then finalize with actual cost.
normal_batch = "d6-normal-flow"
normal_id = ledger.write_pending(
    batch_id=normal_batch,
    api_call_kind="proposer.generate",
    estimated_cost_usd=0.50,
    now=now,
    notes="normal flow pre-charge",
)
normal_spent_pending = ledger.batch_spent_usd(normal_batch)
ledger.finalize(normal_id, actual_cost_usd=0.12, now=now)
normal_spent_final = ledger.batch_spent_usd(normal_batch)

# Interrupted/crash-like flow: pending row remains visible across a new instance.
crash_batch = "d6-crash-like-flow"
crash_id = ledger.write_pending(
    batch_id=crash_batch,
    api_call_kind="proposer.generate",
    estimated_cost_usd=2.00,
    now=now,
    notes="simulated interruption before finalize",
)
ledger_after_restart = BudgetLedger(ledger_path)
crash_spent_pending = ledger_after_restart.batch_spent_usd(crash_batch)
pending_after_restart = ledger_after_restart.pending_entries(batch_id=crash_batch)
ledger_after_restart.mark_crashed(crash_id, now=now, notes="marked crashed for audit")
crash_spent_marked = ledger_after_restart.batch_spent_usd(crash_batch)

ledger_rows = []
for entry in ledger_after_restart.list_entries():
    ledger_rows.append({
        "batch_id": entry.batch_id,
        "status": entry.status,
        "estimated_cost": entry.estimated_cost,
        "actual_cost": entry.actual_cost,
        "counts_as_spent": ledger_after_restart.batch_spent_usd(entry.batch_id),
        "notes": entry.notes,
    })
display(pd.DataFrame(ledger_rows))

budget_checks = [
    check_row("pending pre-charge counts immediately", abs(normal_spent_pending - 0.50) < 1e-12, normal_spent_pending),
    check_row("finalize replaces estimate with actual cost", abs(normal_spent_final - 0.12) < 1e-12, normal_spent_final),
    check_row("pending row survives process-boundary restart", len(pending_after_restart) == 1 and abs(crash_spent_pending - 2.00) < 1e-12, pending_after_restart),
    check_row("crashed row still counts as spent", abs(crash_spent_marked - 2.00) < 1e-12, crash_spent_marked),
    check_row("ledger uses SQLite file artifact", ledger_path.exists(), ledger_path),
]
display_check_table(budget_checks, "Crash-safe budget ledger checks")
D6_ACCEPTANCE["budget_ledger"] = True

,batch_id,status,estimated_cost,actual_cost,counts_as_spent,notes
0,d6-normal-flow,completed,0.5,0.12,0.12,normal flow pre-charge
1,d6-crash-like-flow,crashed,2.0,NaN,2.00,simulated interruption before finalize\nmarked...


Crash-safe budget ledger checks


,check,status,detail
0,pending pre-charge counts immediately,PASS,0.5
1,finalize replaces estimate with actual cost,PASS,0.12
2,pending row survives process-boundary restart,PASS,[LedgerEntry(id='97719f0f-57dd-450e-8841-6d2eb...
3,crashed row still counts as spent,PASS,2.0
4,ledger uses SQLite file artifact,PASS,/var/folders/rq/573stz453md6gpcyqpz5y_dm0000gn...


## I. Backend Replaceability and Stage 2 Boundary

Stage 1 must keep the model backend replaceable. The orchestrator should consume `ProposerOutput` and candidate variants, not Sonnet-specific payloads.

**Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.**

In [10]:
ingest_src = Path("agents/orchestrator/ingest.py").read_text()
budget_src = Path("agents/orchestrator/budget_ledger.py").read_text()
stub_src = Path("agents/proposer/stub_backend.py").read_text()
interface_src_full = Path("agents/proposer/interface.py").read_text()


def has_import(src: str, module: str) -> bool:
    return any(
        line.strip() == f"import {module}" or line.strip().startswith(f"from {module} import")
        for line in src.splitlines()
    )


def imports_prefix(src: str, prefix: str) -> bool:
    return any(
        line.strip().startswith(f"import {prefix}") or line.strip().startswith(f"from {prefix}")
        for line in src.splitlines()
    )

backend_boundary_rows = [
    {
        "component": "ProposerBackend Protocol",
        "evidence": str(inspect.signature(ProposerBackend.generate)),
        "backend_specific_import": False,
    },
    {
        "component": "StubProposerBackend",
        "evidence": f"isinstance(stub, Protocol) = {isinstance(backend, ProposerBackend)}",
        "backend_specific_import": False,
    },
    {
        "component": "ingest.py",
        "evidence": "imports candidate schemas and D3 hash; no Anthropic SDK import",
        "backend_specific_import": has_import(ingest_src, "anthropic"),
    },
    {
        "component": "budget_ledger.py",
        "evidence": "stdlib SQLite ledger; no proposer/backend imports",
        "backend_specific_import": has_import(budget_src, "anthropic") or imports_prefix(budget_src, "agents.proposer"),
    },
]
display(pd.DataFrame(backend_boundary_rows))

boundary_checks = [
    check_row("ProposerBackend exposes a single generate(context) contract", "generate" in interface_src_full and "BatchContext" in interface_src_full and "ProposerOutput" in interface_src_full),
    check_row("stub backend imports no Anthropic SDK", not has_import(stub_src, "anthropic")),
    check_row("ingest path imports no Anthropic SDK", not has_import(ingest_src, "anthropic")),
    check_row("ingest path does not import stub backend", not imports_prefix(ingest_src, "agents.proposer.stub_backend") and "StubProposerBackend" not in ingest_src),
    check_row("budget ledger is backend-agnostic", not has_import(budget_src, "anthropic") and not imports_prefix(budget_src, "agents.proposer")),
]
display_check_table(boundary_checks, "Backend replaceability checks")
D6_ACCEPTANCE["backend_replaceability"] = True


,component,evidence,backend_specific_import
0,ProposerBackend Protocol,"(self, context: 'BatchContext') -> 'ProposerOu...",False
1,StubProposerBackend,"isinstance(stub, Protocol) = True",False
2,ingest.py,imports candidate schemas and D3 hash; no Anth...,False
3,budget_ledger.py,stdlib SQLite ledger; no proposer/backend imports,False


Backend replaceability checks


,check,status,detail
0,ProposerBackend exposes a single generate(cont...,PASS,
1,stub backend imports no Anthropic SDK,PASS,
2,ingest path imports no Anthropic SDK,PASS,
3,ingest path does not import stub backend,PASS,
4,budget ledger is backend-agnostic,PASS,


## J. Test Evidence Summary

The notebook is the reviewer-facing sign-off artifact. Pytest remains the primary regression surface.

Stage 1 test evidence reported for this checkpoint:

- 5 new D6 test files
- 58 new test cases/effective cases
- full suite: 648 passing, 0 regressions

In [11]:
test_files = [
    "tests/test_proposer_interface.py",
    "tests/test_proposer_prompt.py",
    "tests/test_proposer_stub.py",
    "tests/test_orchestrator_ingest.py",
    "tests/test_budget_ledger.py",
]

test_evidence_rows = []
for path in test_files:
    text = Path(path).read_text()
    test_evidence_rows.append({
        "test_file": path,
        "exists": Path(path).exists(),
        "static_test_function_count": sum(1 for line in text.splitlines() if line.startswith("def test_")),
        "theme": {
            "tests/test_proposer_interface.py": "proposer interface and candidate schema",
            "tests/test_proposer_prompt.py": "prompt structure and leakage audit",
            "tests/test_proposer_stub.py": "deterministic stub scenarios",
            "tests/test_orchestrator_ingest.py": "ingest lifecycle and attempt accounting",
            "tests/test_budget_ledger.py": "pre-charge ledger and crash safety",
        }[path],
    })

test_evidence_df = pd.DataFrame(test_evidence_rows)
display(test_evidence_df)

reported_summary = pd.DataFrame([
    {"evidence": "new D6 test files", "reported": 5},
    {"evidence": "new D6 tests/effective cases", "reported": 58},
    {"evidence": "full suite", "reported": "648 passing, 0 regressions"},
])
display(reported_summary)

test_checks = [
    check_row("all D6 test files exist", bool(test_evidence_df["exists"].all()), test_evidence_df),
    check_row("five D6 test files are represented", len(test_evidence_df) == 5),
    check_row("test themes cover interface, prompt, stub, ingest, and ledger", set(test_evidence_df["theme"]).issuperset({
        "proposer interface and candidate schema",
        "prompt structure and leakage audit",
        "deterministic stub scenarios",
        "ingest lifecycle and attempt accounting",
        "pre-charge ledger and crash safety",
    })),
    check_row("reported full-suite evidence is recorded", "648 passing" in reported_summary.loc[2, "reported"]),
]
display_check_table(test_checks, "D6 test evidence checks")
D6_ACCEPTANCE["test_evidence"] = True

,test_file,exists,static_test_function_count,theme
0,tests/test_proposer_interface.py,True,7,proposer interface and candidate schema
1,tests/test_proposer_prompt.py,True,14,prompt structure and leakage audit
2,tests/test_proposer_stub.py,True,12,deterministic stub scenarios
3,tests/test_orchestrator_ingest.py,True,14,ingest lifecycle and attempt accounting
4,tests/test_budget_ledger.py,True,14,pre-charge ledger and crash safety


,evidence,reported
0,new D6 test files,5
1,new D6 tests/effective cases,58
2,full suite,"648 passing, 0 regressions"


D6 test evidence checks


,check,status,detail
0,all D6 test files exist,PASS,test_file exists ...
1,five D6 test files are represented,PASS,
2,"test themes cover interface, prompt, stub, ing...",PASS,
3,reported full-suite evidence is recorded,PASS,


## K. D6 Stage 1 Verdict

Acceptance checklist:

- proposer interface is frozen enough for Stage 2
- prompt leakage audit is in place
- deterministic stub covers major contract scenarios
- ingest classifies valid / invalid / duplicate / complexity / grammar failures deterministically
- invalid DSL counts as attempted
- duplicates do not become fresh hypotheses
- crash-safe pre-charge ledger works
- live-model integration is intentionally deferred

D6 Stage 1 passes acceptance and is ready for Stage 2 live Sonnet integration.

In [12]:
verdict_rows = [
    {"gate": "delivered components", "pass": D6_ACCEPTANCE.get("delivered_components", False)},
    {"gate": "frozen batch-start boundaries", "pass": D6_ACCEPTANCE.get("frozen_boundaries", False)},
    {"gate": "prompt builder + leakage audit", "pass": D6_ACCEPTANCE.get("prompt_leakage_audit", False)},
    {"gate": "deterministic stub scenario coverage", "pass": D6_ACCEPTANCE.get("stub_scenario_coverage", False)},
    {"gate": "ingest lifecycle classification", "pass": D6_ACCEPTANCE.get("ingest_lifecycle", False)},
    {"gate": "attempt accounting discipline", "pass": D6_ACCEPTANCE.get("attempt_accounting", False)},
    {"gate": "crash-safe budget ledger", "pass": D6_ACCEPTANCE.get("budget_ledger", False)},
    {"gate": "backend replaceability", "pass": D6_ACCEPTANCE.get("backend_replaceability", False)},
    {"gate": "test evidence recorded", "pass": D6_ACCEPTANCE.get("test_evidence", False)},
]
verdict_df = pd.DataFrame(verdict_rows)
verdict_df["status"] = verdict_df["pass"].map({True: "PASS", False: "FAIL"})
display(verdict_df[["gate", "status"]])

assert verdict_df["pass"].all(), verdict_df[~verdict_df["pass"]]

print("D6 Stage 1 acceptance verdict: PASS")
print("Stage 1 proves loop discipline without introducing model variability.")
print("Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.")

,gate,status
0,delivered components,PASS
1,frozen batch-start boundaries,PASS
2,prompt builder + leakage audit,PASS
3,deterministic stub scenario coverage,PASS
4,ingest lifecycle classification,PASS
5,attempt accounting discipline,PASS
6,crash-safe budget ledger,PASS
7,backend replaceability,PASS
8,test evidence recorded,PASS


D6 Stage 1 acceptance verdict: PASS
Stage 1 proves loop discipline without introducing model variability.
Stage 2 must swap in the live Sonnet proposer backend without relaxing any Stage 1 accounting, leakage, or frozen-search-space constraints.


## L. D6 Stage 2a Scope and Acceptance Criteria

D6 Stage 2a is the **first live Sonnet smoke test**. It is intentionally narrow:

- batch size = 1
- budget cap = $1
- prompt caching disabled
- objective is not model quality or alpha quality
- objective is live backend integration correctness under the same Stage 1 constraints

Stage 2a passes if all of the following hold:

- actual live prompt passes leakage audit
- live response is captured on disk and can be inspected
- response is either parseable or cleanly classified
- exactly one terminal outcome is recorded for exactly one attempted hypothesis
- no unclassified lifecycle state appears
- cost is fully reconciled in the ledger
- raw prompt/response artifacts exist on disk
- pipeline completes without crash

A live `invalid_dsl` result is **not** a failure for Stage 2a. It is acceptable if the output is safely recorded, classified, counted, and reconciled.

In [13]:
import re
import sqlite3
import subprocess
from datetime import datetime, timezone

from agents.proposer.stub_backend import classify_raw_json
from agents.proposer.sonnet_backend import SONNET_MODEL, compute_cost_usd, estimate_cost_usd

D6_STAGE2A_ACCEPTANCE: dict[str, bool] = {}

# Prefer the original accepted smoke batch when present, but fall back to the
# latest completed live batch with prompt/response artifacts so the notebook
# remains rerunnable after future Stage 2a smoke runs.
STAGE2A_ACCEPTANCE_BATCH_ID = "03d62937-dbe8-46f2-a91b-50fa5696b14e"
# Earlier dated-model smoke attempt retained as operational evidence for Stage 2b notes.
STAGE2A_DATED_MODEL_404_BATCH_ID = "9d812aef-4b17-424e-a6ed-660a7bd72c57"

STAGE2A_LEDGER_PATH = Path("agents/spend_ledger.db")
STAGE2A_BATCH_CAP_USD = 1.0
STAGE2A_BATCH_SIZE = 1
STAGE2A_PROMPT_CACHING = "disabled"

assert STAGE2A_LEDGER_PATH.exists(), STAGE2A_LEDGER_PATH


def _payload_paths(batch_id: str) -> tuple[Path, Path, Path]:
    payload_dir = Path("raw_payloads") / f"batch_{batch_id}"
    return (
        payload_dir,
        payload_dir / "attempt_0001_prompt.txt",
        payload_dir / "attempt_0001_response.txt",
    )


def _ledger_row(batch_id: str) -> dict | None:
    with sqlite3.connect(STAGE2A_LEDGER_PATH) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT * FROM ledger WHERE batch_id = ? ORDER BY created_at_utc DESC LIMIT 1",
            (batch_id,),
        ).fetchone()
    return dict(row) if row is not None else None


def _latest_completed_live_batch() -> tuple[str, dict, Path, Path, Path, str]:
    candidates: list[tuple[str, dict, Path, Path, Path, str]] = []

    # Stable acceptance artifact first when available.
    accepted_dir, accepted_prompt, accepted_response = _payload_paths(STAGE2A_ACCEPTANCE_BATCH_ID)
    accepted_row = _ledger_row(STAGE2A_ACCEPTANCE_BATCH_ID)
    if accepted_row and accepted_row["status"] == STATUS_COMPLETED and accepted_prompt.exists() and accepted_response.exists():
        candidates.append((STAGE2A_ACCEPTANCE_BATCH_ID, accepted_row, accepted_dir, accepted_prompt, accepted_response, "pinned acceptance batch"))

    # Fallback: latest completed ledger row with both artifacts and positive actual cost.
    with sqlite3.connect(STAGE2A_LEDGER_PATH) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            "SELECT * FROM ledger WHERE status = ? ORDER BY created_at_utc DESC",
            (STATUS_COMPLETED,),
        ).fetchall()
    for row_obj in rows:
        row = dict(row_obj)
        batch_id = row["batch_id"]
        payload_dir, prompt_path, response_path = _payload_paths(batch_id)
        if prompt_path.exists() and response_path.exists() and float(row.get("actual_cost") or 0.0) > 0:
            candidates.append((batch_id, row, payload_dir, prompt_path, response_path, "latest completed live batch"))
            break

    if not candidates:
        raise AssertionError("No completed Stage 2a live batch with prompt/response artifacts found")
    return candidates[0]


(
    STAGE2A_BATCH_ID,
    STAGE2A_LEDGER_ROW,
    STAGE2A_PAYLOAD_DIR,
    STAGE2A_PROMPT_PATH,
    STAGE2A_RESPONSE_PATH,
    STAGE2A_BATCH_SELECTION,
) = _latest_completed_live_batch()

assert STAGE2A_PROMPT_PATH.exists(), STAGE2A_PROMPT_PATH
assert STAGE2A_RESPONSE_PATH.exists(), STAGE2A_RESPONSE_PATH
STAGE2A_GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
STAGE2A_MONTHLY_SPEND = BudgetLedger(STAGE2A_LEDGER_PATH).monthly_spent_usd(now=datetime.now(timezone.utc))

stage2a_receipt = pd.DataFrame([
    {"item": "batch_id", "value": STAGE2A_BATCH_ID},
    {"item": "batch selection", "value": STAGE2A_BATCH_SELECTION},
    {"item": "git commit", "value": STAGE2A_GIT_COMMIT},
    {"item": "ledger created_at_utc", "value": STAGE2A_LEDGER_ROW["created_at_utc"]},
    {"item": "batch size", "value": STAGE2A_BATCH_SIZE},
    {"item": "batch budget cap", "value": STAGE2A_BATCH_CAP_USD},
    {"item": "cumulative UTC-month spend", "value": STAGE2A_MONTHLY_SPEND},
    {"item": "model id used by backend", "value": SONNET_MODEL},
    {"item": "prompt caching", "value": STAGE2A_PROMPT_CACHING},
    {"item": "Stage 2b work started", "value": "no"},
    {"item": "raw payload directory", "value": str(STAGE2A_PAYLOAD_DIR)},
])
display(stage2a_receipt)

receipt_checks = [
    check_row("prompt artifact exists", STAGE2A_PROMPT_PATH.exists(), STAGE2A_PROMPT_PATH),
    check_row("response artifact exists", STAGE2A_RESPONSE_PATH.exists(), STAGE2A_RESPONSE_PATH),
    check_row("ledger row exists and completed", STAGE2A_LEDGER_ROW["status"] == STATUS_COMPLETED, STAGE2A_LEDGER_ROW),
    check_row("batch size is exactly 1", STAGE2A_BATCH_SIZE == 1),
    check_row("batch cap is $1", abs(STAGE2A_BATCH_CAP_USD - 1.0) < 1e-12),
    check_row("prompt caching is disabled", STAGE2A_PROMPT_CACHING == "disabled"),
    check_row("current live model uses unversioned Sonnet alias", SONNET_MODEL == "claude-sonnet-4-5", SONNET_MODEL),
]
display_check_table(receipt_checks, "Stage 2a live run receipt checks")
D6_STAGE2A_ACCEPTANCE["live_receipts"] = True

,item,value
0,batch_id,03d62937-dbe8-46f2-a91b-50fa5696b14e
1,batch selection,pinned acceptance batch
2,git commit,cbf6c96
3,ledger created_at_utc,2026-04-17T13:19:46.234Z
4,batch size,1
5,batch budget cap,1.0
6,cumulative UTC-month spend,2.49966
7,model id used by backend,claude-sonnet-4-5
8,prompt caching,disabled
9,Stage 2b work started,no


Stage 2a live run receipt checks


,check,status,detail
0,prompt artifact exists,PASS,raw_payloads/batch_03d62937-dbe8-46f2-a91b-50f...
1,response artifact exists,PASS,raw_payloads/batch_03d62937-dbe8-46f2-a91b-50f...
2,ledger row exists and completed,PASS,"{'id': '2f2ce481-dc2f-4718-871e-f2b5db20b428',..."
3,batch size is exactly 1,PASS,
4,batch cap is $1,PASS,
5,prompt caching is disabled,PASS,
6,current live model uses unversioned Sonnet alias,PASS,claude-sonnet-4-5


## M. Actual Prompt Construction and Leakage Audit

This section audits the **actual live prompt artifact** used for the Sonnet call, not a reconstructed stub prompt.

The prompt should contain the frozen factor menu, DSL grammar constraints, and complexity budget. It should not contain protected evaluation information such as 2022 holdout results, validation/test metrics, leaderboard entries, or per-hypothesis historical performance.

In [14]:
stage2a_prompt_text = STAGE2A_PROMPT_PATH.read_text(encoding="utf-8")
stage2a_prompt_findings = audit_prompt_for_leakage(stage2a_prompt_text)

print("Prompt path:", STAGE2A_PROMPT_PATH)
print("Prompt bytes:", len(stage2a_prompt_text.encode("utf-8")))
print("\nPrompt preview:")
print("\n".join(stage2a_prompt_text.splitlines()[:18]))

prompt_audit_rows = [
    check_row("actual prompt leakage audit is clean", stage2a_prompt_findings == [], stage2a_prompt_findings),
    check_row("no 2022/holdout leakage", not any(x in stage2a_prompt_text.lower() for x in ["2022-", "2022/", "bear_2022", "regime_holdout", "regime holdout"])),
    check_row("no validation/test metric leakage", not any(x in stage2a_prompt_text.lower() for x in ["validation_sharpe", "validation_return", "test_sharpe", "test_return", "test_drawdown"])),
    check_row("no leaderboard or DSR result leakage", not any(x in stage2a_prompt_text.lower() for x in ["leaderboard", "dsr_threshold", "shortlisted"])),
    check_row("frozen factor menu is present", "# Available factors" in stage2a_prompt_text and "return_24h" in stage2a_prompt_text and "rsi_14" in stage2a_prompt_text),
    check_row("frozen DSL/operator constraints are present", all(op in stage2a_prompt_text for op in ALLOWED_OPS), ALLOWED_OPS),
    check_row("complexity budget is present", "complexity budget" in stage2a_prompt_text.lower() and "max_hold_bars" in stage2a_prompt_text),
    check_row("no-new-factor/operator boundary is explicit", "MAY NOT propose new factors" in stage2a_prompt_text and "new operators" in stage2a_prompt_text),
]
display_check_table(prompt_audit_rows, "Actual live prompt leakage and boundary checks")
D6_STAGE2A_ACCEPTANCE["actual_prompt_audit"] = True

Prompt path: raw_payloads/batch_03d62937-dbe8-46f2-a91b-50fa5696b14e/attempt_0001_prompt.txt
Prompt bytes: 2892

Prompt preview:
You are a quantitative researcher proposing BTC trading hypotheses.
Respond ONLY with valid JSON matching the StrategyDSL schema.
Your strategy MUST:
  - Reference only factors from the menu provided by the caller.
  - Use only these operators: <, <=, >, >=, ==, crosses_above, crosses_below.
  - Include a one-sentence economic rationale in `description`.
  - Use long/flat positions only (`position_sizing: "full_equity"`).
  - Respect the complexity budget: entry/exit each ≤ 3 condition groups, each group ≤ 4 conditions, max_hold_bars ≤ 720.

You MAY NOT propose new factors, new operators, inline arithmetic, or any grammar outside the schema. Registry and grammar are frozen for this batch; factor or operator additions require explicit human review outside this loop.
Batch context:
  - batch_id: 03d62937-dbe8-46f2-a91b-50fa5696b14e
  - position: 1/1
  - theme (

,check,status,detail
0,actual prompt leakage audit is clean,PASS,[]
1,no 2022/holdout leakage,PASS,
2,no validation/test metric leakage,PASS,
3,no leaderboard or DSR result leakage,PASS,
4,frozen factor menu is present,PASS,
5,frozen DSL/operator constraints are present,PASS,"(<, <=, >, >=, ==, crosses_above, crosses_below)"
6,complexity budget is present,PASS,
7,no-new-factor/operator boundary is explicit,PASS,


## N. Raw Sonnet Response Capture and Schema Mismatch Inspection

The raw response must be kept as a first-class artifact. Stage 2a should prove that the first live mismatch is inspectable and explainable.

In this run, Sonnet returned a markdown-fenced JSON object. The JSON payload is extractable and human-readable, but its field names do not match the frozen D2 DSL schema.

In [15]:
stage2a_response_text = STAGE2A_RESPONSE_PATH.read_text(encoding="utf-8")
print("Response path:", STAGE2A_RESPONSE_PATH)
print("Response bytes:", len(stage2a_response_text.encode("utf-8")))
print("\nResponse preview:")
print("\n".join(stage2a_response_text.splitlines()[:45]))

fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", stage2a_response_text, flags=re.DOTALL | re.IGNORECASE)
extracted_json_text = fenced.group(1).strip() if fenced else stage2a_response_text.strip()
json_payload = None
json_parse_error = None
try:
    json_payload = json.loads(extracted_json_text)
except Exception as exc:
    json_parse_error = repr(exc)

schema_candidate = classify_raw_json(
    extracted_json_text,
    registry=REGISTRY,
    error_kind_hint="sonnet_response_extracted_payload",
    provenance={"backend": "sonnet", "batch_id": STAGE2A_BATCH_ID, "position": 1},
)

schema_expected_vs_emitted = pd.DataFrame([
    {"Sonnet emitted": "hypothesis_id", "D2 schema expects": "name", "status": "mismatch"},
    {"Sonnet emitted": "entry_conditions", "D2 schema expects": "entry", "status": "mismatch"},
    {"Sonnet emitted": "exit_conditions", "D2 schema expects": "exit", "status": "mismatch"},
    {"Sonnet emitted": "left_factor / right_value", "D2 schema expects": "factor / value", "status": "mismatch"},
    {"Sonnet emitted": "operator", "D2 schema expects": "op", "status": "mismatch"},
    {"Sonnet emitted": "condition_group_id / join_operator", "D2 schema expects": "not present", "status": "extra metadata"},
])
display(schema_expected_vs_emitted)

response_checks = [
    check_row("raw response artifact exists", STAGE2A_RESPONSE_PATH.exists(), STAGE2A_RESPONSE_PATH),
    check_row("response text is non-empty", bool(stage2a_response_text.strip())),
    check_row("response contains extractable JSON payload", json_payload is not None, json_parse_error),
    check_row("extracted payload has Sonnet schema-mismatch fields", isinstance(json_payload, dict) and {"hypothesis_id", "entry_conditions", "exit_conditions"}.issubset(json_payload), json_payload.keys() if isinstance(json_payload, dict) else None),
    check_row("extracted payload does not validate as D2 StrategyDSL", isinstance(schema_candidate, InvalidCandidate), getattr(schema_candidate, "parse_error", "")),
]
display_check_table(response_checks, "Raw response capture and schema mismatch checks")
D6_STAGE2A_ACCEPTANCE["raw_response_capture"] = True

Response path: raw_payloads/batch_03d62937-dbe8-46f2-a91b-50fa5696b14e/attempt_0001_response.txt
Response bytes: 1328

Response preview:
```json
{
  "hypothesis_id": "momentum_rsi_macd_convergence_v1",
  "description": "Enter long when RSI crosses above oversold threshold while MACD histogram turns positive, capturing momentum regime shifts; exit when RSI becomes overbought or MACD reverses negative.",
  "entry_conditions": [
    {
      "condition_group_id": "rsi_oversold_cross",
      "conditions": [
        {
          "left_factor": "rsi_14",
          "operator": "crosses_above",
          "right_value": 35.0
        }
      ],
      "join_operator": "AND"
    },
    {
      "condition_group_id": "macd_positive",
      "conditions": [
        {
          "left_factor": "macd_hist",
          "operator": ">",
          "right_value": 0.0
        }
      ],
      "join_operator": "AND"
    }
  ],
  "exit_conditions": [
    {
      "condition_group_id": "rsi_overbought",
      "condi

,Sonnet emitted,D2 schema expects,status
0,hypothesis_id,name,mismatch
1,entry_conditions,entry,mismatch
2,exit_conditions,exit,mismatch
3,left_factor / right_value,factor / value,mismatch
4,operator,op,mismatch
5,condition_group_id / join_operator,not present,extra metadata


Raw response capture and schema mismatch checks


,check,status,detail
0,raw response artifact exists,PASS,raw_payloads/batch_03d62937-dbe8-46f2-a91b-50f...
1,response text is non-empty,PASS,
2,response contains extractable JSON payload,PASS,None
3,extracted payload has Sonnet schema-mismatch f...,PASS,"(hypothesis_id, description, entry_conditions,..."
4,extracted payload does not validate as D2 Stra...,PASS,6 validation errors for StrategyDSL\nname\n F...


## O. Lifecycle Classification and Attempt Accounting

Stage 2a's critical live-path check is that an invalid model output still becomes exactly one attempted hypothesis and exactly one terminal lifecycle state.

The persisted raw response is markdown-fenced JSON. The backend parser strips fences in memory while preserving the raw payload on disk; this response still classifies as `invalid_dsl` because its extracted JSON uses the wrong schema field names.

In [16]:
raw_candidate = classify_raw_json(
    stage2a_response_text,
    registry=REGISTRY,
    error_kind_hint="sonnet_response",
    provenance={"backend": "sonnet", "batch_id": STAGE2A_BATCH_ID, "position": 1},
)

stage2a_output = ProposerOutput(
    candidates=(raw_candidate,),
    cost_estimate_usd=float(STAGE2A_LEDGER_ROW["estimated_cost"]),
    cost_actual_usd=float(STAGE2A_LEDGER_ROW["actual_cost"]),
    backend_name="sonnet",
    telemetry={"batch_id": STAGE2A_BATCH_ID, "model": SONNET_MODEL},
)
stage2a_state = BatchIngestState(batch_id=STAGE2A_BATCH_ID)
stage2a_records = ingest_outputs(stage2a_state, [stage2a_output])
assert_lifecycle_invariant_at_batch_close(stage2a_state)
stage2a_record = stage2a_records[0]

classification_receipt = pd.DataFrame([
    {"item": "raw text parseable by backend json.loads", "value": not isinstance(raw_candidate, InvalidCandidate) or not str(raw_candidate.parse_error).startswith("json_decode_error")},
    {"item": "fenced JSON payload extractable", "value": json_payload is not None},
    {"item": "extracted payload schema valid", "value": not isinstance(schema_candidate, InvalidCandidate)},
    {"item": "lifecycle_state", "value": stage2a_record.lifecycle_state},
    {"item": "hypotheses_attempted", "value": stage2a_state.hypotheses_attempted},
    {"item": "hypothesis_hash", "value": stage2a_record.hypothesis_hash},
    {"item": "terminal record count", "value": len(stage2a_records)},
    {"item": "lifecycle_counts", "value": dict(stage2a_state.lifecycle_counts)},
])
display(classification_receipt)

classification_checks = [
    check_row("exactly one candidate was ingested", len(stage2a_records) == 1),
    check_row("exactly one hypothesis was attempted", stage2a_state.hypotheses_attempted == 1, stage2a_state.hypotheses_attempted),
    check_row("live invalid output routed to invalid_dsl", stage2a_record.lifecycle_state == INVALID_DSL, stage2a_record.lifecycle_state),
    check_row("invalid output has no hypothesis hash", stage2a_record.hypothesis_hash is None, stage2a_record.hypothesis_hash),
    check_row("lifecycle state is from Stage 1 allowed set", stage2a_record.lifecycle_state in D6_STAGE1_LIFECYCLE_STATES, D6_STAGE1_LIFECYCLE_STATES),
    check_row("lifecycle invariant remains intact", sum(stage2a_state.lifecycle_counts.values()) == stage2a_state.hypotheses_attempted, dict(stage2a_state.lifecycle_counts)),
]
display_check_table(classification_checks, "Stage 2a lifecycle and attempt-accounting checks")
D6_STAGE2A_ACCEPTANCE["lifecycle_attempt_accounting"] = True

,item,value
0,raw text parseable by backend json.loads,True
1,fenced JSON payload extractable,True
2,extracted payload schema valid,False
3,lifecycle_state,invalid_dsl
4,hypotheses_attempted,1
5,hypothesis_hash,None
6,terminal record count,1
7,lifecycle_counts,{'invalid_dsl': 1}


Stage 2a lifecycle and attempt-accounting checks


,check,status,detail
0,exactly one candidate was ingested,PASS,
1,exactly one hypothesis was attempted,PASS,1
2,live invalid output routed to invalid_dsl,PASS,invalid_dsl
3,invalid output has no hypothesis hash,PASS,None
4,lifecycle state is from Stage 1 allowed set,PASS,"(invalid_dsl, rejected_complexity, duplicate, ..."
5,lifecycle invariant remains intact,PASS,{'invalid_dsl': 1}


## P. Cost Reconciliation and Budget Ledger

The Stage 2a ledger should show that the live call was pre-charged, finalized, and left no orphaned pending charge for the smoke batch.

The ledger stores estimated and actual dollars. Exact input/output token counts were printed by the smoke script but are not persisted in the current ledger schema; the actual cost is still reconciled through the backend's API usage path before finalization.

In [17]:
ledger_for_stage2a = BudgetLedger(STAGE2A_LEDGER_PATH)
pending_for_batch = ledger_for_stage2a.pending_entries(batch_id=STAGE2A_BATCH_ID)
actual_cost = float(STAGE2A_LEDGER_ROW["actual_cost"])
estimated_cost = float(STAGE2A_LEDGER_ROW["estimated_cost"])

cost_rows = pd.DataFrame([
    {"item": "estimated pre-charge", "value": estimated_cost},
    {"item": "actual finalized cost", "value": actual_cost},
    {"item": "delta actual - estimated", "value": actual_cost - estimated_cost},
    {"item": "input tokens", "value": "not persisted in ledger/artifact"},
    {"item": "output tokens", "value": "not persisted in ledger/artifact"},
    {"item": "batch spent after finalize", "value": ledger_for_stage2a.batch_spent_usd(STAGE2A_BATCH_ID)},
    {"item": "UTC-month spend after finalize", "value": ledger_for_stage2a.monthly_spent_usd(now=datetime.now(timezone.utc))},
    {"item": "pending rows for smoke batch", "value": len(pending_for_batch)},
])
display(cost_rows)

cost_checks = [
    check_row("ledger row completed", STAGE2A_LEDGER_ROW["status"] == STATUS_COMPLETED, STAGE2A_LEDGER_ROW),
    check_row("estimated pre-charge is below $1 cap", estimated_cost <= STAGE2A_BATCH_CAP_USD, estimated_cost),
    check_row("actual cost is positive for successful live API traffic", actual_cost > 0, actual_cost),
    check_row("actual cost is below conservative estimate", actual_cost <= estimated_cost, {"actual": actual_cost, "estimated": estimated_cost}),
    check_row("batch spent equals finalized actual cost", abs(ledger_for_stage2a.batch_spent_usd(STAGE2A_BATCH_ID) - actual_cost) < 1e-12),
    check_row("no pending orphan remains for smoke batch", len(pending_for_batch) == 0, pending_for_batch),
]
display_check_table(cost_checks, "Stage 2a cost reconciliation and ledger checks")
D6_STAGE2A_ACCEPTANCE["cost_reconciliation"] = True

,item,value
0,estimated pre-charge,0.042
1,actual finalized cost,0.0096
2,delta actual - estimated,-0.0324
3,input tokens,not persisted in ledger/artifact
4,output tokens,not persisted in ledger/artifact
5,batch spent after finalize,0.0096
6,UTC-month spend after finalize,2.49966
7,pending rows for smoke batch,0


Stage 2a cost reconciliation and ledger checks


,check,status,detail
0,ledger row completed,PASS,"{'id': '2f2ce481-dc2f-4718-871e-f2b5db20b428',..."
1,estimated pre-charge is below $1 cap,PASS,0.042
2,actual cost is positive for successful live AP...,PASS,0.0096
3,actual cost is below conservative estimate,PASS,"{'actual': 0.009600000000000001, 'estimated': ..."
4,batch spent equals finalized actual cost,PASS,
5,no pending orphan remains for smoke batch,PASS,[]


## Q. What Stage 2a Proved

What Stage 2a proved:

- live backend integration works through the same proposer interface
- actual live prompt artifact exists and passes the leakage audit
- raw response artifact exists and is inspectable
- invalid real model output is classified cleanly
- invalid output is counted as exactly one attempted hypothesis
- lifecycle accounting remains invariant-preserving
- cost reconciliation closes the pre-charge ledger row
- the pipeline did not crash or leave partial state for the smoke batch

What Stage 2a did **not** prove:

- that the prompt is tuned enough for high valid-DSL yield
- that the proposer can produce high-quality alpha candidates
- that Stage 2b/2c/2d should proceed without prompt/schema review

In [18]:
stage2a_proof_rows = pd.DataFrame([
    {"claim": "live backend integration works", "evidence": "ledger completed + payload files exist"},
    {"claim": "leakage audit works on actual prompt", "evidence": "audit_prompt_for_leakage(actual_prompt) == []"},
    {"claim": "raw response logging works", "evidence": str(STAGE2A_RESPONSE_PATH)},
    {"claim": "invalid output is safely classified", "evidence": stage2a_record.lifecycle_state},
    {"claim": "invalid output is not a free retry", "evidence": f"hypotheses_attempted={stage2a_state.hypotheses_attempted}"},
    {"claim": "budget ledger closed cleanly", "evidence": f"status={STAGE2A_LEDGER_ROW['status']}, actual_cost={actual_cost}"},
])
display(stage2a_proof_rows)

proof_checks = [
    check_row("live prompt artifact and response artifact exist", STAGE2A_PROMPT_PATH.exists() and STAGE2A_RESPONSE_PATH.exists()),
    check_row("prompt audit clean", stage2a_prompt_findings == [], stage2a_prompt_findings),
    check_row("live invalid response classified without crash", stage2a_record.lifecycle_state == INVALID_DSL),
    check_row("attempt accounting survived live output", stage2a_state.hypotheses_attempted == 1),
    check_row("cost reconciliation closed", STAGE2A_LEDGER_ROW["status"] == STATUS_COMPLETED and len(pending_for_batch) == 0),
]
display_check_table(proof_checks, "Stage 2a robustness verdict checks")
D6_STAGE2A_ACCEPTANCE["robustness_verdict"] = True

,claim,evidence
0,live backend integration works,ledger completed + payload files exist
1,leakage audit works on actual prompt,audit_prompt_for_leakage(actual_prompt) == []
2,raw response logging works,raw_payloads/batch_03d62937-dbe8-46f2-a91b-50f...
3,invalid output is safely classified,invalid_dsl
4,invalid output is not a free retry,hypotheses_attempted=1
5,budget ledger closed cleanly,"status=completed, actual_cost=0.00960000000000..."


Stage 2a robustness verdict checks


,check,status,detail
0,live prompt artifact and response artifact exist,PASS,
1,prompt audit clean,PASS,[]
2,live invalid response classified without crash,PASS,
3,attempt accounting survived live output,PASS,
4,cost reconciliation closed,PASS,


## R. Stage 2b Implications and Go/No-Go Recommendation

Concrete Stage 2b implications from this smoke run:

1. **Schema example gap**
   - The live response used plausible strategy language but emitted `hypothesis_id`, `entry_conditions`, `left_factor`, and `right_value` instead of the frozen D2 schema fields.
   - Stage 2b should tighten the prompt with an exact DSL JSON example and stronger output-format constraints.

2. **Model id alias note**
   - The earlier dated model id attempt returned a 404.
   - The unversioned alias `claude-sonnet-4-5` was used successfully for the accepted smoke run.
   - This is operational evidence, not a conceptual design change.

3. **Stage 2a ends here**
   - This notebook accepts Stage 2a integration correctness.
   - Stage 2b should focus on prompt/schema-shape tightening, not infrastructure redesign.

In [19]:
earlier_404_response = Path("raw_payloads") / f"batch_{STAGE2A_DATED_MODEL_404_BATCH_ID}" / "attempt_0001_response.txt"
earlier_404_text = earlier_404_response.read_text(encoding="utf-8") if earlier_404_response.exists() else ""

stage2a_final_rows = [
    {"gate": "live run receipts", "pass": D6_STAGE2A_ACCEPTANCE.get("live_receipts", False)},
    {"gate": "actual prompt leakage audit", "pass": D6_STAGE2A_ACCEPTANCE.get("actual_prompt_audit", False)},
    {"gate": "raw response capture", "pass": D6_STAGE2A_ACCEPTANCE.get("raw_response_capture", False)},
    {"gate": "lifecycle + attempt accounting", "pass": D6_STAGE2A_ACCEPTANCE.get("lifecycle_attempt_accounting", False)},
    {"gate": "cost reconciliation", "pass": D6_STAGE2A_ACCEPTANCE.get("cost_reconciliation", False)},
    {"gate": "robustness verdict", "pass": D6_STAGE2A_ACCEPTANCE.get("robustness_verdict", False)},
    {"gate": "dated model 404 recorded", "pass": earlier_404_response.exists() and "not_found_error" in earlier_404_text},
]
stage2a_final_df = pd.DataFrame(stage2a_final_rows)
stage2a_final_df["status"] = stage2a_final_df["pass"].map({True: "PASS", False: "FAIL"})
display(stage2a_final_df[["gate", "status"]])

assert stage2a_final_df["pass"].all(), stage2a_final_df[~stage2a_final_df["pass"]]

print("D6 Stage 2a acceptance verdict: PASS")
print("First live Sonnet output did not validate as DSL, and that is acceptable for Stage 2a.")
print("The system recorded, classified, counted, reconciled, and preserved invariants without leakage or crash.")
print("Go/No-Go: GO to Stage 2b prompt/schema-shape tightening review.")

,gate,status
0,live run receipts,PASS
1,actual prompt leakage audit,PASS
2,raw response capture,PASS
3,lifecycle + attempt accounting,PASS
4,cost reconciliation,PASS
5,robustness verdict,PASS
6,dated model 404 recorded,PASS


D6 Stage 2a acceptance verdict: PASS
First live Sonnet output did not validate as DSL, and that is acceptable for Stage 2a.
The system recorded, classified, counted, reconciled, and preserved invariants without leakage or crash.
Go/No-Go: GO to Stage 2b prompt/schema-shape tightening review.


## S. D6 Stage 2b Scope and Acceptance Framing

D6 Stage 2b is a **5-call live observation batch**. It is not a parse-rate competition and it is not a model-quality sign-off.

The acceptance goal is to independently audit that live batch semantics held under real Sonnet traffic:

- theme rotation happened in the intended order
- approved examples evolved sequentially from prior valid outputs
- prompt leakage checks stayed clean after examples were injected
- responses re-classify cleanly through the current parser
- D3 hashes remain unique
- ledger rows, costs, and raw artifacts agree with the summary

Stage 2b passes if all calls are processed without contract violations, all prompts pass leakage audit, responses classify cleanly, lifecycle invariant holds, raw artifacts exist, and budget accounting closes cleanly.

In [20]:
import sqlite3
from agents.proposer.interface import ProposerOutput
from agents.proposer.stub_backend import classify_raw_json
from agents.orchestrator.ingest import ingest_outputs
from agents.hypothesis_hash import hash_dsl

D6_STAGE2B_ACCEPTANCE: dict[str, bool] = {}
STAGE2B_LEDGER_PATH = Path("agents/spend_ledger.db")

STAGE2B_SUMMARY_PATH = max(
    Path("raw_payloads").glob("batch_*/stage2b_summary.json"),
    key=lambda p: p.stat().st_mtime,
)
STAGE2B_BATCH_DIR = STAGE2B_SUMMARY_PATH.parent
STAGE2B_BATCH_ID = STAGE2B_BATCH_DIR.name.removeprefix("batch_")
STAGE2B_SUMMARY = json.loads(STAGE2B_SUMMARY_PATH.read_text(encoding="utf-8"))
STAGE2B_CALLS = STAGE2B_SUMMARY["calls"]
STAGE2B_BATCH_SIZE = STAGE2B_SUMMARY["batch_size"]
STAGE2B_BATCH_CAP_USD = STAGE2B_SUMMARY["batch_cap_usd"]

assert STAGE2B_BATCH_SIZE == 5
assert len(STAGE2B_CALLS) == 5

stage2b_receipts = pd.DataFrame([
    {"item": "batch_id", "value": STAGE2B_BATCH_ID},
    {"item": "summary path", "value": str(STAGE2B_SUMMARY_PATH)},
    {"item": "batch size", "value": STAGE2B_BATCH_SIZE},
    {"item": "batch cap", "value": STAGE2B_BATCH_CAP_USD},
    {"item": "hypotheses_attempted", "value": STAGE2B_SUMMARY["hypotheses_attempted"]},
    {"item": "total estimated cost", "value": STAGE2B_SUMMARY["total_estimated_cost_usd"]},
    {"item": "total actual cost", "value": STAGE2B_SUMMARY["total_actual_cost_usd"]},
    {"item": "cumulative monthly spend", "value": STAGE2B_SUMMARY["cumulative_monthly_spend_usd"]},
    {"item": "lifecycle counts", "value": STAGE2B_SUMMARY["lifecycle_counts"]},
])
display(stage2b_receipts)

artifact_rows = []
for call in STAGE2B_CALLS:
    prompt = Path(call["prompt_file"])
    response = Path(call["response_file"])
    artifact_rows.append({
        "call": call["position"],
        "prompt_exists": prompt.exists(),
        "response_exists": response.exists(),
        "prompt_path": str(prompt),
        "response_path": str(response),
        "summary_prompt_exists": call["prompt_file_exists"],
        "summary_response_exists": call["response_file_exists"],
    })
artifact_df = pd.DataFrame(artifact_rows)
display(artifact_df)

receipt_checks = [
    check_row("summary artifact exists", STAGE2B_SUMMARY_PATH.exists(), STAGE2B_SUMMARY_PATH),
    check_row("batch has exactly 5 calls", len(STAGE2B_CALLS) == 5),
    check_row("5 prompt files and 5 response files exist", bool(artifact_df["prompt_exists"].all() and artifact_df["response_exists"].all()), artifact_df),
    check_row("summary artifact paths agree with disk", bool((artifact_df["prompt_exists"] == artifact_df["summary_prompt_exists"]).all() and (artifact_df["response_exists"] == artifact_df["summary_response_exists"]).all()), artifact_df),
    check_row("summary says no truncation", STAGE2B_SUMMARY["truncated"] is False and STAGE2B_SUMMARY["unissued_slots"] == 0, {"truncated": STAGE2B_SUMMARY["truncated"], "unissued": STAGE2B_SUMMARY["unissued_slots"]}),
]
display_check_table(receipt_checks, "Stage 2b batch receipts and artifact checks")
D6_STAGE2B_ACCEPTANCE["batch_receipts_artifacts"] = True

,item,value
0,batch_id,cd2f32ba-1984-4461-8216-1a9ac4ca2c17
1,summary path,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...
2,batch size,5
3,batch cap,3.0
4,hypotheses_attempted,5
5,total estimated cost,0.062289
6,total actual cost,0.04401
7,cumulative monthly spend,0.061188
8,lifecycle counts,{'pending_backtest': 5}


,call,prompt_exists,response_exists,prompt_path,response_path,summary_prompt_exists,summary_response_exists
0,1,True,True,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,True,True
1,2,True,True,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,True,True
2,3,True,True,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,True,True
3,4,True,True,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,True,True
4,5,True,True,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...,True,True


Stage 2b batch receipts and artifact checks


,check,status,detail
0,summary artifact exists,PASS,raw_payloads/batch_cd2f32ba-1984-4461-8216-1a9...
1,batch has exactly 5 calls,PASS,
2,5 prompt files and 5 response files exist,PASS,call prompt_exists response_exists \ 0 ...
3,summary artifact paths agree with disk,PASS,call prompt_exists response_exists \ 0 ...
4,summary says no truncation,PASS,"{'truncated': False, 'unissued': 0}"


## T. Sequential Execution Audit From SQLite

This section independently checks sequencing from durable ledger timestamps. The requirement is stricter than “five rows exist”: each call's ledger row must be completed before the next call's pre-charge row is created.

In [21]:
def _parse_ledger_ts(value: str) -> pd.Timestamp:
    return pd.Timestamp(value.replace("Z", "+00:00"))

with sqlite3.connect(STAGE2B_LEDGER_PATH) as conn:
    conn.row_factory = sqlite3.Row
    ledger_rows_raw = conn.execute(
        "SELECT * FROM ledger WHERE batch_id = ? ORDER BY created_at_utc ASC",
        (STAGE2B_BATCH_ID,),
    ).fetchall()
ledger_rows = [dict(r) for r in ledger_rows_raw]
ledger_df = pd.DataFrame(ledger_rows)
ledger_df["call"] = range(1, len(ledger_df) + 1)
ledger_df["created_ts"] = ledger_df["created_at_utc"].map(_parse_ledger_ts)
ledger_df["completed_ts"] = ledger_df["completed_at_utc"].map(_parse_ledger_ts)
ledger_df["next_created_ts"] = ledger_df["created_ts"].shift(-1)
ledger_df["completed_before_next_created"] = ledger_df["next_created_ts"].isna() | (ledger_df["completed_ts"] <= ledger_df["next_created_ts"])

display(ledger_df[["call", "status", "estimated_cost", "actual_cost", "created_at_utc", "completed_at_utc", "completed_before_next_created"]])

sequential_checks = [
    check_row("ledger has 5 rows for this batch", len(ledger_df) == 5, len(ledger_df)),
    check_row("all rows completed", bool((ledger_df["status"] == STATUS_COMPLETED).all()), ledger_df[["call", "status"]]),
    check_row("created timestamps are strictly increasing", bool(ledger_df["created_ts"].is_monotonic_increasing and ledger_df["created_ts"].is_unique)),
    check_row("completed timestamps are strictly increasing", bool(ledger_df["completed_ts"].is_monotonic_increasing and ledger_df["completed_ts"].is_unique)),
    check_row("call k completes before call k+1 starts", bool(ledger_df["completed_before_next_created"].all()), ledger_df[["call", "completed_at_utc", "next_created_ts", "completed_before_next_created"]]),
]
display_check_table(sequential_checks, "Stage 2b sequential execution checks")
D6_STAGE2B_ACCEPTANCE["sequential_execution"] = True

,call,status,estimated_cost,actual_cost,created_at_utc,completed_at_utc,completed_before_next_created
0,1,completed,0.011475,0.007647,2026-04-17T14:27:10.664Z,2026-04-17T14:27:16.469Z,True
1,2,completed,0.011994,0.008214,2026-04-17T14:27:16.478Z,2026-04-17T14:27:20.811Z,True
2,3,completed,0.012537,0.008820,2026-04-17T14:27:20.815Z,2026-04-17T14:27:25.537Z,True
3,4,completed,0.013128,0.009393,2026-04-17T14:27:25.540Z,2026-04-17T14:27:29.958Z,True
4,5,completed,0.013155,0.009936,2026-04-17T14:27:29.964Z,2026-04-17T14:27:35.077Z,True


Stage 2b sequential execution checks


,check,status,detail
0,ledger has 5 rows for this batch,PASS,5
1,all rows completed,PASS,call status 0 1 completed 1 2 ...
2,created timestamps are strictly increasing,PASS,
3,completed timestamps are strictly increasing,PASS,
4,call k completes before call k+1 starts,PASS,call completed_at_utc ...


## U. Actual Prompt Audit For All 5 Calls

The important Stage 2b prompt risk is that approved examples could leak protected information or distort the prompt contract. This section audits the actual five prompt files.

In [22]:
EXPECTED_STAGE2B_THEMES = ["momentum", "mean_reversion", "volatility_regime", "volume_divergence", "calendar_effect"]
EXPECTED_APPROVED_EXAMPLE_COUNTS = [0, 1, 2, 3, 3]


def _prompt_example_jsons(prompt_text: str) -> list[dict]:
    examples = []
    in_block = False
    for line in prompt_text.splitlines():
        if "up to 3 example approved hypotheses" in line:
            in_block = True
            continue
        if in_block and line.startswith("Propose hypothesis"):
            break
        if in_block and line.strip().startswith("- {"):
            examples.append(json.loads(line.strip()[2:].strip()))
    return examples

prompt_audit_rows = []
prompt_examples_by_call: dict[int, list[dict]] = {}
for call in STAGE2B_CALLS:
    k = call["position"]
    prompt_path = Path(call["prompt_file"])
    text = prompt_path.read_text(encoding="utf-8")
    findings = audit_prompt_for_leakage(text)
    examples = _prompt_example_jsons(text)
    prompt_examples_by_call[k] = examples
    theme = call["theme"]
    prompt_audit_rows.append({
        "call": k,
        "expected_theme": EXPECTED_STAGE2B_THEMES[k - 1],
        "theme_in_prompt": theme if f"theme (rotating): {theme}" in text else "MISSING",
        "leakage_clean": findings == [],
        "leakage_findings": findings,
        "examples_count_in_prompt": len(examples),
        "summary_examples_count": call["approved_examples_count_in_prompt_before_call"],
        "factor_menu_present": "# Available factors" in text and "return_24h" in text,
        "operators_present": all(op in text for op in ALLOWED_OPS),
        "complexity_budget_present": "complexity budget" in text.lower() and "max_hold_bars" in text,
    })
prompt_audit_df = pd.DataFrame(prompt_audit_rows)
display(prompt_audit_df)

prompt_checks = [
    check_row("theme order is expected", prompt_audit_df["expected_theme"].tolist() == EXPECTED_STAGE2B_THEMES and prompt_audit_df["theme_in_prompt"].tolist() == EXPECTED_STAGE2B_THEMES, prompt_audit_df[["call", "expected_theme", "theme_in_prompt"]]),
    check_row("all 5 live prompts pass leakage audit", bool(prompt_audit_df["leakage_clean"].all()), prompt_audit_df[["call", "leakage_findings"]]),
    check_row("approved example counts are [0, 1, 2, 3, 3]", prompt_audit_df["examples_count_in_prompt"].tolist() == EXPECTED_APPROVED_EXAMPLE_COUNTS, prompt_audit_df[["call", "examples_count_in_prompt"]]),
    check_row("prompt example counts match summary", bool((prompt_audit_df["examples_count_in_prompt"] == prompt_audit_df["summary_examples_count"]).all()), prompt_audit_df[["call", "examples_count_in_prompt", "summary_examples_count"]]),
    check_row("factor menu present in all prompts", bool(prompt_audit_df["factor_menu_present"].all())),
    check_row("frozen operators present in all prompts", bool(prompt_audit_df["operators_present"].all())),
    check_row("complexity budget present in all prompts", bool(prompt_audit_df["complexity_budget_present"].all())),
]
display_check_table(prompt_checks, "Stage 2b prompt audit checks")
D6_STAGE2B_ACCEPTANCE["prompt_audit_all_calls"] = True

,call,expected_theme,theme_in_prompt,leakage_clean,leakage_findings,examples_count_in_prompt,summary_examples_count,factor_menu_present,operators_present,complexity_budget_present
0,1,momentum,momentum,True,[],0,0,True,True,True
1,2,mean_reversion,mean_reversion,True,[],1,1,True,True,True
2,3,volatility_regime,volatility_regime,True,[],2,2,True,True,True
3,4,volume_divergence,volume_divergence,True,[],3,3,True,True,True
4,5,calendar_effect,calendar_effect,True,[],3,3,True,True,True


Stage 2b prompt audit checks


,check,status,detail
0,theme order is expected,PASS,call expected_theme theme_in_prompt ...
1,all 5 live prompts pass leakage audit,PASS,call leakage_findings 0 1 ...
2,"approved example counts are [0, 1, 2, 3, 3]",PASS,call examples_count_in_prompt 0 1 ...
3,prompt example counts match summary,PASS,call examples_count_in_prompt summary_exa...
4,factor menu present in all prompts,PASS,
5,frozen operators present in all prompts,PASS,
6,complexity budget present in all prompts,PASS,


## V. Approved Examples Evolution Audit

This is the main Stage 2b semantic check. We reconstruct the examples embedded in each prompt and compare them to prior valid outputs, including the cap=3 sliding-window behavior.

In [23]:
import re
# Parse response DSLs with the same fence-stripping behavior used by the backend.
def _stage2b_response_json_text(raw_text: str) -> str:
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return fenced.group(1).strip() if fenced else raw_text.strip()

response_candidates_by_call = {}
response_dsl_by_call = {}
response_parse_rows = []
for call in STAGE2B_CALLS:
    k = call["position"]
    response_text = Path(call["response_file"]).read_text(encoding="utf-8")
    parse_text = _stage2b_response_json_text(response_text)
    cand = classify_raw_json(
        parse_text,
        registry=REGISTRY,
        error_kind_hint="stage2b_notebook_reclassify",
        provenance={"backend": "sonnet", "batch_id": STAGE2B_BATCH_ID, "position": k},
    )
    response_candidates_by_call[k] = cand
    is_valid = isinstance(cand, ValidCandidate)
    if is_valid:
        response_dsl_by_call[k] = cand.dsl
    response_parse_rows.append({
        "call": k,
        "fence_stripped": parse_text != response_text.strip(),
        "valid_candidate": is_valid,
        "name": cand.dsl.name if is_valid else None,
        "parse_error_excerpt": None if is_valid else getattr(cand, "parse_error", "")[:180],
    })

response_parse_df = pd.DataFrame(response_parse_rows)
display(response_parse_df)
assert response_parse_df["valid_candidate"].all(), response_parse_df

example_evolution_rows = []
for k in range(1, 6):
    actual_names = [ex.get("name") for ex in prompt_examples_by_call[k]]
    expected_prior_calls = list(range(max(1, k - 3), k))
    expected_names_chronological = [response_dsl_by_call[i].name for i in expected_prior_calls]
    expected_names_prompt_order = list(reversed(expected_names_chronological))
    actual_desc = [ex.get("description") for ex in prompt_examples_by_call[k]]
    expected_desc_prompt_order = [response_dsl_by_call[i].description for i in reversed(expected_prior_calls)]
    example_evolution_rows.append({
        "call": k,
        "actual_example_names": actual_names,
        "expected_prompt_order_names": expected_names_prompt_order,
        "actual_count": len(actual_names),
        "expected_count": min(k - 1, 3),
        "identity_match": actual_names == expected_names_prompt_order and actual_desc == expected_desc_prompt_order,
        "call1_slid_out_by_call5": (k == 5 and response_dsl_by_call[1].name not in actual_names) or k != 5,
    })
example_evolution_df = pd.DataFrame(example_evolution_rows)
display(example_evolution_df)

example_checks = [
    check_row("all prior responses parsed as valid DSLs", bool(response_parse_df["valid_candidate"].all()), response_parse_df),
    check_row("approved example counts follow [0,1,2,3,3]", example_evolution_df["actual_count"].tolist() == EXPECTED_APPROVED_EXAMPLE_COUNTS, example_evolution_df[["call", "actual_count"]]),
    check_row("prompt examples are exactly prior valid outputs in reverse-recency order", bool(example_evolution_df["identity_match"].all()), example_evolution_df[["call", "actual_example_names", "expected_prompt_order_names", "identity_match"]]),
    check_row("cap=3 sliding window removes call 1 by call 5", bool(example_evolution_df.loc[example_evolution_df["call"] == 5, "call1_slid_out_by_call5"].iloc[0]), example_evolution_df.loc[example_evolution_df["call"] == 5].to_dict(orient="records")),
]
display_check_table(example_checks, "Stage 2b approved_examples evolution checks")
D6_STAGE2B_ACCEPTANCE["approved_examples_evolution"] = True


,call,fence_stripped,valid_candidate,name,parse_error_excerpt
0,1,True,True,momentum_surge_rsi_breakout,None
1,2,True,True,bb_reversion_oversold,None
2,3,True,True,low_vol_breakout_continuation,None
3,4,True,True,volume_spike_momentum_entry,None
4,5,True,True,weekend_momentum_fade,None


,call,actual_example_names,expected_prompt_order_names,actual_count,expected_count,identity_match,call1_slid_out_by_call5
0,1,[],[],0,0,True,True
1,2,[momentum_surge_rsi_breakout],[momentum_surge_rsi_breakout],1,1,True,True
2,3,"[bb_reversion_oversold, momentum_surge_rsi_bre...","[bb_reversion_oversold, momentum_surge_rsi_bre...",2,2,True,True
3,4,"[low_vol_breakout_continuation, bb_reversion_o...","[low_vol_breakout_continuation, bb_reversion_o...",3,3,True,True
4,5,"[volume_spike_momentum_entry, low_vol_breakout...","[volume_spike_momentum_entry, low_vol_breakout...",3,3,True,True


Stage 2b approved_examples evolution checks


,check,status,detail
0,all prior responses parsed as valid DSLs,PASS,call fence_stripped valid_candidate ...
1,"approved example counts follow [0,1,2,3,3]",PASS,call actual_count 0 1 0 1 ...
2,prompt examples are exactly prior valid output...,PASS,call actual_e...
3,cap=3 sliding window removes call 1 by call 5,PASS,"[{'call': 5, 'actual_example_names': ['volume_..."


## W. Independent Response Re-Classification And Hash Uniqueness

This section re-runs the durable raw response files through the current classifier and ingest path. It does not trust the summary's lifecycle or hash fields.

In [24]:
reclass_state = BatchIngestState(batch_id=STAGE2B_BATCH_ID)
reclass_outputs = []
for call in STAGE2B_CALLS:
    k = call["position"]
    cand = response_candidates_by_call[k]
    reclass_outputs.append(ProposerOutput(
        candidates=(cand,),
        cost_estimate_usd=call["estimated_cost_usd"],
        cost_actual_usd=call["actual_cost_usd"],
        backend_name="sonnet",
        telemetry={"position": k, "batch_id": STAGE2B_BATCH_ID},
    ))
reclass_records = ingest_outputs(reclass_state, reclass_outputs)
assert_lifecycle_invariant_at_batch_close(reclass_state)

reclass_rows = []
for call, rec in zip(STAGE2B_CALLS, reclass_records):
    k = call["position"]
    cand = response_candidates_by_call[k]
    dsl = cand.dsl if isinstance(cand, ValidCandidate) else None
    factors = sorted({
        cond.factor
        for group in (list(dsl.entry) + list(dsl.exit))
        for cond in group.conditions
    }) if dsl is not None else []
    reclass_rows.append({
        "call": k,
        "schema_valid": isinstance(cand, ValidCandidate),
        "reclassified_lifecycle": rec.lifecycle_state,
        "summary_lifecycle": call["lifecycle_state"],
        "reclassified_hash": rec.hypothesis_hash,
        "summary_hash": call["hypothesis_hash"],
        "name": dsl.name if dsl else None,
        "factor_count": len(factors),
        "factors": factors,
        "cardinality_summary": call["cardinality"],
    })
reclass_df = pd.DataFrame(reclass_rows)
display(reclass_df)

hashes = reclass_df["reclassified_hash"].dropna().tolist()
reclass_checks = [
    check_row("all 5 raw responses re-validate as StrategyDSL", bool(reclass_df["schema_valid"].all()), reclass_df[["call", "schema_valid"]]),
    check_row("all 5 lifecycle states reclassify to pending_backtest", bool((reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST).all()), reclass_df[["call", "reclassified_lifecycle"]]),
    check_row("reclassified lifecycle matches summary", bool((reclass_df["reclassified_lifecycle"] == reclass_df["summary_lifecycle"]).all()), reclass_df[["call", "reclassified_lifecycle", "summary_lifecycle"]]),
    check_row("recomputed hashes match summary", bool((reclass_df["reclassified_hash"] == reclass_df["summary_hash"]).all()), reclass_df[["call", "reclassified_hash", "summary_hash"]]),
    check_row("all 5 hashes are non-empty and unique", len(hashes) == 5 and len(set(hashes)) == 5, hashes),
    check_row("all responses are reported as single_object", bool((reclass_df["cardinality_summary"] == "single_object").all()), reclass_df[["call", "cardinality_summary"]]),
]
display_check_table(reclass_checks, "Stage 2b independent response classification and dedup checks")
D6_STAGE2B_ACCEPTANCE["reclassification_hash_uniqueness"] = True

,call,schema_valid,reclassified_lifecycle,summary_lifecycle,reclassified_hash,summary_hash,name,factor_count,factors,cardinality_summary
0,1,True,pending_backtest,pending_backtest,2e41e74c0a798332,2e41e74c0a798332,momentum_surge_rsi_breakout,3,"[macd_hist, return_1h, rsi_14]",single_object
1,2,True,pending_backtest,pending_backtest,9ffba3930e100085,9ffba3930e100085,bb_reversion_oversold,3,"[close, rsi_14, zscore_48]",single_object
2,3,True,pending_backtest,pending_backtest,5c4429fa5fff2149,5c4429fa5fff2149,low_vol_breakout_continuation,3,"[close, realized_vol_24h, return_24h]",single_object
3,4,True,pending_backtest,pending_backtest,dc50c4c91ddf340c,dc50c4c91ddf340c,volume_spike_momentum_entry,3,"[return_1h, rsi_14, volume_zscore_24h]",single_object
4,5,True,pending_backtest,pending_backtest,d352876b38294d93,d352876b38294d93,weekend_momentum_fade,5,"[day_of_week, return_168h, return_24h, rsi_14,...",single_object


Stage 2b independent response classification and dedup checks


,check,status,detail
0,all 5 raw responses re-validate as StrategyDSL,PASS,call schema_valid 0 1 True 1 ...
1,all 5 lifecycle states reclassify to pending_b...,PASS,call reclassified_lifecycle 0 1 p...
2,reclassified lifecycle matches summary,PASS,call reclassified_lifecycle summary_lifecyc...
3,recomputed hashes match summary,PASS,call reclassified_hash summary_hash 0 ...
4,all 5 hashes are non-empty and unique,PASS,"[2e41e74c0a798332, 9ffba3930e100085, 5c4429fa5..."
5,all responses are reported as single_object,PASS,call cardinality_summary 0 1 sing...


## X. Cost And Ledger Reconciliation Audit

This section rebuilds the batch-level cost picture from SQLite rows and compares it to the summary artifact.

In [25]:
ledger_cost_df = ledger_df[["call", "estimated_cost", "actual_cost", "status", "created_at_utc", "completed_at_utc"]].copy()
summary_cost_df = pd.DataFrame([
    {
        "call": c["position"],
        "summary_estimated": c["estimated_cost_usd"],
        "summary_actual": c["actual_cost_usd"],
        "input_tokens": c["input_tokens"],
        "output_tokens": c["output_tokens"],
        "retry_count": c["retry_count"],
    }
    for c in STAGE2B_CALLS
])
cost_join_df = ledger_cost_df.merge(summary_cost_df, on="call", how="inner")
cost_join_df["estimated_matches"] = (cost_join_df["estimated_cost"] - cost_join_df["summary_estimated"]).abs() < 1e-12
cost_join_df["actual_matches"] = (cost_join_df["actual_cost"] - cost_join_df["summary_actual"]).abs() < 1e-12

display(cost_join_df)

total_estimated_from_ledger = float(cost_join_df["estimated_cost"].sum())
total_actual_from_ledger = float(cost_join_df["actual_cost"].sum())
pending_stage2b = BudgetLedger(STAGE2B_LEDGER_PATH).pending_entries(batch_id=STAGE2B_BATCH_ID)

cost_checks = [
    check_row("per-call ledger costs match summary", bool(cost_join_df["estimated_matches"].all() and cost_join_df["actual_matches"].all()), cost_join_df),
    check_row("batch total estimated cost matches summary", abs(total_estimated_from_ledger - STAGE2B_SUMMARY["total_estimated_cost_usd"]) < 1e-12, {"ledger": total_estimated_from_ledger, "summary": STAGE2B_SUMMARY["total_estimated_cost_usd"]}),
    check_row("batch total actual cost matches summary", abs(total_actual_from_ledger - STAGE2B_SUMMARY["total_actual_cost_usd"]) < 1e-12, {"ledger": total_actual_from_ledger, "summary": STAGE2B_SUMMARY["total_actual_cost_usd"]}),
    check_row("batch total actual cost stays under $3 cap", total_actual_from_ledger <= STAGE2B_BATCH_CAP_USD, total_actual_from_ledger),
    check_row("no pending ledger rows remain for Stage 2b batch", len(pending_stage2b) == 0, pending_stage2b),
    check_row("all calls had zero retries", bool((cost_join_df["retry_count"] == 0).all()), cost_join_df[["call", "retry_count"]]),
    check_row("estimator remains conservative", bool((cost_join_df["estimated_cost"] >= cost_join_df["actual_cost"]).all()), cost_join_df[["call", "estimated_cost", "actual_cost"]]),
]
display_check_table(cost_checks, "Stage 2b ledger and cost reconciliation checks")
D6_STAGE2B_ACCEPTANCE["ledger_cost_reconciliation"] = True

,call,estimated_cost,actual_cost,status,created_at_utc,completed_at_utc,summary_estimated,summary_actual,input_tokens,output_tokens,retry_count,estimated_matches,actual_matches
0,1,0.011475,0.007647,completed,2026-04-17T14:27:10.664Z,2026-04-17T14:27:16.469Z,0.011475,0.007647,1304,249,0,True,True
1,2,0.011994,0.008214,completed,2026-04-17T14:27:16.478Z,2026-04-17T14:27:20.811Z,0.011994,0.008214,1473,253,0,True,True
2,3,0.012537,0.008820,completed,2026-04-17T14:27:20.815Z,2026-04-17T14:27:25.537Z,0.012537,0.008820,1645,259,0,True,True
3,4,0.013128,0.009393,completed,2026-04-17T14:27:25.540Z,2026-04-17T14:27:29.958Z,0.013128,0.009393,1821,262,0,True,True
4,5,0.013155,0.009936,completed,2026-04-17T14:27:29.964Z,2026-04-17T14:27:35.077Z,0.013155,0.009936,1827,297,0,True,True


Stage 2b ledger and cost reconciliation checks


,check,status,detail
0,per-call ledger costs match summary,PASS,call estimated_cost actual_cost statu...
1,batch total estimated cost matches summary,PASS,"{'ledger': 0.062289, 'summary': 0.062289}"
2,batch total actual cost matches summary,PASS,"{'ledger': 0.04401, 'summary': 0.04401}"
3,batch total actual cost stays under $3 cap,PASS,0.04401
4,no pending ledger rows remain for Stage 2b batch,PASS,[]
5,all calls had zero retries,PASS,call retry_count 0 1 0 1 ...
6,estimator remains conservative,PASS,call estimated_cost actual_cost 0 1 ...


## Y. Observation Layer: Theme, Factor Usage, And Prompt Growth

These observations are not hard pass/fail gates. They are reviewer inputs for deciding how to proceed into Stage 2c.

In [26]:
observation_rows = []
for call in STAGE2B_CALLS:
    dsl = response_dsl_by_call[call["position"]]
    used = sorted({
        cond.factor
        for group in list(dsl.entry) + list(dsl.exit)
        for cond in group.conditions
    })
    observation_rows.append({
        "call": call["position"],
        "theme": call["theme"],
        "theme_adherence": call["theme_adherence"],
        "input_tokens": call["input_tokens"],
        "output_tokens": call["output_tokens"],
        "name": dsl.name,
        "factors_used": used,
        "description": dsl.description,
    })
observation_df = pd.DataFrame(observation_rows)
display(observation_df)

factor_usage_df = pd.DataFrame([
    {"factor": factor, "count": count}
    for factor, count in sorted(STAGE2B_SUMMARY["factor_usage"].items(), key=lambda kv: (-kv[1], kv[0]))
])
display(factor_usage_df)

observation_checks = [
    check_row("theme adherence distribution is recorded", STAGE2B_SUMMARY["theme_adherence_distribution"] == {"Yes": 1, "Partial": 4}, STAGE2B_SUMMARY["theme_adherence_distribution"]),
    check_row("input tokens increase then flatten after example cap", observation_df["input_tokens"].tolist() == sorted(observation_df["input_tokens"].tolist()) and observation_df["input_tokens"].iloc[-1] - observation_df["input_tokens"].iloc[-2] <= 10, observation_df[["call", "input_tokens"]]),
    check_row("rsi_14 appears in 4 of 5 summary factor counts", STAGE2B_SUMMARY["factor_usage"].get("rsi_14") == 4, STAGE2B_SUMMARY["factor_usage"]),
]
display_check_table(observation_checks, "Stage 2b observation checks", require_all=False)
D6_STAGE2B_ACCEPTANCE["observation_layer"] = True

,call,theme,theme_adherence,input_tokens,output_tokens,name,factors_used,description
0,1,momentum,Yes,1304,249,momentum_surge_rsi_breakout,"[macd_hist, return_1h, rsi_14]",Buy when RSI breaks above 50 and recent hourly...
1,2,mean_reversion,Partial,1473,253,bb_reversion_oversold,"[close, rsi_14, zscore_48]",Buy when price crosses below lower Bollinger B...
2,3,volatility_regime,Partial,1645,259,low_vol_breakout_continuation,"[close, realized_vol_24h, return_24h]",Buy when volatility compresses below average a...
3,4,volume_divergence,Partial,1821,262,volume_spike_momentum_entry,"[return_1h, rsi_14, volume_zscore_24h]",Buy when volume surges above 2 standard deviat...
4,5,calendar_effect,Partial,1827,297,weekend_momentum_fade,"[day_of_week, return_168h, return_24h, rsi_14,...",Buy on Sunday when price has declined over the...


,factor,count
0,rsi_14,4
1,close,2
2,return_1h,2
3,return_24h,2
4,zscore_48,2
5,bb_upper_24_2,1
6,day_of_week,1
7,ema_26,1
8,macd_hist,1
9,realized_vol_24h,1


Stage 2b observation checks


,check,status,detail
0,theme adherence distribution is recorded,PASS,"{'Yes': 1, 'Partial': 4}"
1,input tokens increase then flatten after examp...,PASS,call input_tokens 0 1 1304 1 ...
2,rsi_14 appears in 4 of 5 summary factor counts,PASS,"{'rsi_14': 4, 'return_1h': 2, 'macd_hist': 1, ..."


## Z. Stage 2b Verdict And 2c Recommendation Inputs

What Stage 2b proved:

- live 5-call batch completed under Stage 1 constraints
- theme rotation worked mechanically
- sequential approved-example feedback worked, including cap=3 sliding window
- all 5 outputs were schema-valid and ingestible
- no prompt leakage, accounting, dedup, or cardinality failures were observed
- cost stayed well below the $3 smoke-batch cap

What Stage 2b newly revealed:

- theme adherence is mixed: 1 full adherence, 4 partial
- prompt growth increased input tokens until approved-example cap flattened
- factor usage is broad but somewhat concentrated around `rsi_14`

Recommendation input: this is enough evidence for a human decision to allow Stage 2c. It is not a claim that hypothesis quality is proven.

In [27]:
stage2b_final_rows = [
    {"gate": "batch receipts + artifacts", "pass": D6_STAGE2B_ACCEPTANCE.get("batch_receipts_artifacts", False)},
    {"gate": "sequential execution", "pass": D6_STAGE2B_ACCEPTANCE.get("sequential_execution", False)},
    {"gate": "all prompt leakage audits", "pass": D6_STAGE2B_ACCEPTANCE.get("prompt_audit_all_calls", False)},
    {"gate": "approved_examples evolution", "pass": D6_STAGE2B_ACCEPTANCE.get("approved_examples_evolution", False)},
    {"gate": "response reclassification + hash uniqueness", "pass": D6_STAGE2B_ACCEPTANCE.get("reclassification_hash_uniqueness", False)},
    {"gate": "ledger/cost reconciliation", "pass": D6_STAGE2B_ACCEPTANCE.get("ledger_cost_reconciliation", False)},
]
stage2b_final_df = pd.DataFrame(stage2b_final_rows)
stage2b_final_df["status"] = stage2b_final_df["pass"].map({True: "PASS", False: "FAIL"})
display(stage2b_final_df[["gate", "status"]])

assert stage2b_final_df["pass"].all(), stage2b_final_df[~stage2b_final_df["pass"]]

print("D6 Stage 2b acceptance audit: PASS")
print("The notebook independently reconstructed sequential execution, prompt safety, approved_examples evolution, response classification, hash uniqueness, and ledger closure from durable artifacts.")
print("Recommendation input: sufficient evidence to consider proceeding to Stage 2c review.")

,gate,status
0,batch receipts + artifacts,PASS
1,sequential execution,PASS
2,all prompt leakage audits,PASS
3,approved_examples evolution,PASS
4,response reclassification + hash uniqueness,PASS
5,ledger/cost reconciliation,PASS


D6 Stage 2b acceptance audit: PASS
The notebook independently reconstructed sequential execution, prompt safety, approved_examples evolution, response classification, hash uniqueness, and ledger closure from durable artifacts.
Recommendation input: sufficient evidence to consider proceeding to Stage 2c review.


## AA. D6 Stage 2c Scope And Acceptance Framing

D6 Stage 2c is a **20-call live observation batch audit**. The goal is not to prove hypothesis quality; the goal is to independently reconstruct the operational facts that matter before moving deeper into Phase 2B:

- all 20 calls are truly strict sequential calls
- all 20 live prompts are leakage-clean despite repeated `approved_examples` injection
- the `approved_examples` sliding window is identity-correct, not just count-correct
- all 20 raw responses independently reclassify to `pending_backtest`
- all 20 hypothesis hashes are unique
- per-theme telemetry can be rebuilt from raw DSL factors and `THEME_HINTS`
- catastrophic stop conditions did not trigger

This section treats the CLI summary as a receipt, then rebuilds the key evidence from durable artifacts: raw prompt files, raw response files, the ledger database, and the Stage 2c summary JSON.


In [28]:
import json
import math
import os
import re
import sqlite3
import sys
from collections import Counter
from pathlib import Path

# Allow Stage 2c cells to run from a fresh kernel, from either repo root
# or the test notebooks/ folder, without requiring the Stage 1 setup cell.
_STAGE2C_START_CWD = Path.cwd().resolve()
_STAGE2C_REPO_ROOT = next(
    (p for p in (_STAGE2C_START_CWD, *_STAGE2C_START_CWD.parents) if (p / "pyproject.toml").exists() and (p / "agents").exists()),
    None,
)
if _STAGE2C_REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {_STAGE2C_START_CWD}")
if str(_STAGE2C_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_STAGE2C_REPO_ROOT))
os.chdir(_STAGE2C_REPO_ROOT)

import pandas as pd
from IPython.display import display

from agents.orchestrator.budget_ledger import STATUS_COMPLETED, STATUS_PENDING
from agents.orchestrator.ingest import (
    BatchIngestState,
    PENDING_BACKTEST,
    assert_lifecycle_invariant_at_batch_close,
    ingest_outputs,
)
from agents.proposer import ProposerOutput, ValidCandidate
from agents.proposer.prompt_builder import audit_prompt_for_leakage
from factors.registry import get_registry

REGISTRY = globals().get("REGISTRY", get_registry())
ALLOWED_OPS = globals().get("ALLOWED_OPS", ("<", "<=", ">", ">=", "==", "crosses_above", "crosses_below"))

if "check_row" not in globals():
    def status(ok: bool) -> str:
        return "PASS" if bool(ok) else "FAIL"

    def check_row(check: str, ok: bool, detail="") -> dict:
        return {"check": check, "status": status(ok), "detail": detail}

if "display_check_table" not in globals():
    def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
        print(title)
        df = pd.DataFrame(rows)
        display(df)
        if require_all:
            failed = df[df["status"] != "PASS"]
            assert failed.empty, failed
        return df

print("Stage 2c repo root:", _STAGE2C_REPO_ROOT)

from agents.proposer.stage2c_batch import (
    CARDINALITY_VIOLATION_STOP as STAGE2C_CARDINALITY_VIOLATION_STOP,
    DEFAULT_MOMENTUM_FACTORS as STAGE2C_DEFAULT_MOMENTUM_FACTORS,
    PARSE_RATE_MIN_K as STAGE2C_PARSE_RATE_MIN_K,
    PARSE_RATE_THRESHOLD as STAGE2C_PARSE_RATE_THRESHOLD,
    STAGE2C_BATCH_SIZE,
    STAGE2C_CUMULATIVE_CAP_USD,
    THEME_CYCLE_LEN as STAGE2C_THEME_CYCLE_LEN,
    THEME_HINTS as STAGE2C_THEME_HINTS,
)
from agents.proposer.stub_backend import classify_raw_json
from agents.themes import THEMES as STAGE2C_ALL_THEMES

D6_STAGE2C_ACCEPTANCE: dict[str, bool] = {}
STAGE2C_LEDGER_PATH = Path("agents/spend_ledger.db")
STAGE2C_SUMMARY_CANDIDATES = sorted(
    Path("raw_payloads").glob("batch_*/stage2c_summary.json"),
    key=lambda path: path.stat().st_mtime,
)
assert STAGE2C_SUMMARY_CANDIDATES, "No stage2c_summary.json artifacts found under raw_payloads/"
STAGE2C_SUMMARY_PATH = STAGE2C_SUMMARY_CANDIDATES[-1]
STAGE2C_BATCH_DIR = STAGE2C_SUMMARY_PATH.parent
STAGE2C_BATCH_ID = STAGE2C_BATCH_DIR.name.removeprefix("batch_")
STAGE2C_SUMMARY = json.loads(STAGE2C_SUMMARY_PATH.read_text(encoding="utf-8"))
STAGE2C_CALLS = sorted(STAGE2C_SUMMARY["calls"], key=lambda row: row["position"])
STAGE2C_CONFIG = STAGE2C_SUMMARY["config"]
STAGE2C_EXPECTED_THEMES = list(STAGE2C_ALL_THEMES)[:STAGE2C_THEME_CYCLE_LEN]

receipt_rows = [
    {"item": "batch_id", "value": STAGE2C_BATCH_ID},
    {"item": "summary path", "value": str(STAGE2C_SUMMARY_PATH)},
    {"item": "git commit", "value": STAGE2C_CONFIG.get("git_commit")},
    {"item": "run timestamp UTC", "value": STAGE2C_CONFIG.get("run_timestamp_utc")},
    {"item": "model", "value": STAGE2C_CONFIG.get("model_name")},
    {"item": "batch size", "value": STAGE2C_CONFIG.get("batch_size")},
    {"item": "prompt caching enabled", "value": STAGE2C_CONFIG.get("prompt_caching_enabled")},
    {"item": "total estimated cost", "value": STAGE2C_SUMMARY.get("total_estimated_cost_usd")},
    {"item": "total actual cost", "value": STAGE2C_SUMMARY.get("total_actual_cost_usd")},
    {"item": "cumulative monthly spend", "value": STAGE2C_SUMMARY.get("cumulative_monthly_spend_usd")},
    {"item": "early stop reason", "value": STAGE2C_SUMMARY.get("early_stop_reason")},
    {"item": "truncated", "value": STAGE2C_SUMMARY.get("truncated")},
]
receipt_df = pd.DataFrame(receipt_rows)
display(receipt_df)

artifact_rows = []
for call in STAGE2C_CALLS:
    k = call["position"]
    prompt_path = Path(call["prompt_file"])
    response_path = Path(call["response_file"])
    artifact_rows.append({
        "call": k,
        "theme": call["theme"],
        "prompt_exists": prompt_path.exists(),
        "response_exists": response_path.exists(),
        "prompt_path": str(prompt_path),
        "response_path": str(response_path),
    })
stage2c_artifact_df = pd.DataFrame(artifact_rows)
display(stage2c_artifact_df)

stage2c_receipt_checks = [
    check_row("latest Stage 2c summary artifact exists", STAGE2C_SUMMARY_PATH.exists(), STAGE2C_SUMMARY_PATH),
    check_row("summary batch id matches directory", STAGE2C_SUMMARY.get("batch_id") == STAGE2C_BATCH_ID, {"summary": STAGE2C_SUMMARY.get("batch_id"), "dir": STAGE2C_BATCH_ID}),
    check_row("summary contains 20 calls", len(STAGE2C_CALLS) == STAGE2C_BATCH_SIZE, len(STAGE2C_CALLS)),
    check_row("all 20 prompt artifacts exist", bool(stage2c_artifact_df["prompt_exists"].all()), stage2c_artifact_df.loc[~stage2c_artifact_df["prompt_exists"]]),
    check_row("all 20 response artifacts exist", bool(stage2c_artifact_df["response_exists"].all()), stage2c_artifact_df.loc[~stage2c_artifact_df["response_exists"]]),
    check_row("prompt caching disabled", STAGE2C_CONFIG.get("prompt_caching_enabled") is False, STAGE2C_CONFIG.get("prompt_caching_enabled")),
]
display_check_table(stage2c_receipt_checks, "Stage 2c batch receipts and artifact checks")
D6_STAGE2C_ACCEPTANCE["batch_receipts_artifacts"] = True


Stage 2c repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline


,item,value
0,batch_id,e07f34a2-b532-4f35-a9f3-af97a5a96f1f
1,summary path,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
2,git commit,3b59268
3,run timestamp UTC,2026-04-17T17:46:05.298635+00:00
4,model,claude-sonnet-4-5
5,batch size,20
6,prompt caching enabled,False
7,total estimated cost,0.262776
8,total actual cost,0.195129
9,cumulative monthly spend,0.256317


,call,theme,prompt_exists,response_exists,prompt_path,response_path
0,1,momentum,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
1,2,mean_reversion,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
2,3,volatility_regime,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
3,4,volume_divergence,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
4,5,calendar_effect,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
5,6,momentum,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
6,7,mean_reversion,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
7,8,volatility_regime,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
8,9,volume_divergence,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
9,10,calendar_effect,True,True,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...


Stage 2c batch receipts and artifact checks


,check,status,detail
0,latest Stage 2c summary artifact exists,PASS,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...
1,summary batch id matches directory,PASS,{'summary': 'e07f34a2-b532-4f35-a9f3-af97a5a96...
2,summary contains 20 calls,PASS,20
3,all 20 prompt artifacts exist,PASS,"Empty DataFrame Columns: [call, theme, prompt_..."
4,all 20 response artifacts exist,PASS,"Empty DataFrame Columns: [call, theme, prompt_..."
5,prompt caching disabled,PASS,False


## AB. Strict Sequential Execution Audit

Stage 2c only proves sequential batch semantics if call `k` fully completes before call `k+1` starts. This section checks the ledger timestamps directly with the strict relation:

`completed_at(call k) < created_at(call k+1)`


In [29]:
def _stage2c_parse_ledger_ts(value: str) -> pd.Timestamp:
    return pd.Timestamp(str(value).replace("Z", "+00:00"))


def _stage2c_connect_ledger_readonly():
    return sqlite3.connect(f"file:{STAGE2C_LEDGER_PATH.resolve()}?mode=ro", uri=True)


with _stage2c_connect_ledger_readonly() as conn:
    conn.row_factory = sqlite3.Row
    stage2c_ledger_rows_raw = conn.execute(
        "SELECT * FROM ledger WHERE batch_id = ? ORDER BY created_at_utc ASC",
        (STAGE2C_BATCH_ID,),
    ).fetchall()
stage2c_ledger_rows = [dict(row) for row in stage2c_ledger_rows_raw]
stage2c_ledger_df = pd.DataFrame(stage2c_ledger_rows)
stage2c_ledger_df["call"] = range(1, len(stage2c_ledger_df) + 1)
stage2c_ledger_df["created_ts"] = stage2c_ledger_df["created_at_utc"].map(_stage2c_parse_ledger_ts)
stage2c_ledger_df["completed_ts"] = stage2c_ledger_df["completed_at_utc"].map(_stage2c_parse_ledger_ts)
stage2c_ledger_df["next_created_ts"] = stage2c_ledger_df["created_ts"].shift(-1)
stage2c_ledger_df["completed_strictly_before_next_created"] = (
    stage2c_ledger_df["next_created_ts"].isna()
    | (stage2c_ledger_df["completed_ts"] < stage2c_ledger_df["next_created_ts"])
)

display(stage2c_ledger_df[[
    "call", "status", "estimated_cost", "actual_cost",
    "created_at_utc", "completed_at_utc", "completed_strictly_before_next_created",
]])

stage2c_sequential_checks = [
    check_row("ledger has 20 rows for this batch", len(stage2c_ledger_df) == STAGE2C_BATCH_SIZE, len(stage2c_ledger_df)),
    check_row("all rows completed", bool((stage2c_ledger_df["status"] == STATUS_COMPLETED).all()), stage2c_ledger_df[["call", "status"]]),
    check_row("created timestamps are strictly increasing", bool(stage2c_ledger_df["created_ts"].is_monotonic_increasing and stage2c_ledger_df["created_ts"].is_unique)),
    check_row("completed timestamps are strictly increasing", bool(stage2c_ledger_df["completed_ts"].is_monotonic_increasing and stage2c_ledger_df["completed_ts"].is_unique)),
    check_row("call k completes strictly before call k+1 starts", bool(stage2c_ledger_df["completed_strictly_before_next_created"].all()), stage2c_ledger_df[["call", "completed_at_utc", "next_created_ts", "completed_strictly_before_next_created"]]),
]
display_check_table(stage2c_sequential_checks, "Stage 2c strict sequential execution checks")
D6_STAGE2C_ACCEPTANCE["strict_sequential_execution"] = True


,call,status,estimated_cost,actual_cost,created_at_utc,completed_at_utc,completed_strictly_before_next_created
0,1,completed,0.011475,0.006954,2026-04-17T17:46:05.350Z,2026-04-17T17:46:09.694Z,True
1,2,completed,0.011916,0.007767,2026-04-17T17:46:09.696Z,2026-04-17T17:46:14.246Z,True
2,3,completed,0.012432,0.008385,2026-04-17T17:46:14.248Z,2026-04-17T17:46:19.138Z,True
3,4,completed,0.012984,0.009414,2026-04-17T17:46:19.141Z,2026-04-17T17:46:23.950Z,True
4,5,completed,0.013116,0.009777,2026-04-17T17:46:23.952Z,2026-04-17T17:46:29.270Z,True
5,6,completed,0.013236,0.009621,2026-04-17T17:46:29.272Z,2026-04-17T17:46:33.908Z,True
6,7,completed,0.013296,0.009819,2026-04-17T17:46:33.910Z,2026-04-17T17:46:38.454Z,True
7,8,completed,0.013338,0.009882,2026-04-17T17:46:38.457Z,2026-04-17T17:46:43.532Z,True
8,9,completed,0.013341,0.009825,2026-04-17T17:46:43.535Z,2026-04-17T17:46:48.106Z,True
9,10,completed,0.013353,0.010071,2026-04-17T17:46:48.108Z,2026-04-17T17:46:55.121Z,True


Stage 2c strict sequential execution checks


,check,status,detail
0,ledger has 20 rows for this batch,PASS,20
1,all rows completed,PASS,call status 0 1 completed 1 ...
2,created timestamps are strictly increasing,PASS,
3,completed timestamps are strictly increasing,PASS,
4,call k completes strictly before call k+1 starts,PASS,call completed_at_utc ...


## AC. Prompt Leakage, Theme Rotation, And Example-Count Audit

This section audits all 20 actual live prompt files. The important Stage 2c risk is that a longer sequential batch repeatedly injects approved examples; every prompt still has to remain inside the frozen search-space boundary.


In [30]:
def _stage2c_prompt_example_jsons(prompt_text: str) -> list[dict]:
    examples = []
    in_block = False
    for line in prompt_text.splitlines():
        if "up to 3 example approved hypotheses" in line:
            in_block = True
            continue
        if in_block and line.startswith("Propose hypothesis"):
            break
        if in_block and line.strip().startswith("- {"):
            examples.append(json.loads(line.strip()[2:].strip()))
    return examples

stage2c_prompt_examples_by_call: dict[int, list[dict]] = {}
stage2c_prompt_text_by_call: dict[int, str] = {}
stage2c_prompt_audit_rows = []
for call in STAGE2C_CALLS:
    k = call["position"]
    prompt_path = Path(call["prompt_file"])
    text = prompt_path.read_text(encoding="utf-8")
    examples = _stage2c_prompt_example_jsons(text)
    findings = audit_prompt_for_leakage(text)
    expected_theme = STAGE2C_EXPECTED_THEMES[(k - 1) % STAGE2C_THEME_CYCLE_LEN]
    stage2c_prompt_examples_by_call[k] = examples
    stage2c_prompt_text_by_call[k] = text
    stage2c_prompt_audit_rows.append({
        "call": k,
        "summary_theme": call["theme"],
        "expected_theme": expected_theme,
        "theme_in_prompt": expected_theme if f"theme (rotating): {expected_theme}" in text else "MISSING",
        "leakage_clean": findings == [],
        "leakage_findings": findings,
        "examples_count_in_prompt": len(examples),
        "summary_examples_count": call["approved_examples_count_in_prompt_before_call"],
        "factor_menu_present": "# Available factors" in text and "volume_zscore_24h" in text,
        "operators_present": all(op in text for op in ALLOWED_OPS),
        "complexity_budget_present": "complexity budget" in text.lower() and "max_hold_bars" in text,
    })
stage2c_prompt_audit_df = pd.DataFrame(stage2c_prompt_audit_rows)
display(stage2c_prompt_audit_df)

expected_theme_sequence = [STAGE2C_EXPECTED_THEMES[(k - 1) % STAGE2C_THEME_CYCLE_LEN] for k in range(1, STAGE2C_BATCH_SIZE + 1)]
expected_example_counts = [min(k - 1, 3) for k in range(1, STAGE2C_BATCH_SIZE + 1)]

stage2c_prompt_checks = [
    check_row("theme rotation is interleaved cyclic", stage2c_prompt_audit_df["expected_theme"].tolist() == expected_theme_sequence and stage2c_prompt_audit_df["summary_theme"].tolist() == expected_theme_sequence and stage2c_prompt_audit_df["theme_in_prompt"].tolist() == expected_theme_sequence, stage2c_prompt_audit_df[["call", "summary_theme", "expected_theme", "theme_in_prompt"]]),
    check_row("all 20 live prompts pass leakage audit", bool(stage2c_prompt_audit_df["leakage_clean"].all()), stage2c_prompt_audit_df[["call", "leakage_findings"]]),
    check_row("approved example counts follow cap=3", stage2c_prompt_audit_df["examples_count_in_prompt"].tolist() == expected_example_counts, stage2c_prompt_audit_df[["call", "examples_count_in_prompt"]]),
    check_row("prompt example counts match summary", bool((stage2c_prompt_audit_df["examples_count_in_prompt"] == stage2c_prompt_audit_df["summary_examples_count"]).all()), stage2c_prompt_audit_df[["call", "examples_count_in_prompt", "summary_examples_count"]]),
    check_row("factor menu present in all prompts", bool(stage2c_prompt_audit_df["factor_menu_present"].all())),
    check_row("frozen operators present in all prompts", bool(stage2c_prompt_audit_df["operators_present"].all())),
    check_row("complexity budget present in all prompts", bool(stage2c_prompt_audit_df["complexity_budget_present"].all())),
]
display_check_table(stage2c_prompt_checks, "Stage 2c prompt audit checks")
D6_STAGE2C_ACCEPTANCE["prompt_audit_all_calls"] = True


,call,summary_theme,expected_theme,theme_in_prompt,leakage_clean,leakage_findings,examples_count_in_prompt,summary_examples_count,factor_menu_present,operators_present,complexity_budget_present
0,1,momentum,momentum,momentum,True,[],0,0,True,True,True
1,2,mean_reversion,mean_reversion,mean_reversion,True,[],1,1,True,True,True
2,3,volatility_regime,volatility_regime,volatility_regime,True,[],2,2,True,True,True
3,4,volume_divergence,volume_divergence,volume_divergence,True,[],3,3,True,True,True
4,5,calendar_effect,calendar_effect,calendar_effect,True,[],3,3,True,True,True
5,6,momentum,momentum,momentum,True,[],3,3,True,True,True
6,7,mean_reversion,mean_reversion,mean_reversion,True,[],3,3,True,True,True
7,8,volatility_regime,volatility_regime,volatility_regime,True,[],3,3,True,True,True
8,9,volume_divergence,volume_divergence,volume_divergence,True,[],3,3,True,True,True
9,10,calendar_effect,calendar_effect,calendar_effect,True,[],3,3,True,True,True


Stage 2c prompt audit checks


,check,status,detail
0,theme rotation is interleaved cyclic,PASS,call summary_theme expected_theme...
1,all 20 live prompts pass leakage audit,PASS,call leakage_findings 0 1 ...
2,approved example counts follow cap=3,PASS,call examples_count_in_prompt 0 1 ...
3,prompt example counts match summary,PASS,call examples_count_in_prompt summary_ex...
4,factor menu present in all prompts,PASS,
5,frozen operators present in all prompts,PASS,
6,complexity budget present in all prompts,PASS,


## AD. Independent Response Reclassification And Hash Uniqueness

The summary says all 20 responses are `pending_backtest`. This section re-reads the raw response files, strips markdown fences when present, runs the same raw JSON classifier, re-ingests the candidates into a fresh batch state, and compares lifecycle states and hashes back to the summary.


In [31]:
def _stage2c_response_json_text(raw_text: str) -> str:
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return fenced.group(1).strip() if fenced else raw_text.strip()


def _stage2c_classify_cardinality(raw_text: str) -> str:
    stripped = _stage2c_response_json_text(raw_text)
    if not stripped.strip():
        return "zero_objects"
    try:
        parsed = json.loads(stripped)
    except json.JSONDecodeError:
        if "{" in raw_text or "[" in raw_text:
            return "prose_plus_object"
        return "zero_objects"
    if isinstance(parsed, dict):
        return "single_object"
    if isinstance(parsed, list):
        return f"json_array_{len(parsed)}"
    return "single_object"


def _stage2c_extract_factors_from_dsl(dsl) -> list[str]:
    factors = set()
    for group in list(dsl.entry) + list(dsl.exit):
        for cond in group.conditions:
            factors.add(cond.factor)
            if isinstance(cond.value, str):
                factors.add(cond.value)
    return sorted(factors)

stage2c_response_candidates_by_call = {}
stage2c_response_dsl_by_call = {}
stage2c_response_parse_rows = []
stage2c_cardinality_by_call = {}
for call in STAGE2C_CALLS:
    k = call["position"]
    raw_text = Path(call["response_file"]).read_text(encoding="utf-8")
    parse_text = _stage2c_response_json_text(raw_text)
    cardinality = _stage2c_classify_cardinality(raw_text)
    cand = classify_raw_json(
        parse_text,
        registry=REGISTRY,
        error_kind_hint="stage2c_notebook_reclassify",
        provenance={"backend": "sonnet", "batch_id": STAGE2C_BATCH_ID, "position": k},
    )
    stage2c_response_candidates_by_call[k] = cand
    stage2c_cardinality_by_call[k] = cardinality
    is_valid = isinstance(cand, ValidCandidate)
    if is_valid:
        stage2c_response_dsl_by_call[k] = cand.dsl
    stage2c_response_parse_rows.append({
        "call": k,
        "fence_stripped": parse_text != raw_text.strip(),
        "valid_candidate": is_valid,
        "name": cand.dsl.name if is_valid else None,
        "cardinality_recomputed": cardinality,
        "cardinality_summary": call["cardinality"],
        "parse_error_excerpt": None if is_valid else getattr(cand, "parse_error", "")[:180],
    })
stage2c_response_parse_df = pd.DataFrame(stage2c_response_parse_rows)
display(stage2c_response_parse_df)

stage2c_reclass_state = BatchIngestState(batch_id=STAGE2C_BATCH_ID)
stage2c_reclass_outputs = []
for call in STAGE2C_CALLS:
    k = call["position"]
    cand = stage2c_response_candidates_by_call[k]
    stage2c_reclass_outputs.append(ProposerOutput(
        candidates=(cand,),
        cost_estimate_usd=call["estimated_cost_usd"],
        cost_actual_usd=call["actual_cost_usd"],
        backend_name="sonnet",
        telemetry={"position": k, "batch_id": STAGE2C_BATCH_ID},
    ))
stage2c_reclass_records = ingest_outputs(stage2c_reclass_state, stage2c_reclass_outputs)
assert_lifecycle_invariant_at_batch_close(stage2c_reclass_state)

stage2c_reclass_rows = []
for call, rec in zip(STAGE2C_CALLS, stage2c_reclass_records):
    k = call["position"]
    cand = stage2c_response_candidates_by_call[k]
    dsl = cand.dsl if isinstance(cand, ValidCandidate) else None
    factors = _stage2c_extract_factors_from_dsl(dsl) if dsl is not None else []
    stage2c_reclass_rows.append({
        "call": k,
        "theme": call["theme"],
        "schema_valid": isinstance(cand, ValidCandidate),
        "reclassified_lifecycle": rec.lifecycle_state,
        "summary_lifecycle": call["lifecycle_state"],
        "reclassified_hash": rec.hypothesis_hash,
        "summary_hash": call["hypothesis_hash"],
        "name": dsl.name if dsl else None,
        "factor_count": len(factors),
        "factors": factors,
        "summary_factors": call["factors_used"],
        "cardinality_recomputed": stage2c_cardinality_by_call[k],
        "cardinality_summary": call["cardinality"],
    })
stage2c_reclass_df = pd.DataFrame(stage2c_reclass_rows)
display(stage2c_reclass_df)

stage2c_hashes = stage2c_reclass_df["reclassified_hash"].dropna().tolist()
stage2c_reclass_checks = [
    check_row("all 20 raw responses re-validate as StrategyDSL", bool(stage2c_reclass_df["schema_valid"].all()), stage2c_reclass_df[["call", "schema_valid"]]),
    check_row("all 20 lifecycle states reclassify to pending_backtest", bool((stage2c_reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST).all()), stage2c_reclass_df[["call", "reclassified_lifecycle"]]),
    check_row("reclassified lifecycle matches summary", bool((stage2c_reclass_df["reclassified_lifecycle"] == stage2c_reclass_df["summary_lifecycle"]).all()), stage2c_reclass_df[["call", "reclassified_lifecycle", "summary_lifecycle"]]),
    check_row("recomputed hashes match summary", bool((stage2c_reclass_df["reclassified_hash"] == stage2c_reclass_df["summary_hash"]).all()), stage2c_reclass_df[["call", "reclassified_hash", "summary_hash"]]),
    check_row("all 20 hashes are non-empty and unique", len(stage2c_hashes) == STAGE2C_BATCH_SIZE and len(set(stage2c_hashes)) == STAGE2C_BATCH_SIZE, stage2c_hashes),
    check_row("all responses are single JSON objects", bool((stage2c_reclass_df["cardinality_recomputed"] == "single_object").all()), stage2c_reclass_df[["call", "cardinality_recomputed"]]),
    check_row("recomputed cardinality matches summary", bool((stage2c_reclass_df["cardinality_recomputed"] == stage2c_reclass_df["cardinality_summary"]).all()), stage2c_reclass_df[["call", "cardinality_recomputed", "cardinality_summary"]]),
    check_row("recomputed factor lists match summary", bool((stage2c_reclass_df["factors"].map(tuple) == stage2c_reclass_df["summary_factors"].map(tuple)).all()), stage2c_reclass_df[["call", "factors", "summary_factors"]]),
]
display_check_table(stage2c_reclass_checks, "Stage 2c independent response classification and dedup checks")
D6_STAGE2C_ACCEPTANCE["reclassification_hash_uniqueness"] = True


,call,fence_stripped,valid_candidate,name,cardinality_recomputed,cardinality_summary,parse_error_excerpt
0,1,True,True,momentum_rsi_oversold_reversal,single_object,single_object,None
1,2,True,True,bb_mean_reversion_vol_spike,single_object,single_object,None
2,3,True,True,vol_regime_breakout_fade,single_object,single_object,None
3,4,True,True,volume_surge_momentum_fade,single_object,single_object,None
4,5,True,True,monday_effect_momentum_long,single_object,single_object,None
5,6,True,True,macd_momentum_acceleration,single_object,single_object,None
6,7,True,True,bb_reversal_oversold_bounce,single_object,single_object,None
7,8,True,True,low_vol_breakout_entry,single_object,single_object,None
8,9,True,True,volume_divergence_reversal,single_object,single_object,None
9,10,True,True,monday_dip_buy,single_object,single_object,None


,call,theme,schema_valid,reclassified_lifecycle,summary_lifecycle,reclassified_hash,summary_hash,name,factor_count,factors,summary_factors,cardinality_recomputed,cardinality_summary
0,1,momentum,True,pending_backtest,pending_backtest,3259e128589962f3,3259e128589962f3,momentum_rsi_oversold_reversal,3,"[close, ema_12, rsi_14]","[close, ema_12, rsi_14]",single_object,single_object
1,2,mean_reversion,True,pending_backtest,pending_backtest,33e46e3d182cffea,33e46e3d182cffea,bb_mean_reversion_vol_spike,5,"[bb_upper_24_2, close, realized_vol_24h, sma_2...","[bb_upper_24_2, close, realized_vol_24h, sma_2...",single_object,single_object
2,3,volatility_regime,True,pending_backtest,pending_backtest,f1e4840ef8ead53c,f1e4840ef8ead53c,vol_regime_breakout_fade,5,"[bb_upper_24_2, close, realized_vol_24h, rsi_1...","[bb_upper_24_2, close, realized_vol_24h, rsi_1...",single_object,single_object
3,4,volume_divergence,True,pending_backtest,pending_backtest,f507bdc2f601b422,f507bdc2f601b422,volume_surge_momentum_fade,4,"[close, rsi_14, sma_20, volume_zscore_24h]","[close, rsi_14, sma_20, volume_zscore_24h]",single_object,single_object
4,5,calendar_effect,True,pending_backtest,pending_backtest,0057cb47ad57f718,0057cb47ad57f718,monday_effect_momentum_long,4,"[day_of_week, hour_of_day, return_1h, return_24h]","[day_of_week, hour_of_day, return_1h, return_24h]",single_object,single_object
5,6,momentum,True,pending_backtest,pending_backtest,211be27c38c6846c,211be27c38c6846c,macd_momentum_acceleration,4,"[macd_hist, return_1h, return_24h, rsi_14]","[macd_hist, return_1h, return_24h, rsi_14]",single_object,single_object
6,7,mean_reversion,True,pending_backtest,pending_backtest,d6ff600860cd6428,d6ff600860cd6428,bb_reversal_oversold_bounce,5,"[bb_upper_24_2, close, return_24h, rsi_14, sma...","[bb_upper_24_2, close, return_24h, rsi_14, sma...",single_object,single_object
7,8,volatility_regime,True,pending_backtest,pending_backtest,c455b106941fbb0f,c455b106941fbb0f,low_vol_breakout_entry,4,"[close, realized_vol_24h, sma_20, volume_zscor...","[close, realized_vol_24h, sma_20, volume_zscor...",single_object,single_object
8,9,volume_divergence,True,pending_backtest,pending_backtest,504f7b0bb5b1ce3c,504f7b0bb5b1ce3c,volume_divergence_reversal,3,"[return_24h, rsi_14, volume_zscore_24h]","[return_24h, rsi_14, volume_zscore_24h]",single_object,single_object
9,10,calendar_effect,True,pending_backtest,pending_backtest,827f5c9e3115346a,827f5c9e3115346a,monday_dip_buy,4,"[day_of_week, hour_of_day, return_1h, return_24h]","[day_of_week, hour_of_day, return_1h, return_24h]",single_object,single_object


Stage 2c independent response classification and dedup checks


,check,status,detail
0,all 20 raw responses re-validate as StrategyDSL,PASS,call schema_valid 0 1 True ...
1,all 20 lifecycle states reclassify to pending_...,PASS,call reclassified_lifecycle 0 1 ...
2,reclassified lifecycle matches summary,PASS,call reclassified_lifecycle summary_lifecy...
3,recomputed hashes match summary,PASS,call reclassified_hash summary_hash 0...
4,all 20 hashes are non-empty and unique,PASS,"[3259e128589962f3, 33e46e3d182cffea, f1e4840ef..."
5,all responses are single JSON objects,PASS,call cardinality_recomputed 0 1 ...
6,recomputed cardinality matches summary,PASS,call cardinality_recomputed cardinality_su...
7,recomputed factor lists match summary,PASS,call ...


## AE. Approved Examples Sliding-Window Identity Audit

Counts are not enough here. Stage 2c must prove that each prompt contains the most recent three valid prior DSLs, most-recent first, and that older examples are actually evicted. This section reconstructs that identity-level sequence from the response DSLs and prompt JSON snippets.


In [32]:
stage2c_example_evolution_rows = []
for k in range(1, STAGE2C_BATCH_SIZE + 1):
    actual_examples = stage2c_prompt_examples_by_call[k]
    actual_names = [ex.get("name") for ex in actual_examples]
    actual_desc = [ex.get("description") for ex in actual_examples]
    expected_prior_calls = list(range(max(1, k - 3), k))
    expected_calls_prompt_order = list(reversed(expected_prior_calls))
    expected_names_prompt_order = [stage2c_response_dsl_by_call[i].name for i in expected_calls_prompt_order]
    expected_desc_prompt_order = [stage2c_response_dsl_by_call[i].description for i in expected_calls_prompt_order]
    stage2c_example_evolution_rows.append({
        "call": k,
        "expected_prior_calls_prompt_order": expected_calls_prompt_order,
        "actual_example_names": actual_names,
        "expected_prompt_order_names": expected_names_prompt_order,
        "actual_count": len(actual_names),
        "expected_count": min(k - 1, 3),
        "identity_match": actual_names == expected_names_prompt_order and actual_desc == expected_desc_prompt_order,
        "summary_count": STAGE2C_SUMMARY["approved_examples_trace"][k - 1]["count_in_prompt"],
        "summary_names": STAGE2C_SUMMARY["approved_examples_trace"][k - 1]["names"],
    })
stage2c_example_evolution_df = pd.DataFrame(stage2c_example_evolution_rows)
display(stage2c_example_evolution_df)

stage2c_spotlight_examples_df = stage2c_example_evolution_df[stage2c_example_evolution_df["call"].isin([10, 15, 20])]
display(stage2c_spotlight_examples_df[[
    "call", "expected_prior_calls_prompt_order", "actual_example_names", "expected_prompt_order_names", "identity_match",
]])

stage2c_example_checks = [
    check_row("approved example counts follow cap=3 for all 20 calls", stage2c_example_evolution_df["actual_count"].tolist() == [min(k - 1, 3) for k in range(1, STAGE2C_BATCH_SIZE + 1)], stage2c_example_evolution_df[["call", "actual_count"]]),
    check_row("prompt examples are exactly most recent prior valid outputs", bool(stage2c_example_evolution_df["identity_match"].all()), stage2c_example_evolution_df[["call", "actual_example_names", "expected_prompt_order_names", "identity_match"]]),
    check_row("example counts match summary trace", bool((stage2c_example_evolution_df["actual_count"] == stage2c_example_evolution_df["summary_count"]).all()), stage2c_example_evolution_df[["call", "actual_count", "summary_count"]]),
    check_row("example names match summary trace", bool((stage2c_example_evolution_df["actual_example_names"].map(tuple) == stage2c_example_evolution_df["summary_names"].map(tuple)).all()), stage2c_example_evolution_df[["call", "actual_example_names", "summary_names"]]),
    check_row("calls 10, 15, and 20 use the recent-3 sliding window", bool(stage2c_spotlight_examples_df["identity_match"].all()), stage2c_spotlight_examples_df),
]
display_check_table(stage2c_example_checks, "Stage 2c approved_examples sliding-window checks")
D6_STAGE2C_ACCEPTANCE["approved_examples_identity"] = True


,call,expected_prior_calls_prompt_order,actual_example_names,expected_prompt_order_names,actual_count,expected_count,identity_match,summary_count,summary_names
0,1,[],[],[],0,0,True,0,[]
1,2,[1],[momentum_rsi_oversold_reversal],[momentum_rsi_oversold_reversal],1,1,True,1,[momentum_rsi_oversold_reversal]
2,3,"[2, 1]","[bb_mean_reversion_vol_spike, momentum_rsi_ove...","[bb_mean_reversion_vol_spike, momentum_rsi_ove...",2,2,True,2,"[bb_mean_reversion_vol_spike, momentum_rsi_ove..."
3,4,"[3, 2, 1]","[vol_regime_breakout_fade, bb_mean_reversion_v...","[vol_regime_breakout_fade, bb_mean_reversion_v...",3,3,True,3,"[vol_regime_breakout_fade, bb_mean_reversion_v..."
4,5,"[4, 3, 2]","[volume_surge_momentum_fade, vol_regime_breako...","[volume_surge_momentum_fade, vol_regime_breako...",3,3,True,3,"[volume_surge_momentum_fade, vol_regime_breako..."
5,6,"[5, 4, 3]","[monday_effect_momentum_long, volume_surge_mom...","[monday_effect_momentum_long, volume_surge_mom...",3,3,True,3,"[monday_effect_momentum_long, volume_surge_mom..."
6,7,"[6, 5, 4]","[macd_momentum_acceleration, monday_effect_mom...","[macd_momentum_acceleration, monday_effect_mom...",3,3,True,3,"[macd_momentum_acceleration, monday_effect_mom..."
7,8,"[7, 6, 5]","[bb_reversal_oversold_bounce, macd_momentum_ac...","[bb_reversal_oversold_bounce, macd_momentum_ac...",3,3,True,3,"[bb_reversal_oversold_bounce, macd_momentum_ac..."
8,9,"[8, 7, 6]","[low_vol_breakout_entry, bb_reversal_oversold_...","[low_vol_breakout_entry, bb_reversal_oversold_...",3,3,True,3,"[low_vol_breakout_entry, bb_reversal_oversold_..."
9,10,"[9, 8, 7]","[volume_divergence_reversal, low_vol_breakout_...","[volume_divergence_reversal, low_vol_breakout_...",3,3,True,3,"[volume_divergence_reversal, low_vol_breakout_..."


,call,expected_prior_calls_prompt_order,actual_example_names,expected_prompt_order_names,identity_match
9,10,"[9, 8, 7]","[volume_divergence_reversal, low_vol_breakout_...","[volume_divergence_reversal, low_vol_breakout_...",True
14,15,"[14, 13, 12]","[volume_divergence_reversal, low_vol_breakout,...","[volume_divergence_reversal, low_vol_breakout,...",True
19,20,"[19, 18, 17]","[volume_div_breakout, vol_regime_breakout, ove...","[volume_div_breakout, vol_regime_breakout, ove...",True


Stage 2c approved_examples sliding-window checks


,check,status,detail
0,approved example counts follow cap=3 for all 2...,PASS,call actual_count 0 1 0 ...
1,prompt examples are exactly most recent prior ...,PASS,call actual_...
2,example counts match summary trace,PASS,call actual_count summary_count 0 1...
3,example names match summary trace,PASS,call actual_...
4,"calls 10, 15, and 20 use the recent-3 sliding ...",PASS,call expected_prior_calls_prompt_order \ ...


## AF. Independent Per-Theme Telemetry Rebuild

Stage 2c's most useful observation layer is per-theme behavior. This section rebuilds the call-level and theme-level telemetry from the reclassified DSL factor lists and the post-hoc `THEME_HINTS` mapping, then compares it to `stage2c_summary.json`.


In [33]:
def _stage2c_factor_overlap_metrics(theme: str, factors: list[str]) -> dict:
    factor_set = set(factors)
    theme_hints = set(STAGE2C_THEME_HINTS.get(theme, frozenset()))
    overlap = factor_set & theme_hints
    default_hits = sorted(factor_set & set(STAGE2C_DEFAULT_MOMENTUM_FACTORS))
    return {
        "overlap_count_recomputed": len(overlap),
        "overlap_ratio_recomputed": (len(overlap) / len(factors)) if factors else None,
        "out_of_theme_factor_count_recomputed": len(factors) - len(overlap),
        "contains_default_momentum_factor_recomputed": bool(default_hits),
        "default_momentum_factors_used_recomputed": default_hits,
    }

stage2c_theme_call_rows = []
for call in STAGE2C_CALLS:
    k = call["position"]
    dsl = stage2c_response_dsl_by_call[k]
    factors = _stage2c_extract_factors_from_dsl(dsl)
    metrics = _stage2c_factor_overlap_metrics(call["theme"], factors)
    stage2c_theme_call_rows.append({
        "call": k,
        "theme": call["theme"],
        "factors_used_recomputed": factors,
        "factors_used_summary": call["factors_used"],
        "overlap_count_summary": call["overlap_count"],
        "overlap_ratio_summary": call["overlap_ratio"],
        "out_of_theme_factor_count_summary": call["out_of_theme_factor_count"],
        "contains_default_momentum_factor_summary": call["contains_default_momentum_factor"],
        "default_momentum_factors_used_summary": call["default_momentum_factors_used"],
        **metrics,
    })
stage2c_theme_call_df = pd.DataFrame(stage2c_theme_call_rows)
stage2c_theme_call_df["overlap_ratio_match"] = stage2c_theme_call_df.apply(
    lambda row: (
        row["overlap_ratio_summary"] is None and row["overlap_ratio_recomputed"] is None
    ) or abs(float(row["overlap_ratio_summary"]) - float(row["overlap_ratio_recomputed"])) < 1e-12,
    axis=1,
)
display(stage2c_theme_call_df)

stage2c_theme_aggregate_rows = []
for theme in STAGE2C_EXPECTED_THEMES:
    theme_calls = stage2c_theme_call_df[stage2c_theme_call_df["theme"] == theme]
    valid_calls = stage2c_reclass_df[(stage2c_reclass_df["theme"] == theme) & (stage2c_reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST)]
    lifecycle_mix = stage2c_reclass_df[stage2c_reclass_df["theme"] == theme]["reclassified_lifecycle"].value_counts().to_dict()
    factor_counter = Counter()
    for factors in valid_calls["factors"]:
        factor_counter.update(factors)
    dominant_top3 = [factor for factor, _count in sorted(factor_counter.items(), key=lambda kv: (-kv[1], kv[0]))[:3]]
    summary_row = next(row for row in STAGE2C_SUMMARY["per_theme"] if row["theme"] == theme)
    aggregate_row = {
        "theme": theme,
        "n_calls_recomputed": int(len(theme_calls)),
        "n_calls_summary": summary_row["n_calls"],
        "valid_count_recomputed": int(len(valid_calls)),
        "valid_count_summary": summary_row["valid_count"],
        "lifecycle_mix_recomputed": lifecycle_mix,
        "lifecycle_mix_summary": summary_row["lifecycle_mix"],
        "avg_overlap_count_recomputed": float(theme_calls["overlap_count_recomputed"].mean()),
        "avg_overlap_count_summary": summary_row["avg_overlap_count"],
        "avg_overlap_ratio_recomputed": float(theme_calls["overlap_ratio_recomputed"].mean()),
        "avg_overlap_ratio_summary": summary_row["avg_overlap_ratio"],
        "avg_out_of_theme_factors_recomputed": float(theme_calls["out_of_theme_factor_count_recomputed"].mean()),
        "avg_out_of_theme_factors_summary": summary_row["avg_out_of_theme_factors"],
        "contains_rsi14_count_recomputed": int(valid_calls["factors"].map(lambda fs: "rsi_14" in fs).sum()),
        "contains_rsi14_count_summary": summary_row["contains_rsi14_count"],
        "contains_momentum_default_count_recomputed": int(theme_calls["contains_default_momentum_factor_recomputed"].sum()),
        "contains_momentum_default_count_summary": summary_row["contains_momentum_default_count"],
        "dominant_factors_top3_recomputed": dominant_top3,
        "dominant_factors_top3_summary": summary_row["dominant_factors_top3"],
    }
    stage2c_theme_aggregate_rows.append(aggregate_row)
stage2c_theme_aggregate_df = pd.DataFrame(stage2c_theme_aggregate_rows)
display(stage2c_theme_aggregate_df)

stage2c_call_telemetry_checks = [
    check_row("recomputed factors match summary factors", bool((stage2c_theme_call_df["factors_used_recomputed"].map(tuple) == stage2c_theme_call_df["factors_used_summary"].map(tuple)).all()), stage2c_theme_call_df[["call", "factors_used_recomputed", "factors_used_summary"]]),
    check_row("overlap counts match summary", bool((stage2c_theme_call_df["overlap_count_recomputed"] == stage2c_theme_call_df["overlap_count_summary"]).all()), stage2c_theme_call_df[["call", "overlap_count_recomputed", "overlap_count_summary"]]),
    check_row("overlap ratios match summary", bool(stage2c_theme_call_df["overlap_ratio_match"].all()), stage2c_theme_call_df[["call", "overlap_ratio_recomputed", "overlap_ratio_summary"]]),
    check_row("out-of-theme counts match summary", bool((stage2c_theme_call_df["out_of_theme_factor_count_recomputed"] == stage2c_theme_call_df["out_of_theme_factor_count_summary"]).all()), stage2c_theme_call_df[["call", "out_of_theme_factor_count_recomputed", "out_of_theme_factor_count_summary"]]),
    check_row("default momentum flags match summary", bool((stage2c_theme_call_df["contains_default_momentum_factor_recomputed"] == stage2c_theme_call_df["contains_default_momentum_factor_summary"]).all()), stage2c_theme_call_df[["call", "contains_default_momentum_factor_recomputed", "contains_default_momentum_factor_summary"]]),
]
display_check_table(stage2c_call_telemetry_checks, "Stage 2c per-call theme telemetry checks")

stage2c_aggregate_checks = [
    check_row("per-theme n_calls match summary", bool((stage2c_theme_aggregate_df["n_calls_recomputed"] == stage2c_theme_aggregate_df["n_calls_summary"]).all()), stage2c_theme_aggregate_df[["theme", "n_calls_recomputed", "n_calls_summary"]]),
    check_row("per-theme valid_count match summary", bool((stage2c_theme_aggregate_df["valid_count_recomputed"] == stage2c_theme_aggregate_df["valid_count_summary"]).all()), stage2c_theme_aggregate_df[["theme", "valid_count_recomputed", "valid_count_summary"]]),
    check_row("per-theme lifecycle_mix match summary", bool((stage2c_theme_aggregate_df["lifecycle_mix_recomputed"].map(str) == stage2c_theme_aggregate_df["lifecycle_mix_summary"].map(str)).all()), stage2c_theme_aggregate_df[["theme", "lifecycle_mix_recomputed", "lifecycle_mix_summary"]]),
    check_row("per-theme average overlap counts match summary", bool(stage2c_theme_aggregate_df.apply(lambda row: abs(row["avg_overlap_count_recomputed"] - row["avg_overlap_count_summary"]) < 1e-12, axis=1).all()), stage2c_theme_aggregate_df[["theme", "avg_overlap_count_recomputed", "avg_overlap_count_summary"]]),
    check_row("per-theme average overlap ratios match summary", bool(stage2c_theme_aggregate_df.apply(lambda row: abs(row["avg_overlap_ratio_recomputed"] - row["avg_overlap_ratio_summary"]) < 1e-12, axis=1).all()), stage2c_theme_aggregate_df[["theme", "avg_overlap_ratio_recomputed", "avg_overlap_ratio_summary"]]),
    check_row("per-theme average out-of-theme counts match summary", bool(stage2c_theme_aggregate_df.apply(lambda row: abs(row["avg_out_of_theme_factors_recomputed"] - row["avg_out_of_theme_factors_summary"]) < 1e-12, axis=1).all()), stage2c_theme_aggregate_df[["theme", "avg_out_of_theme_factors_recomputed", "avg_out_of_theme_factors_summary"]]),
    check_row("per-theme dominant factors match summary", bool((stage2c_theme_aggregate_df["dominant_factors_top3_recomputed"].map(tuple) == stage2c_theme_aggregate_df["dominant_factors_top3_summary"].map(tuple)).all()), stage2c_theme_aggregate_df[["theme", "dominant_factors_top3_recomputed", "dominant_factors_top3_summary"]]),
]
display_check_table(stage2c_aggregate_checks, "Stage 2c per-theme aggregate checks")
D6_STAGE2C_ACCEPTANCE["per_theme_telemetry_rebuild"] = True


,call,theme,factors_used_recomputed,factors_used_summary,overlap_count_summary,overlap_ratio_summary,out_of_theme_factor_count_summary,contains_default_momentum_factor_summary,default_momentum_factors_used_summary,overlap_count_recomputed,overlap_ratio_recomputed,out_of_theme_factor_count_recomputed,contains_default_momentum_factor_recomputed,default_momentum_factors_used_recomputed,overlap_ratio_match
0,1,momentum,"[close, ema_12, rsi_14]","[close, ema_12, rsi_14]",1,0.333333,2,True,[rsi_14],1,0.333333,2,True,[rsi_14],True
1,2,mean_reversion,"[bb_upper_24_2, close, realized_vol_24h, sma_2...","[bb_upper_24_2, close, realized_vol_24h, sma_2...",3,0.600000,2,False,[],3,0.600000,2,False,[],True
2,3,volatility_regime,"[bb_upper_24_2, close, realized_vol_24h, rsi_1...","[bb_upper_24_2, close, realized_vol_24h, rsi_1...",2,0.400000,3,True,[rsi_14],2,0.400000,3,True,[rsi_14],True
3,4,volume_divergence,"[close, rsi_14, sma_20, volume_zscore_24h]","[close, rsi_14, sma_20, volume_zscore_24h]",1,0.250000,3,True,[rsi_14],1,0.250000,3,True,[rsi_14],True
4,5,calendar_effect,"[day_of_week, hour_of_day, return_1h, return_24h]","[day_of_week, hour_of_day, return_1h, return_24h]",2,0.500000,2,True,"[return_1h, return_24h]",2,0.500000,2,True,"[return_1h, return_24h]",True
5,6,momentum,"[macd_hist, return_1h, return_24h, rsi_14]","[macd_hist, return_1h, return_24h, rsi_14]",4,1.000000,0,True,"[macd_hist, return_1h, return_24h, rsi_14]",4,1.000000,0,True,"[macd_hist, return_1h, return_24h, rsi_14]",True
6,7,mean_reversion,"[bb_upper_24_2, close, return_24h, rsi_14, sma...","[bb_upper_24_2, close, return_24h, rsi_14, sma...",2,0.400000,3,True,"[return_24h, rsi_14]",2,0.400000,3,True,"[return_24h, rsi_14]",True
7,8,volatility_regime,"[close, realized_vol_24h, sma_20, volume_zscor...","[close, realized_vol_24h, sma_20, volume_zscor...",1,0.250000,3,False,[],1,0.250000,3,False,[],True
8,9,volume_divergence,"[return_24h, rsi_14, volume_zscore_24h]","[return_24h, rsi_14, volume_zscore_24h]",1,0.333333,2,True,"[return_24h, rsi_14]",1,0.333333,2,True,"[return_24h, rsi_14]",True
9,10,calendar_effect,"[day_of_week, hour_of_day, return_1h, return_24h]","[day_of_week, hour_of_day, return_1h, return_24h]",2,0.500000,2,True,"[return_1h, return_24h]",2,0.500000,2,True,"[return_1h, return_24h]",True


,theme,n_calls_recomputed,n_calls_summary,valid_count_recomputed,valid_count_summary,lifecycle_mix_recomputed,lifecycle_mix_summary,avg_overlap_count_recomputed,avg_overlap_count_summary,avg_overlap_ratio_recomputed,avg_overlap_ratio_summary,avg_out_of_theme_factors_recomputed,avg_out_of_theme_factors_summary,contains_rsi14_count_recomputed,contains_rsi14_count_summary,contains_momentum_default_count_recomputed,contains_momentum_default_count_summary,dominant_factors_top3_recomputed,dominant_factors_top3_summary
0,momentum,4,4,4,4,{'pending_backtest': 4},{'pending_backtest': 4},2.50,2.50,0.687500,0.687500,1.00,1.00,3,3,4,4,"[macd_hist, rsi_14, return_1h]","[macd_hist, rsi_14, return_1h]"
1,mean_reversion,4,4,4,4,{'pending_backtest': 4},{'pending_backtest': 4},2.75,2.75,0.562500,0.562500,2.25,2.25,3,3,3,3,"[bb_upper_24_2, close, rsi_14]","[bb_upper_24_2, close, rsi_14]"
2,volatility_regime,4,4,4,4,{'pending_backtest': 4},{'pending_backtest': 4},1.25,1.25,0.266667,0.266667,3.50,3.50,1,1,2,2,"[close, realized_vol_24h, sma_20]","[close, realized_vol_24h, sma_20]"
3,volume_divergence,4,4,4,4,{'pending_backtest': 4},{'pending_backtest': 4},1.00,1.00,0.237500,0.237500,3.50,3.50,3,3,4,4,"[volume_zscore_24h, close, return_24h]","[volume_zscore_24h, close, return_24h]"
4,calendar_effect,4,4,4,4,{'pending_backtest': 4},{'pending_backtest': 4},1.50,1.50,0.350000,0.350000,3.00,3.00,2,2,4,4,"[day_of_week, return_24h, return_1h]","[day_of_week, return_24h, return_1h]"


Stage 2c per-call theme telemetry checks


,check,status,detail
0,recomputed factors match summary factors,PASS,call factors_us...
1,overlap counts match summary,PASS,call overlap_count_recomputed overlap_co...
2,overlap ratios match summary,PASS,call overlap_ratio_recomputed overlap_ra...
3,out-of-theme counts match summary,PASS,call out_of_theme_factor_count_recomputed...
4,default momentum flags match summary,PASS,call contains_default_momentum_factor_rec...


Stage 2c per-theme aggregate checks


,check,status,detail
0,per-theme n_calls match summary,PASS,theme n_calls_recomputed n_ca...
1,per-theme valid_count match summary,PASS,theme valid_count_recomputed ...
2,per-theme lifecycle_mix match summary,PASS,theme lifecycle_mix_recomputed ...
3,per-theme average overlap counts match summary,PASS,theme avg_overlap_count_recomp...
4,per-theme average overlap ratios match summary,PASS,theme avg_overlap_ratio_recomp...
5,per-theme average out-of-theme counts match su...,PASS,theme avg_out_of_theme_factors...
6,per-theme dominant factors match summary,PASS,theme dominant_factors_t...


## AG. Cost, Ledger, And No-Pending Reconciliation

This section ties the Stage 2c summary back to the SQLite budget ledger. It checks that per-call costs match, totals match, no pending charges remain, and the batch stayed comfortably below the Stage 2 cumulative stop threshold.


In [34]:
stage2c_ledger_cost_df = stage2c_ledger_df[["call", "estimated_cost", "actual_cost", "status", "created_at_utc", "completed_at_utc"]].copy()
stage2c_summary_cost_df = pd.DataFrame([
    {
        "call": call["position"],
        "summary_estimated": call["estimated_cost_usd"],
        "summary_actual": call["actual_cost_usd"],
        "input_tokens": call["input_tokens"],
        "output_tokens": call["output_tokens"],
        "retry_count": call["retry_count"],
    }
    for call in STAGE2C_CALLS
])
stage2c_cost_join_df = stage2c_ledger_cost_df.merge(stage2c_summary_cost_df, on="call", how="inner")
stage2c_cost_join_df["estimated_matches"] = (stage2c_cost_join_df["estimated_cost"] - stage2c_cost_join_df["summary_estimated"]).abs() < 1e-12
stage2c_cost_join_df["actual_matches"] = (stage2c_cost_join_df["actual_cost"] - stage2c_cost_join_df["summary_actual"]).abs() < 1e-12

display(stage2c_cost_join_df)

stage2c_total_estimated_from_ledger = float(stage2c_cost_join_df["estimated_cost"].sum())
stage2c_total_actual_from_ledger = float(stage2c_cost_join_df["actual_cost"].sum())
with _stage2c_connect_ledger_readonly() as conn:
    stage2c_pending_df = pd.read_sql_query(
        "SELECT * FROM ledger WHERE batch_id = ? AND status = ?",
        conn,
        params=(STAGE2C_BATCH_ID, STATUS_PENDING),
    )

stage2c_cost_checks = [
    check_row("per-call ledger costs match summary", bool(stage2c_cost_join_df["estimated_matches"].all() and stage2c_cost_join_df["actual_matches"].all()), stage2c_cost_join_df),
    check_row("batch total estimated cost matches summary", abs(stage2c_total_estimated_from_ledger - STAGE2C_SUMMARY["total_estimated_cost_usd"]) < 1e-12, {"ledger": stage2c_total_estimated_from_ledger, "summary": STAGE2C_SUMMARY["total_estimated_cost_usd"]}),
    check_row("batch total actual cost matches summary", abs(stage2c_total_actual_from_ledger - STAGE2C_SUMMARY["total_actual_cost_usd"]) < 1e-12, {"ledger": stage2c_total_actual_from_ledger, "summary": STAGE2C_SUMMARY["total_actual_cost_usd"]}),
    check_row("no pending ledger rows remain for Stage 2c batch", len(stage2c_pending_df) == 0, stage2c_pending_df),
    check_row("all calls had zero retries", bool((stage2c_cost_join_df["retry_count"] == 0).all()), stage2c_cost_join_df[["call", "retry_count"]]),
    check_row("estimator remains conservative", bool((stage2c_cost_join_df["estimated_cost"] >= stage2c_cost_join_df["actual_cost"]).all()), stage2c_cost_join_df[["call", "estimated_cost", "actual_cost"]]),
    check_row("cumulative Stage 2 spend stays under $30 stop threshold", STAGE2C_SUMMARY["cumulative_monthly_spend_usd"] <= STAGE2C_CUMULATIVE_CAP_USD, STAGE2C_SUMMARY["cumulative_monthly_spend_usd"]),
]
display_check_table(stage2c_cost_checks, "Stage 2c ledger and cost reconciliation checks")
D6_STAGE2C_ACCEPTANCE["ledger_cost_reconciliation"] = True


,call,estimated_cost,actual_cost,status,created_at_utc,completed_at_utc,summary_estimated,summary_actual,input_tokens,output_tokens,retry_count,estimated_matches,actual_matches
0,1,0.011475,0.006954,completed,2026-04-17T17:46:05.350Z,2026-04-17T17:46:09.694Z,0.011475,0.006954,1308,202,0,True,True
1,2,0.011916,0.007767,completed,2026-04-17T17:46:09.696Z,2026-04-17T17:46:14.246Z,0.011916,0.007767,1439,230,0,True,True
2,3,0.012432,0.008385,completed,2026-04-17T17:46:14.248Z,2026-04-17T17:46:19.138Z,0.012432,0.008385,1595,240,0,True,True
3,4,0.012984,0.009414,completed,2026-04-17T17:46:19.141Z,2026-04-17T17:46:23.950Z,0.012984,0.009414,1758,276,0,True,True
4,5,0.013116,0.009777,completed,2026-04-17T17:46:23.952Z,2026-04-17T17:46:29.270Z,0.013116,0.009777,1804,291,0,True,True
5,6,0.013236,0.009621,completed,2026-04-17T17:46:29.272Z,2026-04-17T17:46:33.908Z,0.013236,0.009621,1847,272,0,True,True
6,7,0.013296,0.009819,completed,2026-04-17T17:46:33.910Z,2026-04-17T17:46:38.454Z,0.013296,0.009819,1868,281,0,True,True
7,8,0.013338,0.009882,completed,2026-04-17T17:46:38.457Z,2026-04-17T17:46:43.532Z,0.013338,0.009882,1874,284,0,True,True
8,9,0.013341,0.009825,completed,2026-04-17T17:46:43.535Z,2026-04-17T17:46:48.106Z,0.013341,0.009825,1865,282,0,True,True
9,10,0.013353,0.010071,completed,2026-04-17T17:46:48.108Z,2026-04-17T17:46:55.121Z,0.013353,0.010071,1877,296,0,True,True


Stage 2c ledger and cost reconciliation checks


,check,status,detail
0,per-call ledger costs match summary,PASS,call estimated_cost actual_cost stat...
1,batch total estimated cost matches summary,PASS,"{'ledger': 0.262776, 'summary': 0.262776}"
2,batch total actual cost matches summary,PASS,"{'ledger': 0.195129, 'summary': 0.195129}"
3,no pending ledger rows remain for Stage 2c batch,PASS,"Empty DataFrame Columns: [id, batch_id, api_ca..."
4,all calls had zero retries,PASS,call retry_count 0 1 0 1 ...
5,estimator remains conservative,PASS,call estimated_cost actual_cost 0 1...
6,cumulative Stage 2 spend stays under $30 stop ...,PASS,0.256317


## AH. Catastrophic Stop Conditions Audit

Stage 2c defines explicit stop conditions. This section checks them mechanically instead of relying on the summary headline.


In [35]:
first_k = STAGE2C_PARSE_RATE_MIN_K
first_k_valid_rate = float((stage2c_reclass_df.head(first_k)["reclassified_lifecycle"] == PENDING_BACKTEST).mean())
stage2c_cardinality_violations = int((stage2c_reclass_df["cardinality_recomputed"] != "single_object").sum())
stage2c_leakage_defects = int((~stage2c_prompt_audit_df["leakage_clean"]).sum())

stage2c_single_mode_rows = []
for theme in STAGE2C_EXPECTED_THEMES:
    theme_rows = stage2c_reclass_df[stage2c_reclass_df["theme"] == theme]
    all_invalid = bool((theme_rows["reclassified_lifecycle"] != PENDING_BACKTEST).all()) if len(theme_rows) == 4 else False
    stage2c_single_mode_rows.append({
        "theme": theme,
        "n_calls": len(theme_rows),
        "lifecycle_values": theme_rows["reclassified_lifecycle"].tolist(),
        "single_mode_failure_triggered": all_invalid,
    })
stage2c_single_mode_df = pd.DataFrame(stage2c_single_mode_rows)
display(stage2c_single_mode_df)

stage2c_stop_checks = [
    check_row("first 5 parse-valid rate is above 50%", first_k_valid_rate > STAGE2C_PARSE_RATE_THRESHOLD, first_k_valid_rate),
    check_row("no theme has 4/4 failures in one mode", not bool(stage2c_single_mode_df["single_mode_failure_triggered"].any()), stage2c_single_mode_df),
    check_row("cardinality violations do not exceed stop threshold", stage2c_cardinality_violations <= STAGE2C_CARDINALITY_VIOLATION_STOP, stage2c_cardinality_violations),
    check_row("no prompt leakage defects", stage2c_leakage_defects == 0, stage2c_leakage_defects),
    check_row("cumulative Stage 2 spend is <= $30", STAGE2C_SUMMARY["cumulative_monthly_spend_usd"] <= STAGE2C_CUMULATIVE_CAP_USD, STAGE2C_SUMMARY["cumulative_monthly_spend_usd"]),
    check_row("summary reports no early stop", STAGE2C_SUMMARY.get("early_stop_reason") is None and STAGE2C_SUMMARY.get("truncated") is False, {"early_stop_reason": STAGE2C_SUMMARY.get("early_stop_reason"), "truncated": STAGE2C_SUMMARY.get("truncated")}),
    check_row("summary reports no anomaly flags", STAGE2C_SUMMARY.get("anomaly_flags") == [], STAGE2C_SUMMARY.get("anomaly_flags")),
]
display_check_table(stage2c_stop_checks, "Stage 2c catastrophic stop condition checks")
D6_STAGE2C_ACCEPTANCE["catastrophic_stop_absence"] = True


,theme,n_calls,lifecycle_values,single_mode_failure_triggered
0,momentum,4,"[pending_backtest, pending_backtest, pending_b...",False
1,mean_reversion,4,"[pending_backtest, pending_backtest, pending_b...",False
2,volatility_regime,4,"[pending_backtest, pending_backtest, pending_b...",False
3,volume_divergence,4,"[pending_backtest, pending_backtest, pending_b...",False
4,calendar_effect,4,"[pending_backtest, pending_backtest, pending_b...",False


Stage 2c catastrophic stop condition checks


,check,status,detail
0,first 5 parse-valid rate is above 50%,PASS,1.0
1,no theme has 4/4 failures in one mode,PASS,theme n_calls \ 0 m...
2,cardinality violations do not exceed stop thre...,PASS,0
3,no prompt leakage defects,PASS,0
4,cumulative Stage 2 spend is <= $30,PASS,0.256317
5,summary reports no early stop,PASS,"{'early_stop_reason': None, 'truncated': False}"
6,summary reports no anomaly flags,PASS,[]


## AI. Stage 2c Verdict And Review Inputs

What Stage 2c proved:

- 20 live calls completed under Stage 1/2a/2b constraints
- strict sequential execution is visible in ledger timestamps
- all prompts are leakage-clean despite approved-example growth
- approved examples are the latest three prior valid DSLs, most-recent first
- all raw responses independently reclassify to `pending_backtest`
- all hypothesis hashes are unique
- per-theme telemetry and dominant-factor summaries can be rebuilt from raw DSL factors
- ledger cost reconciliation closes with no pending rows
- catastrophic stop conditions did not trigger

What Stage 2c did **not** prove:

- that any hypothesis has alpha
- that theme adherence is perfect
- that Stage 2d should skip human review

Recommendation input: Stage 2c provides a clean 20-call operational audit. The next decision can focus on review quality and downstream backtest selection rather than plumbing repair.


In [36]:
stage2c_final_rows = [
    {"gate": "batch receipts + artifacts", "pass": D6_STAGE2C_ACCEPTANCE.get("batch_receipts_artifacts", False)},
    {"gate": "strict sequential execution", "pass": D6_STAGE2C_ACCEPTANCE.get("strict_sequential_execution", False)},
    {"gate": "all prompt leakage audits", "pass": D6_STAGE2C_ACCEPTANCE.get("prompt_audit_all_calls", False)},
    {"gate": "approved_examples identity", "pass": D6_STAGE2C_ACCEPTANCE.get("approved_examples_identity", False)},
    {"gate": "response reclassification + hash uniqueness", "pass": D6_STAGE2C_ACCEPTANCE.get("reclassification_hash_uniqueness", False)},
    {"gate": "per-theme telemetry rebuild", "pass": D6_STAGE2C_ACCEPTANCE.get("per_theme_telemetry_rebuild", False)},
    {"gate": "ledger/cost reconciliation", "pass": D6_STAGE2C_ACCEPTANCE.get("ledger_cost_reconciliation", False)},
    {"gate": "catastrophic stop absence", "pass": D6_STAGE2C_ACCEPTANCE.get("catastrophic_stop_absence", False)},
]
stage2c_final_df = pd.DataFrame(stage2c_final_rows)
stage2c_final_df["status"] = stage2c_final_df["pass"].map({True: "PASS", False: "FAIL"})
display(stage2c_final_df[["gate", "status"]])

assert stage2c_final_df["pass"].all(), stage2c_final_df[~stage2c_final_df["pass"]]

print("D6 Stage 2c acceptance audit: PASS")
print("The notebook independently reconstructed sequential execution, prompt safety, approved_examples identity, response classification, hash uniqueness, per-theme telemetry, ledger closure, and catastrophic-stop absence from durable artifacts.")
print("Recommendation input: Stage 2c is operationally clean enough for human review of Stage 2d/backtest selection criteria.")


,gate,status
0,batch receipts + artifacts,PASS
1,strict sequential execution,PASS
2,all prompt leakage audits,PASS
3,approved_examples identity,PASS
4,response reclassification + hash uniqueness,PASS
5,per-theme telemetry rebuild,PASS
6,ledger/cost reconciliation,PASS
7,catastrophic stop absence,PASS


D6 Stage 2c acceptance audit: PASS
The notebook independently reconstructed sequential execution, prompt safety, approved_examples identity, response classification, hash uniqueness, per-theme telemetry, ledger closure, and catastrophic-stop absence from durable artifacts.
Recommendation input: Stage 2c is operationally clean enough for human review of Stage 2d/backtest selection criteria.


## AJ. D6 Stage 2d Scope And Acceptance Framing

D6 Stage 2d is a **200-call sequential live production-shape batch audit**. This notebook section does not re-run the batch and does not interpret strategy quality.

Stage 2d is a production-shape execution audit, not prompt tuning or model interpretation.

The acceptance objective is to verify:

- Stage 2d-specific contracts: final/partial summaries, fuses, factor-set telemetry, crash-preservation artifacts
- summary correctness through independent recomputation
- aggregate correctness over all 200 calls
- sampled forensic correctness for key and stratified calls
- reviewer-grade evidence for D7 Critic design

This section audits Stage 2d artifacts only. It does not re-accept Stage 2a, 2b, or 2c.


In [37]:
import json
import math
import os
import random
import re
import sqlite3
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Allow Stage 2d cells to run from a fresh kernel, from either repo root
# or the test notebooks/ folder, without requiring earlier notebook cells.
_STAGE2D_START_CWD = Path.cwd().resolve()
_STAGE2D_REPO_ROOT = next(
    (p for p in (_STAGE2D_START_CWD, *_STAGE2D_START_CWD.parents) if (p / "pyproject.toml").exists() and (p / "agents").exists()),
    None,
)
if _STAGE2D_REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {_STAGE2D_START_CWD}")
if str(_STAGE2D_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_STAGE2D_REPO_ROOT))
os.chdir(_STAGE2D_REPO_ROOT)

RANDOM_SEED_STAGE2D = 42
random.seed(RANDOM_SEED_STAGE2D)
np.random.seed(RANDOM_SEED_STAGE2D)
# DO NOT change this value across re-runs of this notebook.

from agents.orchestrator.budget_ledger import STATUS_COMPLETED, STATUS_PENDING
from agents.orchestrator.ingest import (
    BatchIngestState,
    PENDING_BACKTEST,
    assert_lifecycle_invariant_at_batch_close,
    ingest_outputs,
)
from agents.proposer import ProposerOutput, ValidCandidate
from agents.proposer.prompt_builder import audit_prompt_for_leakage
from agents.proposer.stage2d_batch import (
    APPROVED_EXAMPLES_CAP as STAGE2D_APPROVED_EXAMPLES_CAP,
    BLOCK_SIZE as STAGE2D_BLOCK_SIZE,
    CARDINALITY_VIOLATION_STOP as STAGE2D_CARDINALITY_VIOLATION_STOP,
    DEFAULT_MOMENTUM_FACTORS as STAGE2D_DEFAULT_MOMENTUM_FACTORS,
    MODEL_NAME as STAGE2D_MODEL_NAME,
    PROMPT_CACHING_ENABLED as STAGE2D_PROMPT_CACHING_ENABLED,
    SINGLE_MODE_THRESHOLD as STAGE2D_SINGLE_MODE_THRESHOLD,
    SNAPSHOT_POSITIONS as STAGE2D_SNAPSHOT_POSITIONS,
    STAGE2D_BATCH_CAP_USD,
    STAGE2D_BATCH_SIZE,
    STAGE2D_CUMULATIVE_CAP_USD,
    STAGE_LABEL as STAGE2D_STAGE_LABEL,
    THEME_CALLS_TOTAL as STAGE2D_THEME_CALLS_TOTAL,
    THEME_CYCLE_LEN as STAGE2D_THEME_CYCLE_LEN,
    THEME_HINTS as STAGE2D_THEME_HINTS,
    THEME_ROTATION_MODE as STAGE2D_THEME_ROTATION_MODE,
    TIER1_K as STAGE2D_TIER1_K,
    TIER1_THRESHOLD as STAGE2D_TIER1_THRESHOLD,
    TIER2_K as STAGE2D_TIER2_K,
    TIER2_THRESHOLD as STAGE2D_TIER2_THRESHOLD,
)
from agents.proposer.stub_backend import classify_raw_json
from agents.themes import THEMES as STAGE2D_ALL_THEMES
from factors.registry import get_registry

REGISTRY = globals().get("REGISTRY", get_registry())
ALLOWED_OPS = globals().get("ALLOWED_OPS", ("<", "<=", ">", ">=", "==", "crosses_above", "crosses_below"))

if "check_row" not in globals():
    def status(ok: bool) -> str:
        return "PASS" if bool(ok) else "FAIL"

    def check_row(check: str, ok: bool, detail="") -> dict:
        return {"check": check, "status": status(ok), "detail": detail}

if "display_check_table" not in globals():
    def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
        print(title)
        df = pd.DataFrame(rows)
        display(df)
        if require_all:
            failed = df[df["status"] != "PASS"]
            assert failed.empty, failed
        return df

D6_STAGE2D_ACCEPTANCE: dict[str, bool] = {}
STAGE2D_LEDGER_PATH = Path("agents/spend_ledger.db")
STAGE2D_EXPECTED_THEMES = list(STAGE2D_ALL_THEMES)[:STAGE2D_THEME_CYCLE_LEN]
STAGE2D_SUMMARY_CANDIDATES = sorted(
    Path("raw_payloads").glob("batch_*/stage2d_summary.json"),
    key=lambda path: path.stat().st_mtime,
)
assert STAGE2D_SUMMARY_CANDIDATES, "No completed stage2d_summary.json artifacts found under raw_payloads/"
STAGE2D_SUMMARY_PATH = STAGE2D_SUMMARY_CANDIDATES[-1]
STAGE2D_BATCH_DIR = STAGE2D_SUMMARY_PATH.parent
STAGE2D_PARTIAL_SUMMARY_PATH = STAGE2D_BATCH_DIR / "stage2d_summary_partial.json"
STAGE2D_BATCH_ID = STAGE2D_BATCH_DIR.name.removeprefix("batch_")
STAGE2D_SUMMARY = json.loads(STAGE2D_SUMMARY_PATH.read_text(encoding="utf-8"))
STAGE2D_PARTIAL_SUMMARY = json.loads(STAGE2D_PARTIAL_SUMMARY_PATH.read_text(encoding="utf-8")) if STAGE2D_PARTIAL_SUMMARY_PATH.exists() else {}
STAGE2D_CALLS = sorted(STAGE2D_SUMMARY["calls"], key=lambda row: row["position"])
STAGE2D_CALL_BY_POSITION = {call["position"]: call for call in STAGE2D_CALLS}
STAGE2D_CONFIG = STAGE2D_SUMMARY["config"]
STAGE2D_PARTIAL_CONFIG = STAGE2D_PARTIAL_SUMMARY.get("config", {})

print("Stage 2d repo root:", _STAGE2D_REPO_ROOT)
print("Stage 2d random seed:", RANDOM_SEED_STAGE2D)
print("Stage 2d batch:", STAGE2D_BATCH_ID)
print("Stage 2d summary:", STAGE2D_SUMMARY_PATH)


Stage 2d repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Stage 2d random seed: 42
Stage 2d batch: 5cf76668-47d1-48d7-bd90-db06d31982ed
Stage 2d summary: raw_payloads/batch_5cf76668-47d1-48d7-bd90-db06d31982ed/stage2d_summary.json


## AK. Batch Receipts And Artifact Inventory

This section binds the audit to one durable Stage 2d batch. It checks the final summary, partial summary, raw payload directory, all 200 prompt files, all 200 response files, and retry files if any exist.


In [38]:
receipt_rows = [
    {"item": "batch_id", "value": STAGE2D_BATCH_ID},
    {"item": "summary path", "value": str(STAGE2D_SUMMARY_PATH)},
    {"item": "partial summary path", "value": str(STAGE2D_PARTIAL_SUMMARY_PATH)},
    {"item": "git commit", "value": STAGE2D_CONFIG.get("git_commit")},
    {"item": "run timestamp UTC", "value": STAGE2D_CONFIG.get("run_timestamp_utc")},
    {"item": "stage label", "value": STAGE2D_CONFIG.get("stage_label")},
    {"item": "model", "value": STAGE2D_CONFIG.get("model_name")},
    {"item": "prompt caching enabled", "value": STAGE2D_CONFIG.get("prompt_caching_enabled")},
    {"item": "batch size", "value": STAGE2D_CONFIG.get("batch_size")},
    {"item": "batch cap USD", "value": STAGE2D_CONFIG.get("batch_cap_usd")},
    {"item": "theme rotation", "value": STAGE2D_CONFIG.get("theme_rotation_mode")},
    {"item": "approved examples cap", "value": STAGE2D_CONFIG.get("approved_examples_cap")},
    {"item": "total estimated cost", "value": STAGE2D_SUMMARY.get("total_estimated_cost_usd")},
    {"item": "total actual cost", "value": STAGE2D_SUMMARY.get("total_actual_cost_usd")},
    {"item": "cumulative monthly spend", "value": STAGE2D_SUMMARY.get("cumulative_monthly_spend_usd")},
    {"item": "batch status", "value": STAGE2D_SUMMARY.get("batch_status")},
    {"item": "truncated", "value": STAGE2D_SUMMARY.get("truncated")},
    {"item": "unissued slots", "value": STAGE2D_SUMMARY.get("unissued_slots")},
    {"item": "batch duration seconds", "value": STAGE2D_SUMMARY.get("batch_duration_seconds")},
]
display(pd.DataFrame(receipt_rows))

artifact_rows = []
for k in range(1, STAGE2D_BATCH_SIZE + 1):
    prompt_path = STAGE2D_BATCH_DIR / f"attempt_{k:04d}_prompt.txt"
    response_path = STAGE2D_BATCH_DIR / f"attempt_{k:04d}_response.txt"
    retry_paths = sorted(STAGE2D_BATCH_DIR.glob(f"attempt_{k:04d}_retry_*_response.txt"))
    artifact_rows.append({
        "call": k,
        "theme": STAGE2D_CALL_BY_POSITION.get(k, {}).get("theme"),
        "prompt_exists": prompt_path.exists(),
        "response_exists": response_path.exists(),
        "retry_file_count": len(retry_paths),
        "summary_retry_count": STAGE2D_CALL_BY_POSITION.get(k, {}).get("retry_count", 0),
        "prompt_path": str(prompt_path),
        "response_path": str(response_path),
    })
stage2d_artifact_df = pd.DataFrame(artifact_rows)
display(stage2d_artifact_df.head())
display(stage2d_artifact_df.tail())

stage2d_artifact_checks = [
    check_row("final summary artifact exists", STAGE2D_SUMMARY_PATH.exists(), STAGE2D_SUMMARY_PATH),
    check_row("partial summary artifact exists", STAGE2D_PARTIAL_SUMMARY_PATH.exists(), STAGE2D_PARTIAL_SUMMARY_PATH),
    check_row("raw payload directory exists", STAGE2D_BATCH_DIR.is_dir(), STAGE2D_BATCH_DIR),
    check_row("summary batch id matches directory", STAGE2D_SUMMARY.get("batch_id") == STAGE2D_BATCH_ID, {"summary": STAGE2D_SUMMARY.get("batch_id"), "dir": STAGE2D_BATCH_ID}),
    check_row("summary contains 200 calls", len(STAGE2D_CALLS) == STAGE2D_BATCH_SIZE, len(STAGE2D_CALLS)),
    check_row("all 200 prompt files exist", bool(stage2d_artifact_df["prompt_exists"].all()), stage2d_artifact_df.loc[~stage2d_artifact_df["prompt_exists"]]),
    check_row("all 200 response files exist", bool(stage2d_artifact_df["response_exists"].all()), stage2d_artifact_df.loc[~stage2d_artifact_df["response_exists"]]),
    check_row("retry file counts match summary retry_count", bool((stage2d_artifact_df["retry_file_count"] == stage2d_artifact_df["summary_retry_count"]).all()), stage2d_artifact_df.loc[stage2d_artifact_df["retry_file_count"] != stage2d_artifact_df["summary_retry_count"]]),
    check_row("batch completed without truncation", STAGE2D_SUMMARY.get("batch_status") == "completed" and STAGE2D_SUMMARY.get("truncated") is False and STAGE2D_SUMMARY.get("unissued_slots") == 0, {"status": STAGE2D_SUMMARY.get("batch_status"), "truncated": STAGE2D_SUMMARY.get("truncated"), "unissued_slots": STAGE2D_SUMMARY.get("unissued_slots")}),
]
display_check_table(stage2d_artifact_checks, "Stage 2d batch artifact checks")
D6_STAGE2D_ACCEPTANCE["batch_artifacts_complete"] = True


,item,value
0,batch_id,5cf76668-47d1-48d7-bd90-db06d31982ed
1,summary path,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
2,partial summary path,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
3,git commit,141552e
4,run timestamp UTC,2026-04-17T19:00:35.719690+00:00
5,stage label,D6_STAGE2D
6,model,claude-sonnet-4-5
7,prompt caching enabled,False
8,batch size,200
9,batch cap USD,20.0


,call,theme,prompt_exists,response_exists,retry_file_count,summary_retry_count,prompt_path,response_path
0,1,momentum,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
1,2,mean_reversion,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
2,3,volatility_regime,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
3,4,volume_divergence,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
4,5,calendar_effect,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...


,call,theme,prompt_exists,response_exists,retry_file_count,summary_retry_count,prompt_path,response_path
195,196,momentum,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
196,197,mean_reversion,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
197,198,volatility_regime,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
198,199,volume_divergence,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
199,200,calendar_effect,True,True,0,0,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...


Stage 2d batch artifact checks


,check,status,detail
0,final summary artifact exists,PASS,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
1,partial summary artifact exists,PASS,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
2,raw payload directory exists,PASS,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...
3,summary batch id matches directory,PASS,{'summary': '5cf76668-47d1-48d7-bd90-db06d3198...
4,summary contains 200 calls,PASS,200
5,all 200 prompt files exist,PASS,"Empty DataFrame Columns: [call, theme, prompt_..."
6,all 200 response files exist,PASS,"Empty DataFrame Columns: [call, theme, prompt_..."
7,retry file counts match summary retry_count,PASS,"Empty DataFrame Columns: [call, theme, prompt_..."
8,batch completed without truncation,PASS,"{'status': 'completed', 'truncated': False, 'u..."


## AL. Batch-Level Config Block Audit

Stage 2d adds a durable config contract. The final summary and partial summary must both carry the correct config block so crash recovery and post-run review see the same batch identity.


In [39]:
expected_config = {
    "stage_label": STAGE2D_STAGE_LABEL,
    "model_name": STAGE2D_MODEL_NAME,
    "prompt_caching_enabled": STAGE2D_PROMPT_CACHING_ENABLED,
    "batch_size": STAGE2D_BATCH_SIZE,
    "batch_cap_usd": STAGE2D_BATCH_CAP_USD,
    "theme_rotation_mode": STAGE2D_THEME_ROTATION_MODE,
    "approved_examples_cap": STAGE2D_APPROVED_EXAMPLES_CAP,
    "parse_rate_gate_k5_threshold": STAGE2D_TIER1_THRESHOLD,
    "parse_rate_gate_k20_threshold": STAGE2D_TIER2_THRESHOLD,
}
config_rows = []
for source_name, cfg in [("final", STAGE2D_CONFIG), ("partial", STAGE2D_PARTIAL_CONFIG)]:
    for key, expected in expected_config.items():
        actual = cfg.get(key)
        config_rows.append({"source": source_name, "field": key, "expected": expected, "actual": actual, "match": actual == expected})
    config_rows.append({"source": source_name, "field": "git_commit", "expected": "non-empty", "actual": cfg.get("git_commit"), "match": bool(cfg.get("git_commit"))})
    config_rows.append({"source": source_name, "field": "batch_status", "expected": "legal status", "actual": cfg.get("batch_status"), "match": cfg.get("batch_status") in {"completed", "in_progress"} or str(cfg.get("batch_status", "")).startswith("crashed_at_call_")})
    config_rows.append({"source": source_name, "field": "batch_duration_seconds", "expected": "positive number", "actual": cfg.get("batch_duration_seconds"), "match": isinstance(cfg.get("batch_duration_seconds"), (int, float)) and cfg.get("batch_duration_seconds") > 0})
config_df = pd.DataFrame(config_rows)
display(config_df)

config_checks = [
    check_row("final and partial config fields match expected values", bool(config_df["match"].all()), config_df.loc[~config_df["match"]]),
    check_row("final config batch_status completed", STAGE2D_CONFIG.get("batch_status") == "completed", STAGE2D_CONFIG.get("batch_status")),
    check_row("partial config final state completed", STAGE2D_PARTIAL_CONFIG.get("batch_status") == "completed", STAGE2D_PARTIAL_CONFIG.get("batch_status")),
    check_row("partial summary last_completed_call is 200", STAGE2D_PARTIAL_SUMMARY.get("last_completed_call") == STAGE2D_BATCH_SIZE, STAGE2D_PARTIAL_SUMMARY.get("last_completed_call")),
]
display_check_table(config_checks, "Stage 2d config block checks")
D6_STAGE2D_ACCEPTANCE["config_block_correct"] = True


,source,field,expected,actual,match
0,final,stage_label,D6_STAGE2D,D6_STAGE2D,True
1,final,model_name,claude-sonnet-4-5,claude-sonnet-4-5,True
2,final,prompt_caching_enabled,False,False,True
3,final,batch_size,200,200,True
4,final,batch_cap_usd,20.0,20.0,True
5,final,theme_rotation_mode,interleaved_cyclic,interleaved_cyclic,True
6,final,approved_examples_cap,3,3,True
7,final,parse_rate_gate_k5_threshold,0.5,0.5,True
8,final,parse_rate_gate_k20_threshold,0.75,0.75,True
9,final,git_commit,non-empty,141552e,True


Stage 2d config block checks


,check,status,detail
0,final and partial config fields match expected...,PASS,"Empty DataFrame Columns: [source, field, expec..."
1,final config batch_status completed,PASS,completed
2,partial config final state completed,PASS,completed
3,partial summary last_completed_call is 200,PASS,200


## AM. Strict Sequential Execution Audit

Stage 2d only proves production-shape sequential semantics if each call completes before the next starts. This section checks the SQLite ledger directly with the strict relation `completed_at(call k) < created_at(call k+1)`.


In [40]:
def _stage2d_connect_ledger_readonly():
    return sqlite3.connect(f"file:{STAGE2D_LEDGER_PATH.resolve()}?mode=ro", uri=True)


def _stage2d_parse_ledger_ts(value: str) -> pd.Timestamp:
    return pd.Timestamp(str(value).replace("Z", "+00:00"))

with _stage2d_connect_ledger_readonly() as conn:
    conn.row_factory = sqlite3.Row
    stage2d_ledger_rows_raw = conn.execute(
        "SELECT * FROM ledger WHERE batch_id = ? ORDER BY created_at_utc ASC",
        (STAGE2D_BATCH_ID,),
    ).fetchall()
stage2d_ledger_rows = [dict(row) for row in stage2d_ledger_rows_raw]
stage2d_ledger_df = pd.DataFrame(stage2d_ledger_rows)
stage2d_ledger_df["call"] = range(1, len(stage2d_ledger_df) + 1)
stage2d_ledger_df["created_ts"] = stage2d_ledger_df["created_at_utc"].map(_stage2d_parse_ledger_ts)
stage2d_ledger_df["completed_ts"] = stage2d_ledger_df["completed_at_utc"].map(_stage2d_parse_ledger_ts)
stage2d_ledger_df["next_created_ts"] = stage2d_ledger_df["created_ts"].shift(-1)
stage2d_ledger_df["completed_strictly_before_next_created"] = (
    stage2d_ledger_df["next_created_ts"].isna()
    | (stage2d_ledger_df["completed_ts"] < stage2d_ledger_df["next_created_ts"])
)

display(stage2d_ledger_df[["call", "status", "estimated_cost", "actual_cost", "created_at_utc", "completed_at_utc", "completed_strictly_before_next_created"]].head())
display(stage2d_ledger_df[["call", "status", "estimated_cost", "actual_cost", "created_at_utc", "completed_at_utc", "completed_strictly_before_next_created"]].tail())

offending_pairs = stage2d_ledger_df.loc[~stage2d_ledger_df["completed_strictly_before_next_created"], ["call", "completed_at_utc", "next_created_ts"]]
sequential_checks = [
    check_row("ledger has 200 rows for this batch", len(stage2d_ledger_df) == STAGE2D_BATCH_SIZE, len(stage2d_ledger_df)),
    check_row("all issued calls completed", bool((stage2d_ledger_df["status"] == STATUS_COMPLETED).all()), stage2d_ledger_df["status"].value_counts().to_dict()),
    check_row("created timestamps are strictly increasing", bool(stage2d_ledger_df["created_ts"].is_monotonic_increasing and stage2d_ledger_df["created_ts"].is_unique)),
    check_row("completed timestamps are strictly increasing", bool(stage2d_ledger_df["completed_ts"].is_monotonic_increasing and stage2d_ledger_df["completed_ts"].is_unique)),
    check_row("call k completes strictly before call k+1 starts", bool(stage2d_ledger_df["completed_strictly_before_next_created"].all()), offending_pairs),
]
display_check_table(sequential_checks, "Stage 2d strict sequential execution checks")
D6_STAGE2D_ACCEPTANCE["strict_sequential_execution"] = True


,call,status,estimated_cost,actual_cost,created_at_utc,completed_at_utc,completed_strictly_before_next_created
0,1,completed,0.011475,0.006999,2026-04-17T19:00:35.747Z,2026-04-17T19:00:42.629Z,True
1,2,completed,0.011898,0.008205,2026-04-17T19:00:42.635Z,2026-04-17T19:00:47.317Z,True
2,3,completed,0.012477,0.009057,2026-04-17T19:00:47.321Z,2026-04-17T19:00:51.870Z,True
3,4,completed,0.013104,0.009315,2026-04-17T19:00:51.874Z,2026-04-17T19:00:56.560Z,True
4,5,completed,0.013221,0.009234,2026-04-17T19:00:56.566Z,2026-04-17T19:01:00.749Z,True


,call,status,estimated_cost,actual_cost,created_at_utc,completed_at_utc,completed_strictly_before_next_created
195,196,completed,0.013761,0.011109,2026-04-17T19:19:43.189Z,2026-04-17T19:19:48.905Z,True
196,197,completed,0.013710,0.011241,2026-04-17T19:19:48.922Z,2026-04-17T19:19:54.770Z,True
197,198,completed,0.013749,0.011262,2026-04-17T19:19:54.785Z,2026-04-17T19:20:00.986Z,True
198,199,completed,0.013824,0.011640,2026-04-17T19:20:01.003Z,2026-04-17T19:20:07.080Z,True
199,200,completed,0.013887,0.011763,2026-04-17T19:20:07.094Z,2026-04-17T19:20:13.992Z,True


Stage 2d strict sequential execution checks


,check,status,detail
0,ledger has 200 rows for this batch,PASS,200
1,all issued calls completed,PASS,{'completed': 200}
2,created timestamps are strictly increasing,PASS,
3,completed timestamps are strictly increasing,PASS,
4,call k completes strictly before call k+1 starts,PASS,"Empty DataFrame Columns: [call, completed_at_u..."


## AN. Prompt Safety And Rotation Audit For All 200 Calls

This is a full O(n) prompt audit. Every prompt must pass leakage audit, carry the expected rotating theme, preserve the factor/operator/complexity menu, and contain the correct approved-example count.


In [41]:
def _stage2d_prompt_example_jsons(prompt_text: str) -> list[dict]:
    examples = []
    in_block = False
    for line in prompt_text.splitlines():
        if "up to 3 example approved hypotheses" in line:
            in_block = True
            continue
        if in_block and line.startswith("Propose hypothesis"):
            break
        if in_block and line.strip().startswith("- {"):
            examples.append(json.loads(line.strip()[2:].strip()))
    return examples

stage2d_prompt_examples_by_call: dict[int, list[dict]] = {}
stage2d_prompt_text_by_call: dict[int, str] = {}
stage2d_prompt_audit_rows = []
for call in STAGE2D_CALLS:
    k = call["position"]
    prompt_path = STAGE2D_BATCH_DIR / f"attempt_{k:04d}_prompt.txt"
    text = prompt_path.read_text(encoding="utf-8")
    examples = _stage2d_prompt_example_jsons(text)
    findings = audit_prompt_for_leakage(text)
    expected_theme = STAGE2D_EXPECTED_THEMES[(k - 1) % STAGE2D_THEME_CYCLE_LEN]
    stage2d_prompt_examples_by_call[k] = examples
    stage2d_prompt_text_by_call[k] = text
    stage2d_prompt_audit_rows.append({
        "call": k,
        "summary_theme": call["theme"],
        "expected_theme": expected_theme,
        "theme_in_prompt": expected_theme if f"theme (rotating): {expected_theme}" in text else "MISSING",
        "leakage_clean": findings == [],
        "leakage_findings": findings,
        "examples_count_in_prompt": len(examples),
        "summary_examples_count": call["approved_examples_count_in_prompt_before_call"],
        "factor_menu_present": "# Available factors" in text and "volume_zscore_24h" in text,
        "operators_present": all(op in text for op in ALLOWED_OPS),
        "complexity_budget_present": "complexity budget" in text.lower() and "max_hold_bars" in text,
    })
stage2d_prompt_audit_df = pd.DataFrame(stage2d_prompt_audit_rows)
display(stage2d_prompt_audit_df.head())
display(stage2d_prompt_audit_df.tail())
display(stage2d_prompt_audit_df["examples_count_in_prompt"].value_counts().sort_index().rename("count").reset_index().rename(columns={"index": "examples_count"}))

expected_theme_sequence = [STAGE2D_EXPECTED_THEMES[(k - 1) % STAGE2D_THEME_CYCLE_LEN] for k in range(1, STAGE2D_BATCH_SIZE + 1)]
expected_example_counts = [min(k - 1, STAGE2D_APPROVED_EXAMPLES_CAP) for k in range(1, STAGE2D_BATCH_SIZE + 1)]
prompt_checks = [
    check_row("theme rotation matches (k - 1) % 5", stage2d_prompt_audit_df["expected_theme"].tolist() == expected_theme_sequence and stage2d_prompt_audit_df["summary_theme"].tolist() == expected_theme_sequence and stage2d_prompt_audit_df["theme_in_prompt"].tolist() == expected_theme_sequence, stage2d_prompt_audit_df[["call", "summary_theme", "expected_theme", "theme_in_prompt"]]),
    check_row("all 200 prompts pass leakage audit", bool(stage2d_prompt_audit_df["leakage_clean"].all()), stage2d_prompt_audit_df.loc[~stage2d_prompt_audit_df["leakage_clean"], ["call", "leakage_findings"]]),
    check_row("approved example counts follow cap=3", stage2d_prompt_audit_df["examples_count_in_prompt"].tolist() == expected_example_counts, stage2d_prompt_audit_df[["call", "examples_count_in_prompt"]]),
    check_row("approved example counts match summary", bool((stage2d_prompt_audit_df["examples_count_in_prompt"] == stage2d_prompt_audit_df["summary_examples_count"]).all()), stage2d_prompt_audit_df[["call", "examples_count_in_prompt", "summary_examples_count"]]),
    check_row("factor menu present in all prompts", bool(stage2d_prompt_audit_df["factor_menu_present"].all())),
    check_row("frozen operators present in all prompts", bool(stage2d_prompt_audit_df["operators_present"].all())),
    check_row("complexity budget present in all prompts", bool(stage2d_prompt_audit_df["complexity_budget_present"].all())),
]
display_check_table(prompt_checks, "Stage 2d prompt safety checks")
D6_STAGE2D_ACCEPTANCE["prompt_safety_preserved"] = True


,call,summary_theme,expected_theme,theme_in_prompt,leakage_clean,leakage_findings,examples_count_in_prompt,summary_examples_count,factor_menu_present,operators_present,complexity_budget_present
0,1,momentum,momentum,momentum,True,[],0,0,True,True,True
1,2,mean_reversion,mean_reversion,mean_reversion,True,[],1,1,True,True,True
2,3,volatility_regime,volatility_regime,volatility_regime,True,[],2,2,True,True,True
3,4,volume_divergence,volume_divergence,volume_divergence,True,[],3,3,True,True,True
4,5,calendar_effect,calendar_effect,calendar_effect,True,[],3,3,True,True,True


,call,summary_theme,expected_theme,theme_in_prompt,leakage_clean,leakage_findings,examples_count_in_prompt,summary_examples_count,factor_menu_present,operators_present,complexity_budget_present
195,196,momentum,momentum,momentum,True,[],3,3,True,True,True
196,197,mean_reversion,mean_reversion,mean_reversion,True,[],3,3,True,True,True
197,198,volatility_regime,volatility_regime,volatility_regime,True,[],3,3,True,True,True
198,199,volume_divergence,volume_divergence,volume_divergence,True,[],3,3,True,True,True
199,200,calendar_effect,calendar_effect,calendar_effect,True,[],3,3,True,True,True


,examples_count_in_prompt,count
0,0,1
1,1,1
2,2,1
3,3,197


Stage 2d prompt safety checks


,check,status,detail
0,theme rotation matches (k - 1) % 5,PASS,call summary_theme expected_them...
1,all 200 prompts pass leakage audit,PASS,"Empty DataFrame Columns: [call, leakage_findin..."
2,approved example counts follow cap=3,PASS,call examples_count_in_prompt 0 1 ...
3,approved example counts match summary,PASS,call examples_count_in_prompt summary_e...
4,factor menu present in all prompts,PASS,
5,frozen operators present in all prompts,PASS,
6,complexity budget present in all prompts,PASS,


## AO. Independent Response Reclassification And Sample Plan

This section re-reads all 200 raw response files, strips markdown fences, classifies DSL schema validity, re-ingests outputs into a fresh batch state, and computes the sample set used for deeper forensic checks.

The fixed random seed is `42` and must not change across notebook reruns.


In [42]:
def _stage2d_response_json_text(raw_text: str) -> str:
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", raw_text, flags=re.DOTALL | re.IGNORECASE)
    return fenced.group(1).strip() if fenced else raw_text.strip()


def _stage2d_classify_cardinality(raw_text: str) -> str:
    stripped = _stage2d_response_json_text(raw_text)
    if not stripped.strip():
        return "zero_objects"
    try:
        parsed = json.loads(stripped)
    except json.JSONDecodeError:
        if "{" in raw_text or "[" in raw_text:
            return "prose_plus_object"
        return "zero_objects"
    if isinstance(parsed, dict):
        return "single_object"
    if isinstance(parsed, list):
        return f"json_array_{len(parsed)}"
    return "single_object"


def _stage2d_extract_factors_from_dsl(dsl) -> list[str]:
    factors = set()
    for group in list(dsl.entry) + list(dsl.exit):
        for cond in group.conditions:
            factors.add(cond.factor)
            if isinstance(cond.value, str):
                factors.add(cond.value)
    return sorted(factors)

stage2d_response_candidates_by_call = {}
stage2d_response_dsl_by_call = {}
stage2d_cardinality_by_call = {}
stage2d_parse_rows = []
for call in STAGE2D_CALLS:
    k = call["position"]
    raw_text = (STAGE2D_BATCH_DIR / f"attempt_{k:04d}_response.txt").read_text(encoding="utf-8")
    parse_text = _stage2d_response_json_text(raw_text)
    cardinality = _stage2d_classify_cardinality(raw_text)
    cand = classify_raw_json(
        parse_text,
        registry=REGISTRY,
        error_kind_hint="stage2d_notebook_reclassify",
        provenance={"backend": "sonnet", "batch_id": STAGE2D_BATCH_ID, "position": k},
    )
    stage2d_response_candidates_by_call[k] = cand
    stage2d_cardinality_by_call[k] = cardinality
    is_valid = isinstance(cand, ValidCandidate)
    if is_valid:
        stage2d_response_dsl_by_call[k] = cand.dsl
    stage2d_parse_rows.append({
        "call": k,
        "fence_stripped": parse_text != raw_text.strip(),
        "valid_candidate": is_valid,
        "name": cand.dsl.name if is_valid else None,
        "cardinality_recomputed": cardinality,
        "cardinality_summary": call["cardinality"],
        "parse_error_excerpt": None if is_valid else getattr(cand, "parse_error", "")[:180],
    })
stage2d_parse_df = pd.DataFrame(stage2d_parse_rows)
display(stage2d_parse_df.head())
display(stage2d_parse_df[~stage2d_parse_df["valid_candidate"]])

stage2d_reclass_state = BatchIngestState(batch_id=STAGE2D_BATCH_ID)
stage2d_outputs = []
for call in STAGE2D_CALLS:
    k = call["position"]
    stage2d_outputs.append(ProposerOutput(
        candidates=(stage2d_response_candidates_by_call[k],),
        cost_estimate_usd=call["estimated_cost_usd"],
        cost_actual_usd=call["actual_cost_usd"],
        backend_name="sonnet",
        telemetry={"position": k, "batch_id": STAGE2D_BATCH_ID},
    ))
stage2d_reclass_records = ingest_outputs(stage2d_reclass_state, stage2d_outputs)
assert_lifecycle_invariant_at_batch_close(stage2d_reclass_state)

stage2d_reclass_rows = []
for call, rec in zip(STAGE2D_CALLS, stage2d_reclass_records):
    k = call["position"]
    cand = stage2d_response_candidates_by_call[k]
    dsl = cand.dsl if isinstance(cand, ValidCandidate) else None
    factors = _stage2d_extract_factors_from_dsl(dsl) if dsl is not None else []
    stage2d_reclass_rows.append({
        "call": k,
        "theme": call["theme"],
        "schema_valid": isinstance(cand, ValidCandidate),
        "reclassified_lifecycle": rec.lifecycle_state,
        "summary_lifecycle": call["lifecycle_state"],
        "reclassified_hash": rec.hypothesis_hash,
        "summary_hash": call["hypothesis_hash"],
        "name": dsl.name if dsl else None,
        "factor_count": len(factors),
        "factors": factors,
        "summary_factors": call["factors_used"],
        "cardinality_recomputed": stage2d_cardinality_by_call[k],
        "cardinality_summary": call["cardinality"],
    })
stage2d_reclass_df = pd.DataFrame(stage2d_reclass_rows)
display(stage2d_reclass_df.head())
display(stage2d_reclass_df[stage2d_reclass_df["reclassified_lifecycle"] != PENDING_BACKTEST])

valid_positions = stage2d_reclass_df.loc[stage2d_reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST, "call"].tolist()
invalid_positions = stage2d_reclass_df.loc[stage2d_reclass_df["reclassified_lifecycle"] != PENDING_BACKTEST, "call"].tolist()
retry_positions = [call["position"] for call in STAGE2D_CALLS if call.get("retry_count", 0) > 0]
summary_hash_counts = Counter(call.get("hypothesis_hash") for call in STAGE2D_CALLS if call.get("hypothesis_hash"))
duplicate_hash_positions = [call["position"] for call in STAGE2D_CALLS if call.get("hypothesis_hash") and summary_hash_counts[call.get("hypothesis_hash")] > 1]
empty_factor_positions = STAGE2D_SUMMARY.get("valid_with_empty_factor_set_calls", [])
repeated_factor_first_positions = [r["occurring_at_calls"][0] for r in STAGE2D_SUMMARY.get("repeated_factor_sets", []) if r.get("occurrence_count", 0) >= 3]

key_positions = [1, 4, 50, 100, 150, 200]
random_positions = sorted(random.sample(range(1, STAGE2D_BATCH_SIZE + 1), 15))
special_positions = []
for source in [invalid_positions[:1], retry_positions[:1], repeated_factor_first_positions[:1], duplicate_hash_positions[:1], empty_factor_positions[:1]]:
    special_positions.extend(source)
stage2d_sample_positions = sorted(set(key_positions + random_positions + special_positions))

sample_plan_df = pd.DataFrame([{
    "category": "key positions", "positions": key_positions,
}, {
    "category": "fixed random sample", "positions": random_positions,
}, {
    "category": "special strata", "positions": special_positions,
}, {
    "category": "combined audited sample", "positions": stage2d_sample_positions,
}])
display(sample_plan_df)

reclass_checks = [
    check_row("all 200 raw responses have single-object cardinality", bool((stage2d_reclass_df["cardinality_recomputed"] == "single_object").all()), stage2d_reclass_df[["call", "cardinality_recomputed"]]),
    check_row("recomputed cardinality matches summary", bool((stage2d_reclass_df["cardinality_recomputed"] == stage2d_reclass_df["cardinality_summary"]).all()), stage2d_reclass_df[["call", "cardinality_recomputed", "cardinality_summary"]]),
    check_row("reclassified lifecycle matches summary", bool((stage2d_reclass_df["reclassified_lifecycle"] == stage2d_reclass_df["summary_lifecycle"]).all()), stage2d_reclass_df[["call", "reclassified_lifecycle", "summary_lifecycle"]]),
    check_row("recomputed hashes match summary", bool((stage2d_reclass_df["reclassified_hash"].fillna("<NA>") == stage2d_reclass_df["summary_hash"].fillna("<NA>")).all()), stage2d_reclass_df[["call", "reclassified_hash", "summary_hash"]]),
    check_row("recomputed factor lists match summary", bool((stage2d_reclass_df["factors"].map(tuple) == stage2d_reclass_df["summary_factors"].map(tuple)).all()), stage2d_reclass_df[["call", "factors", "summary_factors"]]),
    check_row("sample plan includes key positions", set(key_positions).issubset(stage2d_sample_positions), stage2d_sample_positions),
]
display_check_table(reclass_checks, "Stage 2d response reclassification checks")
D6_STAGE2D_ACCEPTANCE["response_reclassification_consistent"] = True


,call,fence_stripped,valid_candidate,name,cardinality_recomputed,cardinality_summary,parse_error_excerpt
0,1,True,True,momentum_rsi_oversold_reversal,single_object,single_object,NaN
1,2,True,True,bb_mean_reversion_zscore,single_object,single_object,NaN
2,3,True,True,low_vol_trend_continuation,single_object,single_object,NaN
3,4,True,True,volume_divergence_breakout,single_object,single_object,NaN
4,5,True,True,weekend_mean_reversion,single_object,single_object,NaN


,call,fence_stripped,valid_candidate,name,cardinality_recomputed,cardinality_summary,parse_error_excerpt
115,116,True,False,NaN,single_object,single_object,1 validation error for StrategyDSL\ndescriptio...


,call,theme,schema_valid,reclassified_lifecycle,summary_lifecycle,reclassified_hash,summary_hash,name,factor_count,factors,summary_factors,cardinality_recomputed,cardinality_summary
0,1,momentum,True,pending_backtest,pending_backtest,0d4d6ce39746d2c2,0d4d6ce39746d2c2,momentum_rsi_oversold_reversal,2,"[return_24h, rsi_14]","[return_24h, rsi_14]",single_object,single_object
1,2,mean_reversion,True,pending_backtest,pending_backtest,925c75d78974e39f,925c75d78974e39f,bb_mean_reversion_zscore,5,"[bb_upper_24_2, close, return_24h, sma_24, zsc...","[bb_upper_24_2, close, return_24h, sma_24, zsc...",single_object,single_object
2,3,volatility_regime,True,pending_backtest,pending_backtest,af84d7c3e5abb23f,af84d7c3e5abb23f,low_vol_trend_continuation,7,"[close, ema_12, ema_26, macd_hist, realized_vo...","[close, ema_12, ema_26, macd_hist, realized_vo...",single_object,single_object
3,4,volume_divergence,True,pending_backtest,pending_backtest,3b3a146fd8fd69eb,3b3a146fd8fd69eb,volume_divergence_breakout,5,"[close, ema_12, macd_hist, return_24h, volume_...","[close, ema_12, macd_hist, return_24h, volume_...",single_object,single_object
4,5,calendar_effect,True,pending_backtest,pending_backtest,27beb128fab7a8f7,27beb128fab7a8f7,weekend_mean_reversion,3,"[day_of_week, rsi_14, zscore_48]","[day_of_week, rsi_14, zscore_48]",single_object,single_object


,call,theme,schema_valid,reclassified_lifecycle,summary_lifecycle,reclassified_hash,summary_hash,name,factor_count,factors,summary_factors,cardinality_recomputed,cardinality_summary
115,116,momentum,False,rejected_complexity,rejected_complexity,NaN,NaN,NaN,0,[],[],single_object,single_object


,category,positions
0,key positions,"[1, 4, 50, 100, 150, 200]"
1,fixed random sample,"[7, 23, 27, 29, 36, 58, 63, 71, 109, 140, 152,..."
2,special strata,"[116, 17]"
3,combined audited sample,"[1, 4, 7, 17, 23, 27, 29, 36, 50, 58, 63, 71, ..."


Stage 2d response reclassification checks


,check,status,detail
0,all 200 raw responses have single-object cardi...,PASS,call cardinality_recomputed 0 1 ...
1,recomputed cardinality matches summary,PASS,call cardinality_recomputed cardinality_s...
2,reclassified lifecycle matches summary,PASS,call reclassified_lifecycle summary_lifec...
3,recomputed hashes match summary,PASS,call reclassified_hash summary_hash ...
4,recomputed factor lists match summary,PASS,call ...
5,sample plan includes key positions,PASS,"[1, 4, 7, 17, 23, 27, 29, 36, 50, 58, 63, 71, ..."


## AP. Approved Examples Sliding-Window Identity Audit

This section performs identity-level checks for key positions and the random/stratified sample. The expected examples are the latest three prior `pending_backtest` DSLs, in most-recent-first order.


In [43]:
def _stage2d_expected_example_calls(k: int) -> list[int]:
    prior_valid = [pos for pos in range(1, k) if stage2d_reclass_df.loc[stage2d_reclass_df["call"] == pos, "reclassified_lifecycle"].iloc[0] == PENDING_BACKTEST]
    return list(reversed(prior_valid[-STAGE2D_APPROVED_EXAMPLES_CAP:]))

stage2d_example_rows = []
for k in stage2d_sample_positions:
    actual_examples = stage2d_prompt_examples_by_call[k]
    actual_names = [ex.get("name") for ex in actual_examples]
    actual_desc = [ex.get("description") for ex in actual_examples]
    expected_calls = _stage2d_expected_example_calls(k)
    expected_names = [stage2d_response_dsl_by_call[pos].name for pos in expected_calls]
    expected_desc = [stage2d_response_dsl_by_call[pos].description for pos in expected_calls]
    summary_trace = STAGE2D_SUMMARY["approved_examples_trace"][k - 1]
    stage2d_example_rows.append({
        "call": k,
        "expected_prior_valid_calls": expected_calls,
        "actual_example_names": actual_names,
        "expected_example_names": expected_names,
        "actual_count": len(actual_names),
        "expected_count": min(len([pos for pos in range(1, k) if pos in stage2d_response_dsl_by_call]), STAGE2D_APPROVED_EXAMPLES_CAP),
        "identity_match": actual_names == expected_names and actual_desc == expected_desc,
        "summary_count": summary_trace["count_in_prompt"],
        "summary_names": summary_trace["names"],
    })
stage2d_example_df = pd.DataFrame(stage2d_example_rows)
display(stage2d_example_df)

key_example_df = stage2d_example_df[stage2d_example_df["call"].isin(key_positions)]
display(key_example_df[["call", "expected_prior_valid_calls", "actual_example_names", "expected_example_names", "identity_match"]])

example_checks = [
    check_row("key positions have identity-correct approved examples", bool(key_example_df["identity_match"].all()), key_example_df),
    check_row("sample positions have identity-correct approved examples", bool(stage2d_example_df["identity_match"].all()), stage2d_example_df[["call", "actual_example_names", "expected_example_names", "identity_match"]]),
    check_row("sample example counts match expected cap", bool((stage2d_example_df["actual_count"] == stage2d_example_df["expected_count"]).all()), stage2d_example_df[["call", "actual_count", "expected_count"]]),
    check_row("sample example counts match summary trace", bool((stage2d_example_df["actual_count"] == stage2d_example_df["summary_count"]).all()), stage2d_example_df[["call", "actual_count", "summary_count"]]),
    check_row("sample example names match summary trace", bool((stage2d_example_df["actual_example_names"].map(tuple) == stage2d_example_df["summary_names"].map(tuple)).all()), stage2d_example_df[["call", "actual_example_names", "summary_names"]]),
]
display_check_table(example_checks, "Stage 2d approved_examples identity checks")
D6_STAGE2D_ACCEPTANCE["sliding_window_identity"] = True


,call,expected_prior_valid_calls,actual_example_names,expected_example_names,actual_count,expected_count,identity_match,summary_count,summary_names
0,1,[],[],[],0,0,True,0,[]
1,4,"[3, 2, 1]","[low_vol_trend_continuation, bb_mean_reversion...","[low_vol_trend_continuation, bb_mean_reversion...",3,3,True,3,"[low_vol_trend_continuation, bb_mean_reversion..."
2,7,"[6, 5, 4]","[macd_crossover_momentum, weekend_mean_reversi...","[macd_crossover_momentum, weekend_mean_reversi...",3,3,True,3,"[macd_crossover_momentum, weekend_mean_reversi..."
3,17,"[16, 15, 14]","[macd_momentum_surge, monday_close_long, volum...","[macd_momentum_surge, monday_close_long, volum...",3,3,True,3,"[macd_momentum_surge, monday_close_long, volum..."
4,23,"[22, 21, 20]","[bb_reversion_oversold, momentum_continuation_...","[bb_reversion_oversold, momentum_continuation_...",3,3,True,3,"[bb_reversion_oversold, momentum_continuation_..."
5,27,"[26, 25, 24]","[momentum_exhaustion_rsi_divergence, monday_di...","[momentum_exhaustion_rsi_divergence, monday_di...",3,3,True,3,"[momentum_exhaustion_rsi_divergence, monday_di..."
6,29,"[28, 27, 26]","[vol_spike_fade_regime_shift, bb_squeeze_rever...","[vol_spike_fade_regime_shift, bb_squeeze_rever...",3,3,True,3,"[vol_spike_fade_regime_shift, bb_squeeze_rever..."
7,36,"[35, 34, 33]","[weekday_momentum_continuation, volume_diverge...","[weekday_momentum_continuation, volume_diverge...",3,3,True,3,"[weekday_momentum_continuation, volume_diverge..."
8,50,"[49, 48, 47]","[volume_divergence_momentum_entry, vol_regime_...","[volume_divergence_momentum_entry, vol_regime_...",3,3,True,3,"[volume_divergence_momentum_entry, vol_regime_..."
9,58,"[57, 56, 55]","[zscore_mean_reversion_oversold_entry, macd_hi...","[zscore_mean_reversion_oversold_entry, macd_hi...",3,3,True,3,"[zscore_mean_reversion_oversold_entry, macd_hi..."


,call,expected_prior_valid_calls,actual_example_names,expected_example_names,identity_match
0,1,[],[],[],True
1,4,"[3, 2, 1]","[low_vol_trend_continuation, bb_mean_reversion...","[low_vol_trend_continuation, bb_mean_reversion...",True
8,50,"[49, 48, 47]","[volume_divergence_momentum_entry, vol_regime_...","[volume_divergence_momentum_entry, vol_regime_...",True
12,100,"[99, 98, 97]","[volume_spike_reversal_low_vol_regime, low_vol...","[volume_spike_reversal_low_vol_regime, low_vol...",True
16,150,"[149, 148, 147]","[volume_divergence_reversal_149, volatility_co...","[volume_divergence_reversal_149, volatility_co...",True
22,200,"[199, 198, 197]","[volume_divergence_reversal_199, volatility_br...","[volume_divergence_reversal_199, volatility_br...",True


Stage 2d approved_examples identity checks


,check,status,detail
0,key positions have identity-correct approved e...,PASS,call expected_prior_valid_calls \ 0 ...
1,sample positions have identity-correct approve...,PASS,call actual_...
2,sample example counts match expected cap,PASS,call actual_count expected_count 0 ...
3,sample example counts match summary trace,PASS,call actual_count summary_count 0 1...
4,sample example names match summary trace,PASS,call actual_...


## AQ. Aggregate Recalculation Audit For All 200 Calls

This is the core Stage 2d audit. It rebuilds lifecycle/cardinality distributions, per-theme aggregates, factor-set saturation, 50-call block trends, and interim snapshots from the independently reclassified calls and compares them to the final summary.


In [44]:
def _stage2d_factor_overlap_metrics(theme: str, factors: list[str]) -> dict:
    factor_set = set(factors)
    hints = set(STAGE2D_THEME_HINTS.get(theme, frozenset()))
    overlap = factor_set & hints
    default_hits = sorted(factor_set & set(STAGE2D_DEFAULT_MOMENTUM_FACTORS))
    return {
        "overlap_count": len(overlap),
        "overlap_ratio": (len(overlap) / len(factors)) if factors else None,
        "out_of_theme_factor_count": len(factors) - len(overlap),
        "contains_default_momentum_factor": bool(default_hits),
        "default_momentum_factors_used": default_hits,
    }

stage2d_recomputed_calls = []
for call in STAGE2D_CALLS:
    k = call["position"]
    re_row = stage2d_reclass_df.loc[stage2d_reclass_df["call"] == k].iloc[0]
    factors = list(re_row["factors"])
    if re_row["reclassified_lifecycle"] == PENDING_BACKTEST:
        tel = _stage2d_factor_overlap_metrics(call["theme"], factors)
    else:
        tel = {
            "overlap_count": 0,
            "overlap_ratio": None,
            "out_of_theme_factor_count": 0,
            "contains_default_momentum_factor": False,
            "default_momentum_factors_used": [],
        }
    stage2d_recomputed_calls.append({
        **call,
        "lifecycle_state": re_row["reclassified_lifecycle"],
        "hypothesis_hash": None if pd.isna(re_row["reclassified_hash"]) else re_row["reclassified_hash"],
        "factors_used": factors,
        **tel,
    })

# Headline aggregates
attempted_calls = [c for c in stage2d_recomputed_calls if not c.get("truncated_by_cap")]
valid_calls = [c for c in attempted_calls if c.get("lifecycle_state") == PENDING_BACKTEST]
headline = {
    "hypotheses_attempted": len(attempted_calls),
    "lifecycle_counts": dict(Counter(c["lifecycle_state"] for c in attempted_calls)),
    "parse_rate": len(valid_calls) / len(attempted_calls),
    "cardinality_distribution": dict(Counter(c["cardinality"] for c in attempted_calls)),
    "cardinality_violation_count": sum(1 for c in attempted_calls if c["cardinality"] != "single_object"),
    "total_estimated_cost_usd": sum(c["estimated_cost_usd"] for c in attempted_calls if c.get("estimated_cost_usd") is not None),
    "total_actual_cost_usd": sum(c["actual_cost_usd"] for c in attempted_calls if c.get("actual_cost_usd") is not None),
}
display(pd.DataFrame([headline]).T.rename(columns={0: "recomputed"}))

# Per-theme aggregates
per_theme_rows = []
for theme in STAGE2D_EXPECTED_THEMES:
    theme_calls = [c for c in attempted_calls if c["theme"] == theme]
    theme_valid = [c for c in theme_calls if c["lifecycle_state"] == PENDING_BACKTEST]
    factor_counter = Counter()
    for c in theme_valid:
        factor_counter.update(c["factors_used"])
    per_theme_rows.append({
        "theme": theme,
        "n_calls": len(theme_calls),
        "valid_count": len(theme_valid),
        "lifecycle_mix": dict(Counter(c["lifecycle_state"] for c in theme_calls)),
        "avg_overlap_count": sum(c["overlap_count"] for c in theme_valid) / len(theme_valid) if theme_valid else None,
        "avg_overlap_ratio": sum(c["overlap_ratio"] for c in theme_valid if c["overlap_ratio"] is not None) / len([c for c in theme_valid if c["overlap_ratio"] is not None]) if any(c["overlap_ratio"] is not None for c in theme_valid) else None,
        "avg_out_of_theme_factors": sum(c["out_of_theme_factor_count"] for c in theme_valid) / len(theme_valid) if theme_valid else None,
        "contains_rsi14_count": sum(1 for c in theme_valid if "rsi_14" in c["factors_used"]),
        "contains_momentum_default_count": sum(1 for c in theme_valid if c["contains_default_momentum_factor"]),
        "dominant_factors_top3": [f for f, _ in sorted(factor_counter.items(), key=lambda kv: (-kv[1], kv[0]))[:3]],
    })
per_theme_recomputed_df = pd.DataFrame(per_theme_rows)
per_theme_summary_df = pd.DataFrame(STAGE2D_SUMMARY["per_theme"])
per_theme_compare_df = per_theme_recomputed_df.merge(per_theme_summary_df, on="theme", suffixes=("_recomputed", "_summary"))
display(per_theme_compare_df)

# Factor-set saturation
valid_hashes = [c["hypothesis_hash"] for c in valid_calls if c.get("hypothesis_hash")]
non_empty_valid = [c for c in valid_calls if c.get("factors_used")]
empty_factor_calls = [c["position"] for c in valid_calls if not c.get("factors_used")]
factor_sets = [tuple(sorted(c["factors_used"])) for c in non_empty_valid]
fs_counter = Counter(factor_sets)
repeated_factor_sets = []
for fs, count in fs_counter.most_common():
    if count < 2:
        break
    matching = [c for c in non_empty_valid if tuple(sorted(c["factors_used"])) == fs]
    repeated_factor_sets.append({
        "factor_set": list(fs),
        "occurrence_count": count,
        "distinct_hashes_within_factor_set": len({c.get("hypothesis_hash") for c in matching}),
        "occurring_at_calls": [c["position"] for c in matching],
        "themes_used_in": [c["theme"] for c in matching],
    })
repeated_factor_sets.sort(key=lambda r: (-r["occurrence_count"], r["occurring_at_calls"][0]))
factor_set_recomputed = {
    "total_valid_count": len(valid_calls),
    "distinct_hash_count": len(set(valid_hashes)),
    "valid_with_empty_factor_set_count": len(empty_factor_calls),
    "valid_with_empty_factor_set_calls": empty_factor_calls,
    "distinct_factor_set_count": len(set(factor_sets)),
    "unique_factor_set_ratio": round(len(set(factor_sets)) / (len(valid_calls) - len(empty_factor_calls)), 4) if len(valid_calls) > len(empty_factor_calls) else None,
    "repeated_factor_sets": repeated_factor_sets,
}
display(pd.DataFrame([{k: v for k, v in factor_set_recomputed.items() if k != "repeated_factor_sets"}]))
display(pd.DataFrame(repeated_factor_sets).head(10))

# Block trends
def _stage2d_block_trends_from_calls(calls: list[dict]) -> list[dict]:
    rows = []
    for b in range(STAGE2D_BATCH_SIZE // STAGE2D_BLOCK_SIZE):
        start = b * STAGE2D_BLOCK_SIZE
        end = start + STAGE2D_BLOCK_SIZE
        block_calls = [c for c in calls[start:end] if not c.get("truncated_by_cap")]
        in_toks = [c["input_tokens"] for c in block_calls if c.get("input_tokens") is not None]
        out_toks = [c["output_tokens"] for c in block_calls if c.get("output_tokens") is not None]
        costs = [c["actual_cost_usd"] for c in block_calls if c.get("actual_cost_usd") is not None]
        valid_in_block = [c for c in block_calls if c.get("lifecycle_state") == PENDING_BACKTEST]
        fs_in_block = {tuple(sorted(c["factors_used"])) for c in valid_in_block if c.get("factors_used")}
        rows.append({
            "block": b + 1,
            "range": f"{start + 1}-{end}",
            "complete": len(block_calls) == STAGE2D_BLOCK_SIZE,
            "call_count": len(block_calls),
            "input_tokens_mean": round(sum(in_toks) / len(in_toks), 1) if in_toks else None,
            "input_tokens_median": round(float(np.median(in_toks)), 1) if in_toks else None,
            "input_tokens_max": max(in_toks) if in_toks else None,
            "output_tokens_mean": round(sum(out_toks) / len(out_toks), 1) if out_toks else None,
            "output_tokens_median": round(float(np.median(out_toks)), 1) if out_toks else None,
            "output_tokens_max": max(out_toks) if out_toks else None,
            "actual_cost_mean": round(sum(costs) / len(costs), 6) if costs else None,
            "actual_cost_total": round(sum(costs), 6) if costs else 0.0,
            "valid_count": len(valid_in_block),
            "valid_rate": round(len(valid_in_block) / len(block_calls), 4) if block_calls else None,
            "distinct_factor_set_count_within_block": len(fs_in_block),
        })
    return rows
block_trends_recomputed = _stage2d_block_trends_from_calls(stage2d_recomputed_calls)
block_compare_df = pd.DataFrame(block_trends_recomputed).merge(pd.DataFrame(STAGE2D_SUMMARY["block_trends"]), on="block", suffixes=("_recomputed", "_summary"))
display(block_compare_df)

# Interim snapshots
def _stage2d_interim_snapshot_from_calls(calls: list[dict], at_call: int, prev_snapshot_call: int) -> dict:
    cum_calls = [c for c in calls[:at_call] if not c.get("truncated_by_cap")]
    valid = [c for c in cum_calls if c.get("lifecycle_state") == PENDING_BACKTEST]
    hashes = {c.get("hypothesis_hash") for c in valid if c.get("hypothesis_hash")}
    non_empty_valid = [c for c in valid if c.get("factors_used")]
    fs = {tuple(sorted(c.get("factors_used", []))) for c in non_empty_valid}
    ratio = round(len(fs) / len(non_empty_valid), 4) if non_empty_valid else None
    costs = [c.get("actual_cost_usd", 0) for c in cum_calls if c.get("actual_cost_usd") is not None]
    since_last = [c for c in calls[prev_snapshot_call:at_call] if not c.get("truncated_by_cap")]
    in_since = [c["input_tokens"] for c in since_last if c.get("input_tokens") is not None]
    out_since = [c["output_tokens"] for c in since_last if c.get("output_tokens") is not None]
    return {
        "at_call": at_call,
        "cumulative_valid_count": len(valid),
        "cumulative_distinct_hash_count": len(hashes),
        "cumulative_distinct_factor_set_count": len(fs),
        "cumulative_unique_factor_set_ratio": ratio,
        "cumulative_actual_cost_usd": round(sum(costs), 6),
        "cumulative_lifecycle_mix": dict(Counter(c.get("lifecycle_state") or "unknown" for c in cum_calls)),
        "avg_input_tokens_since_last_snapshot": round(sum(in_since) / len(in_since), 1) if in_since else None,
        "avg_output_tokens_since_last_snapshot": round(sum(out_since) / len(out_since), 1) if out_since else None,
    }
interim_recomputed = []
prev_snapshot = 0
for snap_k in STAGE2D_SNAPSHOT_POSITIONS:
    interim_recomputed.append(_stage2d_interim_snapshot_from_calls(stage2d_recomputed_calls, snap_k, prev_snapshot))
    prev_snapshot = snap_k
interim_compare_df = pd.DataFrame(interim_recomputed).merge(pd.DataFrame(STAGE2D_SUMMARY["interim_snapshots"]), on="at_call", suffixes=("_recomputed", "_summary"))
display(interim_compare_df)

# Comparisons
summary_repeated_slim = STAGE2D_SUMMARY["repeated_factor_sets"]
headline_checks = [
    check_row("hypotheses_attempted matches", headline["hypotheses_attempted"] == STAGE2D_SUMMARY["hypotheses_attempted"], {"recomputed": headline["hypotheses_attempted"], "summary": STAGE2D_SUMMARY["hypotheses_attempted"]}),
    check_row("lifecycle_counts match", headline["lifecycle_counts"] == STAGE2D_SUMMARY["lifecycle_counts"], {"recomputed": headline["lifecycle_counts"], "summary": STAGE2D_SUMMARY["lifecycle_counts"]}),
    check_row("parse_rate matches", abs(headline["parse_rate"] - STAGE2D_SUMMARY["parse_rate"]) < 1e-12, {"recomputed": headline["parse_rate"], "summary": STAGE2D_SUMMARY["parse_rate"]}),
    check_row("cardinality distribution matches", headline["cardinality_distribution"] == STAGE2D_SUMMARY["cardinality_distribution"], {"recomputed": headline["cardinality_distribution"], "summary": STAGE2D_SUMMARY["cardinality_distribution"]}),
    check_row("total estimated cost matches", abs(headline["total_estimated_cost_usd"] - STAGE2D_SUMMARY["total_estimated_cost_usd"]) < 1e-12, {"recomputed": headline["total_estimated_cost_usd"], "summary": STAGE2D_SUMMARY["total_estimated_cost_usd"]}),
    check_row("total actual cost matches", abs(headline["total_actual_cost_usd"] - STAGE2D_SUMMARY["total_actual_cost_usd"]) < 1e-12, {"recomputed": headline["total_actual_cost_usd"], "summary": STAGE2D_SUMMARY["total_actual_cost_usd"]}),
]
display_check_table(headline_checks, "Stage 2d headline aggregate checks")

per_theme_checks = [
    check_row("per-theme n_calls match", bool((per_theme_compare_df["n_calls_recomputed"] == per_theme_compare_df["n_calls_summary"]).all()), per_theme_compare_df[["theme", "n_calls_recomputed", "n_calls_summary"]]),
    check_row("per-theme valid_count match", bool((per_theme_compare_df["valid_count_recomputed"] == per_theme_compare_df["valid_count_summary"]).all()), per_theme_compare_df[["theme", "valid_count_recomputed", "valid_count_summary"]]),
    check_row("per-theme lifecycle_mix match", bool((per_theme_compare_df["lifecycle_mix_recomputed"].map(str) == per_theme_compare_df["lifecycle_mix_summary"].map(str)).all()), per_theme_compare_df[["theme", "lifecycle_mix_recomputed", "lifecycle_mix_summary"]]),
    check_row("per-theme averages match", bool(per_theme_compare_df.apply(lambda row: abs(row["avg_overlap_count_recomputed"] - row["avg_overlap_count_summary"]) < 1e-12 and abs(row["avg_overlap_ratio_recomputed"] - row["avg_overlap_ratio_summary"]) < 1e-12 and abs(row["avg_out_of_theme_factors_recomputed"] - row["avg_out_of_theme_factors_summary"]) < 1e-12, axis=1).all()), per_theme_compare_df),
    check_row("per-theme dominant factors match", bool((per_theme_compare_df["dominant_factors_top3_recomputed"].map(tuple) == per_theme_compare_df["dominant_factors_top3_summary"].map(tuple)).all()), per_theme_compare_df[["theme", "dominant_factors_top3_recomputed", "dominant_factors_top3_summary"]]),
]
display_check_table(per_theme_checks, "Stage 2d per-theme aggregate checks")

factor_set_checks = [
    check_row("total_valid_count matches", factor_set_recomputed["total_valid_count"] == STAGE2D_SUMMARY["total_valid_count"]),
    check_row("distinct_hash_count matches", factor_set_recomputed["distinct_hash_count"] == STAGE2D_SUMMARY["distinct_hash_count"]),
    check_row("distinct_factor_set_count matches", factor_set_recomputed["distinct_factor_set_count"] == STAGE2D_SUMMARY["distinct_factor_set_count"]),
    check_row("unique_factor_set_ratio matches", factor_set_recomputed["unique_factor_set_ratio"] == STAGE2D_SUMMARY["unique_factor_set_ratio"]),
    check_row("empty factor-set calls match", factor_set_recomputed["valid_with_empty_factor_set_calls"] == STAGE2D_SUMMARY["valid_with_empty_factor_set_calls"]),
    check_row("repeated_factor_sets match", repeated_factor_sets == summary_repeated_slim, {"recomputed_count": len(repeated_factor_sets), "summary_count": len(summary_repeated_slim)}),
]
display_check_table(factor_set_checks, "Stage 2d factor-set saturation checks")

block_checks = [
    check_row("block trends match summary", block_trends_recomputed == STAGE2D_SUMMARY["block_trends"], block_compare_df),
]
display_check_table(block_checks, "Stage 2d block trend checks")

interim_checks = [
    check_row("interim snapshots match summary", interim_recomputed == STAGE2D_SUMMARY["interim_snapshots"], interim_compare_df),
]
display_check_table(interim_checks, "Stage 2d interim snapshot checks")
D6_STAGE2D_ACCEPTANCE["aggregate_recomputation_matches"] = True


,recomputed
hypotheses_attempted,200
lifecycle_counts,"{'pending_backtest': 199, 'rejected_complexity..."
parse_rate,0.995
cardinality_distribution,{'single_object': 200}
cardinality_violation_count,0
total_estimated_cost_usd,2.756349
total_actual_cost_usd,2.243343


,theme,n_calls_recomputed,valid_count_recomputed,lifecycle_mix_recomputed,avg_overlap_count_recomputed,avg_overlap_ratio_recomputed,avg_out_of_theme_factors_recomputed,contains_rsi14_count_recomputed,contains_momentum_default_count_recomputed,dominant_factors_top3_recomputed,n_calls_summary,valid_count_summary,lifecycle_mix_summary,avg_overlap_count_summary,avg_overlap_ratio_summary,avg_out_of_theme_factors_summary,contains_rsi14_count_summary,contains_momentum_default_count_summary,dominant_factors_top3_summary
0,momentum,40,39,"{'pending_backtest': 39, 'rejected_complexity'...",3.179487,0.662943,1.820513,38,39,"[macd_hist, rsi_14, return_24h]",40,39,"{'pending_backtest': 39, 'rejected_complexity'...",3.179487,0.662943,1.820513,38,39,"[macd_hist, rsi_14, return_24h]"
1,mean_reversion,40,40,{'pending_backtest': 40},2.900000,0.481071,3.200000,39,40,"[close, bb_upper_24_2, rsi_14]",40,40,{'pending_backtest': 40},2.900000,0.481071,3.200000,39,40,"[close, bb_upper_24_2, rsi_14]"
2,volatility_regime,40,40,{'pending_backtest': 40},1.150000,0.194821,4.825000,30,40,"[realized_vol_24h, close, return_24h]",40,40,{'pending_backtest': 40},1.150000,0.194821,4.825000,30,40,"[realized_vol_24h, close, return_24h]"
3,volume_divergence,40,40,{'pending_backtest': 40},1.000000,0.192440,4.425000,39,40,"[volume_zscore_24h, rsi_14, return_24h]",40,40,{'pending_backtest': 40},1.000000,0.192440,4.425000,39,40,"[volume_zscore_24h, rsi_14, return_24h]"
4,calendar_effect,40,40,{'pending_backtest': 40},1.150000,0.208780,4.500000,39,40,"[day_of_week, rsi_14, return_24h]",40,40,{'pending_backtest': 40},1.150000,0.208780,4.500000,39,40,"[day_of_week, rsi_14, return_24h]"


,total_valid_count,distinct_hash_count,valid_with_empty_factor_set_count,valid_with_empty_factor_set_calls,distinct_factor_set_count,unique_factor_set_ratio
0,199,199,0,[],133,0.6683


,factor_set,occurrence_count,distinct_hashes_within_factor_set,occurring_at_calls,themes_used_in
0,"[bb_upper_24_2, close, realized_vol_24h, retur...",8,8,"[17, 27, 102, 112, 147, 152, 162, 197]","[mean_reversion, mean_reversion, mean_reversio..."
1,"[bb_upper_24_2, close, realized_vol_24h, retur...",6,6,"[72, 87, 92, 142, 157, 192]","[mean_reversion, mean_reversion, mean_reversio..."
2,"[bb_upper_24_2, close, return_24h, rsi_14, sma...",5,5,"[32, 82, 97, 122, 127]","[mean_reversion, mean_reversion, mean_reversio..."
3,"[macd_hist, return_24h, rsi_14, volume_zscore_...",5,5,"[36, 41, 44, 49, 131]","[momentum, momentum, volume_divergence, volume..."
4,"[close, return_24h, rsi_14, sma_50, volume_zsc...",5,5,"[39, 149, 154, 159, 169]","[volume_divergence, volume_divergence, volume_..."
5,"[close, ema_26, macd_hist, return_24h, rsi_14]",5,5,"[51, 156, 171, 176, 186]","[momentum, momentum, momentum, momentum, momen..."
6,"[close, return_1h, return_24h, rsi_14, sma_50,...",4,4,"[9, 84, 94, 109]","[volume_divergence, volume_divergence, volume_..."
7,"[close, day_of_week, return_24h, rsi_14, sma_20]",4,4,"[35, 80, 185, 195]","[calendar_effect, calendar_effect, calendar_ef..."
8,"[bb_upper_24_2, close, return_24h, rsi_14, zsc...",4,4,"[37, 42, 47, 137]","[mean_reversion, mean_reversion, mean_reversio..."
9,"[close, return_1h, return_24h, rsi_14, sma_20,...",4,4,"[64, 69, 89, 199]","[volume_divergence, volume_divergence, volume_..."


,block,range_recomputed,complete_recomputed,call_count_recomputed,input_tokens_mean_recomputed,input_tokens_median_recomputed,input_tokens_max_recomputed,output_tokens_mean_recomputed,output_tokens_median_recomputed,output_tokens_max_recomputed,...,input_tokens_median_summary,input_tokens_max_summary,output_tokens_mean_summary,output_tokens_median_summary,output_tokens_max_summary,actual_cost_mean_summary,actual_cost_total_summary,valid_count_summary,valid_rate_summary,distinct_factor_set_count_within_block_summary
0,1,1-50,True,50,1979.8,2036.5,2067,342.8,356.0,372,...,2036.5,2067,342.8,356.0,372,0.011082,0.554079,50,1.00,41
1,2,51-100,True,50,2031.8,2038.5,2106,350.8,350.0,385,...,2038.5,2106,350.8,350.0,385,0.011357,0.567837,50,1.00,43
2,3,101-150,True,50,2023.1,2016.0,2095,345.7,345.0,383,...,2016.0,2095,345.7,345.0,383,0.011255,0.562731,49,0.98,45
3,4,151-200,True,50,2011.7,2009.5,2053,342.6,341.5,374,...,2009.5,2053,342.6,341.5,374,0.011174,0.558696,50,1.00,39


...

,check,status,detail
0,total_valid_count matches,PASS,
1,distinct_hash_count matches,PASS,
2,distinct_factor_set_count matches,PASS,
3,unique_factor_set_ratio matches,PASS,
4,empty factor-set calls match,PASS,
5,repeated_factor_sets match,PASS,"{'recomputed_count': 32, 'summary_count': 32}"


Stage 2d block trend checks


,check,status,detail
0,block trends match summary,PASS,block range_recomputed complete_recomputed...


Stage 2d interim snapshot checks


,check,status,detail
0,interim snapshots match summary,PASS,at_call cumulative_valid_count_recomputed ...


## AR. Ledger, Cost, And Retry Audit

This section ties all costs back to SQLite ledger rows and verifies retry artifacts. There were no retries in this batch, so retry checks should resolve to PASS / N/A.


In [45]:
stage2d_ledger_cost_df = stage2d_ledger_df[["call", "estimated_cost", "actual_cost", "status", "created_at_utc", "completed_at_utc"]].copy()
stage2d_summary_cost_df = pd.DataFrame([
    {
        "call": call["position"],
        "summary_estimated": call["estimated_cost_usd"],
        "summary_actual": call["actual_cost_usd"],
        "input_tokens": call["input_tokens"],
        "output_tokens": call["output_tokens"],
        "retry_count": call["retry_count"],
    }
    for call in STAGE2D_CALLS
])
stage2d_cost_join_df = stage2d_ledger_cost_df.merge(stage2d_summary_cost_df, on="call", how="inner")
stage2d_cost_join_df["estimated_matches"] = (stage2d_cost_join_df["estimated_cost"] - stage2d_cost_join_df["summary_estimated"]).abs() < 1e-12
stage2d_cost_join_df["actual_matches"] = (stage2d_cost_join_df["actual_cost"] - stage2d_cost_join_df["summary_actual"]).abs() < 1e-12

display(stage2d_cost_join_df.head())
display(stage2d_cost_join_df.tail())

with _stage2d_connect_ledger_readonly() as conn:
    stage2d_pending_df = pd.read_sql_query(
        "SELECT * FROM ledger WHERE batch_id = ? AND status = ?",
        conn,
        params=(STAGE2D_BATCH_ID, STATUS_PENDING),
    )

retry_rows = []
stage2c_mean_cost = globals().get("stage2c_total_actual_from_ledger", None)
if stage2c_mean_cost is not None:
    stage2c_mean_per_call = stage2c_mean_cost / 20
else:
    stage2c_mean_per_call = None
for call in STAGE2D_CALLS:
    k = call["position"]
    retry_paths = sorted(STAGE2D_BATCH_DIR.glob(f"attempt_{k:04d}_retry_*_response.txt"))
    if call.get("retry_count", 0) > 0 or retry_paths:
        retry_rows.append({
            "call": k,
            "summary_retry_count": call.get("retry_count", 0),
            "retry_files": [str(p) for p in retry_paths],
            "retry_file_count": len(retry_paths),
            "actual_cost_usd": call.get("actual_cost_usd"),
            "investigate_cost": bool(stage2c_mean_per_call is not None and call.get("actual_cost_usd", 0) > 1.5 * stage2c_mean_per_call and call.get("retry_count", 0) > 0),
        })
retry_df = pd.DataFrame(retry_rows)
display(retry_df if len(retry_df) else pd.DataFrame([{"retry audit": "PASS / N/A", "detail": "No retry files and no retry_count > 0"}]))

ledger_cost_checks = [
    check_row("per-call ledger costs match summary", bool(stage2d_cost_join_df["estimated_matches"].all() and stage2d_cost_join_df["actual_matches"].all()), stage2d_cost_join_df.loc[~(stage2d_cost_join_df["estimated_matches"] & stage2d_cost_join_df["actual_matches"])]),
    check_row("total estimated cost matches summary", abs(float(stage2d_cost_join_df["estimated_cost"].sum()) - STAGE2D_SUMMARY["total_estimated_cost_usd"]) < 1e-12, {"ledger": float(stage2d_cost_join_df["estimated_cost"].sum()), "summary": STAGE2D_SUMMARY["total_estimated_cost_usd"]}),
    check_row("total actual cost matches summary", abs(float(stage2d_cost_join_df["actual_cost"].sum()) - STAGE2D_SUMMARY["total_actual_cost_usd"]) < 1e-12, {"ledger": float(stage2d_cost_join_df["actual_cost"].sum()), "summary": STAGE2D_SUMMARY["total_actual_cost_usd"]}),
    check_row("no pending ledger rows remain", len(stage2d_pending_df) == 0, stage2d_pending_df),
    check_row("actual cost stays under $20 batch cap", STAGE2D_SUMMARY["total_actual_cost_usd"] <= STAGE2D_BATCH_CAP_USD, STAGE2D_SUMMARY["total_actual_cost_usd"]),
    check_row("cumulative Stage 2 spend stays under $30", STAGE2D_SUMMARY["cumulative_monthly_spend_usd"] <= STAGE2D_CUMULATIVE_CAP_USD, STAGE2D_SUMMARY["cumulative_monthly_spend_usd"]),
    check_row("retry files match retry_count", retry_df.empty or bool((retry_df["retry_file_count"] == retry_df["summary_retry_count"]).all()), retry_df),
    check_row("no retry cost investigate flags", retry_df.empty or not bool(retry_df.get("investigate_cost", pd.Series(dtype=bool)).any()), retry_df),
]
display_check_table(ledger_cost_checks, "Stage 2d ledger and retry checks")
D6_STAGE2D_ACCEPTANCE["ledger_cost_reconciliation"] = True


,call,estimated_cost,actual_cost,status,created_at_utc,completed_at_utc,summary_estimated,summary_actual,input_tokens,output_tokens,retry_count,estimated_matches,actual_matches
0,1,0.011475,0.006999,completed,2026-04-17T19:00:35.747Z,2026-04-17T19:00:42.629Z,0.011475,0.006999,1303,206,0,True,True
1,2,0.011898,0.008205,completed,2026-04-17T19:00:42.635Z,2026-04-17T19:00:47.317Z,0.011898,0.008205,1440,259,0,True,True
2,3,0.012477,0.009057,completed,2026-04-17T19:00:47.321Z,2026-04-17T19:00:51.870Z,0.012477,0.009057,1619,280,0,True,True
3,4,0.013104,0.009315,completed,2026-04-17T19:00:51.874Z,2026-04-17T19:00:56.560Z,0.013104,0.009315,1810,259,0,True,True
4,5,0.013221,0.009234,completed,2026-04-17T19:00:56.566Z,2026-04-17T19:01:00.749Z,0.013221,0.009234,1843,247,0,True,True


,call,estimated_cost,actual_cost,status,created_at_utc,completed_at_utc,summary_estimated,summary_actual,input_tokens,output_tokens,retry_count,estimated_matches,actual_matches
195,196,0.013761,0.011109,completed,2026-04-17T19:19:43.189Z,2026-04-17T19:19:48.905Z,0.013761,0.011109,2013,338,0,True,True
196,197,0.013710,0.011241,completed,2026-04-17T19:19:48.922Z,2026-04-17T19:19:54.770Z,0.013710,0.011241,2007,348,0,True,True
197,198,0.013749,0.011262,completed,2026-04-17T19:19:54.785Z,2026-04-17T19:20:00.986Z,0.013749,0.011262,2009,349,0,True,True
198,199,0.013824,0.011640,completed,2026-04-17T19:20:01.003Z,2026-04-17T19:20:07.080Z,0.013824,0.011640,2020,372,0,True,True
199,200,0.013887,0.011763,completed,2026-04-17T19:20:07.094Z,2026-04-17T19:20:13.992Z,0.013887,0.011763,2051,374,0,True,True


,retry audit,detail
0,PASS / N/A,No retry files and no retry_count > 0


Stage 2d ledger and retry checks


,check,status,detail
0,per-call ledger costs match summary,PASS,"Empty DataFrame Columns: [call, estimated_cost..."
1,total estimated cost matches summary,PASS,"{'ledger': 2.756349, 'summary': 2.756348999999..."
2,total actual cost matches summary,PASS,"{'ledger': 2.2433430000000003, 'summary': 2.24..."
3,no pending ledger rows remain,PASS,"Empty DataFrame Columns: [id, batch_id, api_ca..."
4,actual cost stays under $20 batch cap,PASS,2.243343
5,cumulative Stage 2 spend stays under $30,PASS,2.49966
6,retry files match retry_count,PASS,Empty DataFrame Columns: [] Index: []
7,no retry cost investigate flags,PASS,Empty DataFrame Columns: [] Index: []


## AS. Early-Stop Fuse And Crash-Preservation Audit

Stage 2d adds two parse-rate fuses, a theme single-mode failure fuse, and a partial-summary crash preservation contract. This section verifies that none should have fired and that the completed partial summary remains durable.


In [46]:
first5 = stage2d_reclass_df.head(STAGE2D_TIER1_K)
first20 = stage2d_reclass_df.head(STAGE2D_TIER2_K)
first5_valid_rate = float((first5["reclassified_lifecycle"] == PENDING_BACKTEST).mean())
first20_valid_rate = float((first20["reclassified_lifecycle"] == PENDING_BACKTEST).mean())

single_mode_rows = []
for theme in STAGE2D_EXPECTED_THEMES:
    theme_calls = [c for c in attempted_calls if c["theme"] == theme]
    invalid_theme = [c for c in theme_calls if c["lifecycle_state"] != PENDING_BACKTEST]
    signatures = Counter()
    for c in invalid_theme:
        err = c.get("error_info") or {}
        signatures[(err.get("error_category"), err.get("error_signature"))] += 1
    top_sig, top_count = (None, 0)
    if signatures:
        top_sig, top_count = signatures.most_common(1)[0]
    single_mode_rows.append({
        "theme": theme,
        "completed_theme_calls": len(theme_calls),
        "invalid_count": len(invalid_theme),
        "top_error_signature": top_sig,
        "top_error_count": top_count,
        "would_trigger": len(theme_calls) >= STAGE2D_THEME_CALLS_TOTAL and top_count >= STAGE2D_SINGLE_MODE_THRESHOLD,
    })
single_mode_df = pd.DataFrame(single_mode_rows)
display(single_mode_df)

partial_vs_final_rows = [
    {"field": "batch_id", "final": STAGE2D_SUMMARY.get("batch_id"), "partial": STAGE2D_PARTIAL_SUMMARY.get("batch_id"), "match": STAGE2D_SUMMARY.get("batch_id") == STAGE2D_PARTIAL_SUMMARY.get("batch_id")},
    {"field": "batch_status", "final": STAGE2D_SUMMARY.get("batch_status"), "partial": STAGE2D_PARTIAL_SUMMARY.get("batch_status"), "match": STAGE2D_PARTIAL_SUMMARY.get("batch_status") == "completed"},
    {"field": "calls_len", "final": len(STAGE2D_SUMMARY.get("calls", [])), "partial": len(STAGE2D_PARTIAL_SUMMARY.get("calls", [])), "match": len(STAGE2D_SUMMARY.get("calls", [])) == len(STAGE2D_PARTIAL_SUMMARY.get("calls", [])) == STAGE2D_BATCH_SIZE},
    {"field": "last_completed_call", "final": STAGE2D_BATCH_SIZE, "partial": STAGE2D_PARTIAL_SUMMARY.get("last_completed_call"), "match": STAGE2D_PARTIAL_SUMMARY.get("last_completed_call") == STAGE2D_BATCH_SIZE},
]
partial_vs_final_df = pd.DataFrame(partial_vs_final_rows)
display(partial_vs_final_df)

fuse_checks = [
    check_row("tier-1 k=5 valid rate does not trigger fail-fast", first5_valid_rate >= STAGE2D_TIER1_THRESHOLD, first5_valid_rate),
    check_row("tier-2 k=20 valid rate does not trigger mid-batch gate", first20_valid_rate > STAGE2D_TIER2_THRESHOLD, first20_valid_rate),
    check_row("no theme single-mode failure", not bool(single_mode_df["would_trigger"].any()), single_mode_df),
    check_row("summary reports no early stop", STAGE2D_SUMMARY.get("early_stop_reason") is None and STAGE2D_SUMMARY.get("truncated") is False, {"early_stop_reason": STAGE2D_SUMMARY.get("early_stop_reason"), "truncated": STAGE2D_SUMMARY.get("truncated")}),
    check_row("partial summary exists and is not an empty shell", STAGE2D_PARTIAL_SUMMARY_PATH.exists() and len(STAGE2D_PARTIAL_SUMMARY.get("calls", [])) == STAGE2D_BATCH_SIZE, {"exists": STAGE2D_PARTIAL_SUMMARY_PATH.exists(), "calls": len(STAGE2D_PARTIAL_SUMMARY.get("calls", []))}),
    check_row("partial final state matches completed batch", bool(partial_vs_final_df["match"].all()), partial_vs_final_df),
    check_row("batch_duration_seconds is positive", isinstance(STAGE2D_SUMMARY.get("batch_duration_seconds"), (int, float)) and STAGE2D_SUMMARY.get("batch_duration_seconds") > 0, STAGE2D_SUMMARY.get("batch_duration_seconds")),
]
display_check_table(fuse_checks, "Stage 2d fuse and partial-summary checks")
D6_STAGE2D_ACCEPTANCE["fuses_and_partial_summary"] = True


,theme,completed_theme_calls,invalid_count,top_error_signature,top_error_count,would_trigger
0,momentum,40,1,"(complexity_rejection, over_complex)",1,False
1,mean_reversion,40,0,None,0,False
2,volatility_regime,40,0,None,0,False
3,volume_divergence,40,0,None,0,False
4,calendar_effect,40,0,None,0,False


,field,final,partial,match
0,batch_id,5cf76668-47d1-48d7-bd90-db06d31982ed,5cf76668-47d1-48d7-bd90-db06d31982ed,True
1,batch_status,completed,completed,True
2,calls_len,200,200,True
3,last_completed_call,200,200,True


Stage 2d fuse and partial-summary checks


,check,status,detail
0,tier-1 k=5 valid rate does not trigger fail-fast,PASS,1.0
1,tier-2 k=20 valid rate does not trigger mid-ba...,PASS,1.0
2,no theme single-mode failure,PASS,theme completed_theme_calls i...
3,summary reports no early stop,PASS,"{'early_stop_reason': None, 'truncated': False}"
4,partial summary exists and is not an empty shell,PASS,"{'exists': True, 'calls': 200}"
5,partial final state matches completed batch,PASS,field ...
6,batch_duration_seconds is positive,PASS,1178.3


## AT. Special-Case Branches Audit

This section audits the paths that can hide local issues inside a clean aggregate: repeated factor sets, duplicate hashes, empty factor sets, retries, and error taxonomy continuity.


In [47]:
# Repeated factor sets with occurrence_count >= 3
repeated_deep_rows = []
for rfs in STAGE2D_SUMMARY.get("repeated_factor_sets", []):
    if rfs.get("occurrence_count", 0) < 3:
        continue
    expected_fs = tuple(rfs["factor_set"])
    for pos in rfs["occurring_at_calls"]:
        call = STAGE2D_CALL_BY_POSITION[pos]
        recomputed_fs = tuple(stage2d_reclass_df.loc[stage2d_reclass_df["call"] == pos, "factors"].iloc[0])
        prompt_examples = stage2d_prompt_examples_by_call[pos]
        repeated_deep_rows.append({
            "factor_set": list(expected_fs),
            "call": pos,
            "theme_summary": call["theme"],
            "theme_expected": STAGE2D_EXPECTED_THEMES[(pos - 1) % STAGE2D_THEME_CYCLE_LEN],
            "factor_set_matches": recomputed_fs == expected_fs,
            "approved_example_names_before_call": call["approved_example_names_before_call"],
            "prompt_example_names": [ex.get("name") for ex in prompt_examples],
            "hash": call.get("hypothesis_hash"),
        })
repeated_deep_df = pd.DataFrame(repeated_deep_rows)
display(repeated_deep_df.head(20) if len(repeated_deep_df) else pd.DataFrame([{"repeated_factor_sets": "PASS / N/A", "detail": "No occurrence_count >= 3 sets"}]))

# Hash duplicate forensic
valid_hash_series = stage2d_reclass_df.loc[stage2d_reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST, "reclassified_hash"].dropna()
dup_hashes = valid_hash_series[valid_hash_series.duplicated(keep=False)].tolist()
duplicate_hash_df = stage2d_reclass_df[stage2d_reclass_df["reclassified_hash"].isin(dup_hashes)] if dup_hashes else pd.DataFrame()
display(duplicate_hash_df if len(duplicate_hash_df) else pd.DataFrame([{"duplicate_hash_cases": "PASS / N/A", "detail": "No duplicate hashes"}]))

# Empty factor set forensic
empty_factor_df = stage2d_reclass_df[(stage2d_reclass_df["reclassified_lifecycle"] == PENDING_BACKTEST) & (stage2d_reclass_df["factors"].map(len) == 0)]
display(empty_factor_df if len(empty_factor_df) else pd.DataFrame([{"empty_factor_set_cases": "PASS / N/A", "detail": "No valid empty factor sets"}]))

# Error taxonomy continuity
error_rows = []
for call in STAGE2D_CALLS:
    err = call.get("error_info")
    if err:
        error_rows.append({
            "call": call["position"],
            "lifecycle": call["lifecycle_state"],
            "error_category": err.get("error_category"),
            "error_signature": err.get("error_signature"),
            "parse_error_prefix": err.get("parse_error_prefix", "")[:120],
        })
error_taxonomy_df = pd.DataFrame(error_rows)
display(error_taxonomy_df if len(error_taxonomy_df) else pd.DataFrame([{"error taxonomy": "PASS / N/A", "detail": "No invalid/error calls"}]))

special_checks = [
    check_row("all occurrence_count>=3 repeated factor sets match byte-level factor lists", repeated_deep_df.empty or bool(repeated_deep_df["factor_set_matches"].all()), repeated_deep_df.loc[~repeated_deep_df.get("factor_set_matches", pd.Series(dtype=bool))] if not repeated_deep_df.empty else "N/A"),
    check_row("repeated factor set themes match rotation", repeated_deep_df.empty or bool((repeated_deep_df["theme_summary"] == repeated_deep_df["theme_expected"]).all()), repeated_deep_df if not repeated_deep_df.empty else "N/A"),
    check_row("duplicate hash cases are absent or displayed", len(dup_hashes) == 0 or not duplicate_hash_df.empty, duplicate_hash_df if len(dup_hashes) else "PASS / N/A"),
    check_row("valid empty factor-set cases are absent or displayed", len(empty_factor_df) == STAGE2D_SUMMARY.get("valid_with_empty_factor_set_count", 0), empty_factor_df),
    check_row("summary empty factor-set count matches forensic count", len(empty_factor_df) == STAGE2D_SUMMARY.get("valid_with_empty_factor_set_count", 0), {"forensic": len(empty_factor_df), "summary": STAGE2D_SUMMARY.get("valid_with_empty_factor_set_count", 0)}),
    check_row("retry branch is PASS / N/A for this batch", int(stage2d_cost_join_df["retry_count"].sum()) == 0, int(stage2d_cost_join_df["retry_count"].sum())),
    check_row("error taxonomy names are stable for observed errors", set(error_taxonomy_df.get("error_category", pd.Series(dtype=str)).dropna()).issubset({"complexity_rejection", "schema_validation", "json_decode", "backend_empty", "duplicate_condition", "other"}), error_taxonomy_df),
]
display_check_table(special_checks, "Stage 2d special-case branch checks")
D6_STAGE2D_ACCEPTANCE["special_cases_audited"] = True


,factor_set,call,theme_summary,theme_expected,factor_set_matches,approved_example_names_before_call,prompt_example_names,hash
0,"[bb_upper_24_2, close, realized_vol_24h, retur...",17,mean_reversion,mean_reversion,True,"[macd_momentum_surge, monday_close_long, volum...","[macd_momentum_surge, monday_close_long, volum...",17396a8e291dae75
1,"[bb_upper_24_2, close, realized_vol_24h, retur...",27,mean_reversion,mean_reversion,True,"[momentum_exhaustion_rsi_divergence, monday_di...","[momentum_exhaustion_rsi_divergence, monday_di...",57f884d5e47b14e6
2,"[bb_upper_24_2, close, realized_vol_24h, retur...",102,mean_reversion,mean_reversion,True,"[macd_histogram_surge_momentum_confirm, monday...","[macd_histogram_surge_momentum_confirm, monday...",611638897f1796b9
3,"[bb_upper_24_2, close, realized_vol_24h, retur...",112,mean_reversion,mean_reversion,True,"[rsi_oversold_momentum_reversal, monday_open_g...","[rsi_oversold_momentum_reversal, monday_open_g...",293d6f4a5553178b
4,"[bb_upper_24_2, close, realized_vol_24h, retur...",147,mean_reversion,mean_reversion,True,"[macd_momentum_thrust_146, weekend_volatility_...","[macd_momentum_thrust_146, weekend_volatility_...",96a3cfb044e8db33
5,"[bb_upper_24_2, close, realized_vol_24h, retur...",152,mean_reversion,mean_reversion,True,"[macd_momentum_surge_151, monday_oversold_reve...","[macd_momentum_surge_151, monday_oversold_reve...",e9cdc1cb41f6d223
6,"[bb_upper_24_2, close, realized_vol_24h, retur...",162,mean_reversion,mean_reversion,True,"[momentum_continuation_161, monday_mean_revers...","[momentum_continuation_161, monday_mean_revers...",0bf3523a661bc85b
7,"[bb_upper_24_2, close, realized_vol_24h, retur...",197,mean_reversion,mean_reversion,True,"[macd_momentum_surge_196, monday_dip_buying_19...","[macd_momentum_surge_196, monday_dip_buying_19...",83c8fc0c57b3e694
8,"[bb_upper_24_2, close, realized_vol_24h, retur...",72,mean_reversion,mean_reversion,True,"[momentum_acceleration_rsi_surge, monday_low_v...","[momentum_acceleration_rsi_surge, monday_low_v...",ca08df07cc55d138
9,"[bb_upper_24_2, close, realized_vol_24h, retur...",87,mean_reversion,mean_reversion,True,"[macd_momentum_surge_with_vol_confirm, weekend...","[macd_momentum_surge_with_vol_confirm, weekend...",3985d6832addc018


,duplicate_hash_cases,detail
0,PASS / N/A,No duplicate hashes


,empty_factor_set_cases,detail
0,PASS / N/A,No valid empty factor sets


,call,lifecycle,error_category,error_signature,parse_error_prefix
0,116,rejected_complexity,complexity_rejection,over_complex,1 validation error for StrategyDSL\ndescriptio...


Stage 2d special-case branch checks


,check,status,detail
0,all occurrence_count>=3 repeated factor sets m...,PASS,"Empty DataFrame Columns: [factor_set, call, th..."
1,repeated factor set themes match rotation,PASS,fac...
2,duplicate hash cases are absent or displayed,PASS,PASS / N/A
3,valid empty factor-set cases are absent or dis...,PASS,"Empty DataFrame Columns: [call, theme, schema_..."
4,summary empty factor-set count matches forensi...,PASS,"{'forensic': 0, 'summary': 0}"
5,retry branch is PASS / N/A for this batch,PASS,0
6,error taxonomy names are stable for observed e...,PASS,call lifecycle error_cate...


## AU. Mode-Collapse Review Inputs And Final Verdict

This section is not D7 interpretation. It prepares reviewer-grade inputs for D7: diversity over time, repeated factor-set concentration, and theme coherence/fallback metrics.


In [48]:
# Diversity trend from interim snapshots
interim_df = pd.DataFrame(STAGE2D_SUMMARY["interim_snapshots"])
display(interim_df[["at_call", "cumulative_valid_count", "cumulative_distinct_hash_count", "cumulative_distinct_factor_set_count", "cumulative_unique_factor_set_ratio", "cumulative_actual_cost_usd"]])

# Repeated factor-set concentration curve
repeat_counts = [r["occurrence_count"] for r in STAGE2D_SUMMARY.get("repeated_factor_sets", [])]
total_valid = STAGE2D_SUMMARY["total_valid_count"]
concentration_rows = []
for top_n in [1, 3, 5, 10]:
    covered = sum(repeat_counts[:top_n]) if repeat_counts else 0
    concentration_rows.append({
        "top_repeated_sets": top_n,
        "covered_occurrences": covered,
        "coverage_of_valid_calls": covered / total_valid if total_valid else None,
    })
concentration_df = pd.DataFrame(concentration_rows)
display(concentration_df)

# Theme coherence/fallback summary
coherence_df = per_theme_recomputed_df[[
    "theme", "n_calls", "valid_count", "avg_overlap_ratio", "avg_out_of_theme_factors", "contains_momentum_default_count", "dominant_factors_top3",
]].copy()
coherence_df["default_momentum_usage_ratio"] = coherence_df["contains_momentum_default_count"] / coherence_df["valid_count"]
display(coherence_df)

gates = [
    ("Batch artifacts complete", D6_STAGE2D_ACCEPTANCE.get("batch_artifacts_complete", False)),
    ("Config block correct", D6_STAGE2D_ACCEPTANCE.get("config_block_correct", False)),
    ("Strict sequential execution", D6_STAGE2D_ACCEPTANCE.get("strict_sequential_execution", False)),
    ("Prompt safety preserved", D6_STAGE2D_ACCEPTANCE.get("prompt_safety_preserved", False)),
    ("Sliding-window identity correctness", D6_STAGE2D_ACCEPTANCE.get("sliding_window_identity", False)),
    ("Independent response reclassification consistent", D6_STAGE2D_ACCEPTANCE.get("response_reclassification_consistent", False)),
    ("Aggregate recomputation matches summary", D6_STAGE2D_ACCEPTANCE.get("aggregate_recomputation_matches", False)),
    ("Ledger and cost reconciliation clean", D6_STAGE2D_ACCEPTANCE.get("ledger_cost_reconciliation", False)),
    ("Early-stop fuses + partial summary honored", D6_STAGE2D_ACCEPTANCE.get("fuses_and_partial_summary", False)),
    ("Special-case branches audited", D6_STAGE2D_ACCEPTANCE.get("special_cases_audited", False)),
]
stage2d_final_df = pd.DataFrame([{"gate": gate, "pass": ok, "status": "PASS" if ok else "FAIL"} for gate, ok in gates])
display(stage2d_final_df[["gate", "status"]])

assert stage2d_final_df["pass"].all(), stage2d_final_df[~stage2d_final_df["pass"]]

print("D6 Stage 2d production-shape acceptance audit: PASS")
print("All 10 Stage 2d notebook gates passed. D6 can be signed off subject to reviewer interpretation of mode-collapse/theme-coherence signals for D7.")
print("Observed anomaly flags are reviewer inputs, not gate failures:", STAGE2D_SUMMARY.get("anomaly_flags"))


,at_call,cumulative_valid_count,cumulative_distinct_hash_count,cumulative_distinct_factor_set_count,cumulative_unique_factor_set_ratio,cumulative_actual_cost_usd
0,50,50,50,41,0.8200,0.554079
1,100,100,100,78,0.7800,1.121916
2,150,149,149,109,0.7315,1.684647
3,200,199,199,133,0.6683,2.243343


,top_repeated_sets,covered_occurrences,coverage_of_valid_calls
0,1,8,0.040201
1,3,19,0.095477
2,5,29,0.145729
3,10,50,0.251256


,theme,n_calls,valid_count,avg_overlap_ratio,avg_out_of_theme_factors,contains_momentum_default_count,dominant_factors_top3,default_momentum_usage_ratio
0,momentum,40,39,0.662943,1.820513,39,"[macd_hist, rsi_14, return_24h]",1.0
1,mean_reversion,40,40,0.481071,3.200000,40,"[close, bb_upper_24_2, rsi_14]",1.0
2,volatility_regime,40,40,0.194821,4.825000,40,"[realized_vol_24h, close, return_24h]",1.0
3,volume_divergence,40,40,0.192440,4.425000,40,"[volume_zscore_24h, rsi_14, return_24h]",1.0
4,calendar_effect,40,40,0.208780,4.500000,40,"[day_of_week, rsi_14, return_24h]",1.0


,gate,status
0,Batch artifacts complete,PASS
1,Config block correct,PASS
2,Strict sequential execution,PASS
3,Prompt safety preserved,PASS
4,Sliding-window identity correctness,PASS
5,Independent response reclassification consistent,PASS
6,Aggregate recomputation matches summary,PASS
7,Ledger and cost reconciliation clean,PASS
8,Early-stop fuses + partial summary honored,PASS
9,Special-case branches audited,PASS


D6 Stage 2d production-shape acceptance audit: PASS
All 10 Stage 2d notebook gates passed. D6 can be signed off subject to reviewer interpretation of mode-collapse/theme-coherence signals for D7.
Observed anomaly flags are reviewer inputs, not gate failures: [{'kind': 'rsi_14_dominance', 'scope': 'first_50_valid_calls', 'count': 46, 'total': 50, 'ratio': 0.92, 'threshold': 0.8}]


## AV. Stage 2d Reviewer Probes 1-6

These probes validate the reviewer observations that came out of Stage 2d. They are not additional pass/fail gates for production-shape execution. They are reviewer-grade evidence for D7 Critic design.

The six probes are:

1. Production-shape contract stability: cost ratio, parse rate, and the single non-valid lifecycle state.
2. `rsi_14` dominance: Stage 2d first-50 valid calls versus Stage 2c's smaller N=20 observation.
3. Factor-set diversity trajectory: monotonic decline and delta shape across interim snapshots.
4. Repeated factor-set forensic: true duplicate collapse versus distinct-hash parameter exploration.
5. Thin-theme momentum bleed: theme overlap, out-of-theme factors, and default-momentum usage.
6. Theme-coherence scoring need: calendar overlap decline and momentum adherence increase from 2c to 2d.


In [49]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# Load Stage 2c receipt for comparison. Stage 2d variables come from AJ-AU;
# if this cell is run after a restart without AJ-AU, load the final Stage 2d summary too.
if "STAGE2D_SUMMARY" not in globals():
    STAGE2D_SUMMARY_PATH = sorted(Path("raw_payloads").glob("batch_*/stage2d_summary.json"), key=lambda path: path.stat().st_mtime)[-1]
    STAGE2D_BATCH_DIR = STAGE2D_SUMMARY_PATH.parent
    STAGE2D_BATCH_ID = STAGE2D_BATCH_DIR.name.removeprefix("batch_")
    STAGE2D_SUMMARY = json.loads(STAGE2D_SUMMARY_PATH.read_text(encoding="utf-8"))
    STAGE2D_CALLS = sorted(STAGE2D_SUMMARY["calls"], key=lambda row: row["position"])
if "STAGE2D_EXPECTED_THEMES" not in globals():
    STAGE2D_EXPECTED_THEMES = ["momentum", "mean_reversion", "volatility_regime", "volume_divergence", "calendar_effect"]
if "check_row" not in globals():
    def status(ok: bool) -> str:
        return "PASS" if bool(ok) else "FAIL"
    def check_row(check: str, ok: bool, detail="") -> dict:
        return {"check": check, "status": status(ok), "detail": detail}
if "display_check_table" not in globals():
    def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
        print(title)
        df = pd.DataFrame(rows)
        display(df)
        if require_all:
            failed = df[df["status"] != "PASS"]
            assert failed.empty, failed
        return df

STAGE2C_SUMMARY_PATH_FOR_PROBES = sorted(
    Path("raw_payloads").glob("batch_*/stage2c_summary.json"),
    key=lambda path: path.stat().st_mtime,
)[-1]
STAGE2C_SUMMARY_FOR_PROBES = json.loads(STAGE2C_SUMMARY_PATH_FOR_PROBES.read_text(encoding="utf-8"))

probe_receipt_df = pd.DataFrame([
    {"stage": "2c", "batch_id": STAGE2C_SUMMARY_FOR_PROBES["batch_id"], "summary": str(STAGE2C_SUMMARY_PATH_FOR_PROBES), "calls": len(STAGE2C_SUMMARY_FOR_PROBES["calls"])},
    {"stage": "2d", "batch_id": STAGE2D_SUMMARY["batch_id"], "summary": str(STAGE2D_SUMMARY_PATH), "calls": len(STAGE2D_CALLS)},
])
display(probe_receipt_df)

D6_STAGE2D_PROBE_ACCEPTANCE = {}


,stage,batch_id,summary,calls
0,2c,e07f34a2-b532-4f35-a9f3-af97a5a96f1f,raw_payloads/batch_e07f34a2-b532-4f35-a9f3-af9...,20
1,2d,5cf76668-47d1-48d7-bd90-db06d31982ed,raw_payloads/batch_5cf76668-47d1-48d7-bd90-db0...,200


### Probe 1 — Contract Stability Under 200 Calls

This probe verifies the headline claim: Stage 2d stayed stable under production-shape load. The single non-valid output must be a clean complexity rejection, not an infrastructure, leakage, retry, or parsing failure.


In [50]:
stage2d_total_est = STAGE2D_SUMMARY["total_estimated_cost_usd"]
stage2d_total_actual = STAGE2D_SUMMARY["total_actual_cost_usd"]
stage2d_cost_ratio = stage2d_total_est / stage2d_total_actual
stage2d_non_valid_calls = [c for c in STAGE2D_CALLS if c["lifecycle_state"] != "pending_backtest"]
non_valid_error = (stage2d_non_valid_calls[0].get("error_info") or {}) if stage2d_non_valid_calls else {}
non_valid_parse_prefix = non_valid_error.get("parse_error_prefix", "")
if "description" in non_valid_parse_prefix and "at most 300 characters" in non_valid_parse_prefix:
    non_valid_complexity_gate = "description length > 300 characters"
elif "max_hold_bars" in non_valid_parse_prefix:
    non_valid_complexity_gate = "max_hold_bars complexity budget"
elif "entry" in non_valid_parse_prefix or "exit" in non_valid_parse_prefix or "conditions" in non_valid_parse_prefix:
    non_valid_complexity_gate = "entry/exit group or condition-count complexity budget"
else:
    non_valid_complexity_gate = "unclassified complexity gate; inspect parse_error_prefix"

contract_probe_df = pd.DataFrame([
    {"metric": "cost_ratio", "actual": stage2d_cost_ratio, "expected / interpretation": "~1.23x and improved estimator stability"},
    {"metric": "parse_rate", "actual": STAGE2D_SUMMARY["parse_rate"], "expected / interpretation": "199/200 = 99.5%"},
    {"metric": "lifecycle_counts", "actual": STAGE2D_SUMMARY["lifecycle_counts"], "expected / interpretation": "199 pending_backtest, 1 rejected_complexity"},
    {"metric": "non_valid_calls", "actual": stage2d_non_valid_calls, "expected / interpretation": "single clean complexity rejection"},
    {"metric": "non_valid_complexity_gate", "actual": non_valid_complexity_gate, "expected / interpretation": "exact reason for the only non-valid output"},
])
display(contract_probe_df)

non_valid_ok = (
    len(stage2d_non_valid_calls) == 1
    and stage2d_non_valid_calls[0]["lifecycle_state"] == "rejected_complexity"
    and (stage2d_non_valid_calls[0].get("error_info") or {}).get("error_category") == "complexity_rejection"
)
contract_checks = [
    check_row("cost ratio recomputes to ~1.23x", abs(stage2d_cost_ratio - STAGE2D_SUMMARY["cost_ratio"]) < 1e-12 and 1.20 <= stage2d_cost_ratio <= 1.25, stage2d_cost_ratio),
    check_row("parse rate is 99.5%", STAGE2D_SUMMARY["parse_rate"] == 0.995, STAGE2D_SUMMARY["parse_rate"]),
    check_row("only non-valid call is clean complexity_rejection", non_valid_ok, stage2d_non_valid_calls),
    check_row("only non-valid call is description length over 300 chars", non_valid_complexity_gate == "description length > 300 characters", {"gate": non_valid_complexity_gate, "parse_error_prefix": non_valid_parse_prefix}),
    check_row("no retries or cardinality violations", sum(c.get("retry_count", 0) for c in STAGE2D_CALLS) == 0 and STAGE2D_SUMMARY["cardinality_violation_count"] == 0, {"retries": sum(c.get("retry_count", 0) for c in STAGE2D_CALLS), "cardinality_violations": STAGE2D_SUMMARY["cardinality_violation_count"]}),
]
display_check_table(contract_checks, "Probe 1 — Stage 2d contract stability checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_1_contract_stability"] = True


,metric,actual,expected / interpretation
0,cost_ratio,1.228679,~1.23x and improved estimator stability
1,parse_rate,0.995,199/200 = 99.5%
2,lifecycle_counts,"{'pending_backtest': 199, 'rejected_complexity...","199 pending_backtest, 1 rejected_complexity"
3,non_valid_calls,"[{'position': 116, 'theme': 'momentum', 'trunc...",single clean complexity rejection
4,non_valid_complexity_gate,description length > 300 characters,exact reason for the only non-valid output


Probe 1 — Stage 2d contract stability checks


,check,status,detail
0,cost ratio recomputes to ~1.23x,PASS,1.228679
1,parse rate is 99.5%,PASS,0.995
2,only non-valid call is clean complexity_rejection,PASS,"[{'position': 116, 'theme': 'momentum', 'trunc..."
3,only non-valid call is description length over...,PASS,{'gate': 'description length > 300 characters'...
4,no retries or cardinality violations,PASS,"{'retries': 0, 'cardinality_violations': 0}"


### Probe 2 — `rsi_14` Dominance Anomaly

This probe verifies that the Stage 2d anomaly flag is real and that Stage 2c's smaller N=20 `rsi_14` rate was an underpowered observation, not the stable prior-bias estimate.


In [51]:
stage2c_calls = STAGE2C_SUMMARY_FOR_PROBES["calls"]
stage2c_valid = [c for c in stage2c_calls if c["lifecycle_state"] == "pending_backtest"]
stage2d_first50_valid = [c for c in STAGE2D_CALLS[:50] if c["lifecycle_state"] == "pending_backtest"]
stage2c_rsi_count = sum(1 for c in stage2c_valid if "rsi_14" in c.get("factors_used", []))
stage2d_first50_rsi_count = sum(1 for c in stage2d_first50_valid if "rsi_14" in c.get("factors_used", []))
stage2c_rsi_ratio = stage2c_rsi_count / len(stage2c_valid)
stage2d_first50_rsi_ratio = stage2d_first50_rsi_count / len(stage2d_first50_valid)
rsi_anomaly = next((flag for flag in STAGE2D_SUMMARY.get("anomaly_flags", []) if flag.get("kind") == "rsi_14_dominance"), None)

rsi_probe_df = pd.DataFrame([
    {"stage": "2c", "scope": "all valid calls", "rsi_count": stage2c_rsi_count, "valid_total": len(stage2c_valid), "ratio": stage2c_rsi_ratio},
    {"stage": "2d", "scope": "first 50 valid calls", "rsi_count": stage2d_first50_rsi_count, "valid_total": len(stage2d_first50_valid), "ratio": stage2d_first50_rsi_ratio},
])
display(rsi_probe_df)
display(pd.DataFrame([rsi_anomaly]) if rsi_anomaly else pd.DataFrame([{"anomaly": "missing"}]))

rsi_checks = [
    check_row("Stage 2c rsi_14 usage is 12/20 = 60%", stage2c_rsi_count == 12 and len(stage2c_valid) == 20 and abs(stage2c_rsi_ratio - 0.60) < 1e-12, rsi_probe_df),
    check_row("Stage 2d first-50 rsi_14 usage is 46/50 = 92%", stage2d_first50_rsi_count == 46 and len(stage2d_first50_valid) == 50 and abs(stage2d_first50_rsi_ratio - 0.92) < 1e-12, rsi_probe_df),
    check_row("rsi_14_dominance anomaly flag is present and matches recomputation", rsi_anomaly is not None and rsi_anomaly["count"] == stage2d_first50_rsi_count and rsi_anomaly["total"] == len(stage2d_first50_valid) and abs(rsi_anomaly["ratio"] - stage2d_first50_rsi_ratio) < 1e-12, rsi_anomaly),
]
display_check_table(rsi_checks, "Probe 2 — rsi_14 dominance checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_2_rsi_dominance"] = True


,stage,scope,rsi_count,valid_total,ratio
0,2c,all valid calls,12,20,0.60
1,2d,first 50 valid calls,46,50,0.92


,kind,scope,count,total,ratio,threshold
0,rsi_14_dominance,first_50_valid_calls,46,50,0.92,0.8


Probe 2 — rsi_14 dominance checks


,check,status,detail
0,Stage 2c rsi_14 usage is 12/20 = 60%,PASS,stage scope rsi_count vali...
1,Stage 2d first-50 rsi_14 usage is 46/50 = 92%,PASS,stage scope rsi_count vali...
2,rsi_14_dominance anomaly flag is present and m...,PASS,"{'kind': 'rsi_14_dominance', 'scope': 'first_5..."


### Probe 3 — Unique Factor-Set Ratio Trajectory

This probe verifies the monotonic decline `0.82 → 0.78 → 0.7315 → 0.6683` and the slight acceleration in decline magnitude across Stage 2d snapshots.


In [52]:
snapshot_rows = []
prev_ratio = None
prev_drop = None
for snap in STAGE2D_SUMMARY["interim_snapshots"]:
    ratio = snap["cumulative_unique_factor_set_ratio"]
    drop = None if prev_ratio is None else prev_ratio - ratio
    acceleration = None if prev_drop is None or drop is None else drop - prev_drop
    snapshot_rows.append({
        "at_call": snap["at_call"],
        "unique_factor_set_ratio": ratio,
        "drop_from_previous": drop,
        "drop_acceleration": acceleration,
        "distinct_factor_sets": snap["cumulative_distinct_factor_set_count"],
        "valid_count": snap["cumulative_valid_count"],
    })
    prev_ratio = ratio
    if drop is not None:
        prev_drop = drop
snapshot_probe_df = pd.DataFrame(snapshot_rows)
display(snapshot_probe_df)

ratios = snapshot_probe_df["unique_factor_set_ratio"].tolist()
drops = [x for x in snapshot_probe_df["drop_from_previous"].tolist() if x is not None and not pd.isna(x)]
diversity_checks = [
    check_row("unique factor-set ratios match expected path", ratios == [0.82, 0.78, 0.7315, 0.6683], ratios),
    check_row("unique factor-set ratio is monotonically decreasing", all(a >= b for a, b in zip(ratios, ratios[1:])), ratios),
    check_row("decline magnitude lightly accelerates", drops == sorted(drops), drops),
    check_row("final ratio is 0.6683", STAGE2D_SUMMARY["unique_factor_set_ratio"] == 0.6683, STAGE2D_SUMMARY["unique_factor_set_ratio"]),
]
display_check_table(diversity_checks, "Probe 3 — factor-set diversity trajectory checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_3_unique_fs_trajectory"] = True


,at_call,unique_factor_set_ratio,drop_from_previous,drop_acceleration,distinct_factor_sets,valid_count
0,50,0.8200,NaN,NaN,41,50
1,100,0.7800,0.0400,NaN,78,100
2,150,0.7315,0.0485,0.0085,109,149
3,200,0.6683,0.0632,0.0147,133,199


Probe 3 — factor-set diversity trajectory checks


,check,status,detail
0,unique factor-set ratios match expected path,PASS,"[0.82, 0.78, 0.7315, 0.6683]"
1,unique factor-set ratio is monotonically decre...,PASS,"[0.82, 0.78, 0.7315, 0.6683]"
2,decline magnitude lightly accelerates,PASS,"[0.039999999999999925, 0.04849999999999999, 0...."
3,final ratio is 0.6683,PASS,0.6683


### Probe 4 — Repeated Factor Sets: Collapse Or Parameter Exploration?

This probe checks repeated factor sets through `distinct_hashes_within_factor_set`. If repeated factor sets mostly have distinct hashes equal to occurrence count, the evidence points to parameter/condition exploration within the same factor menu rather than exact duplicate collapse.


In [53]:
repeated_rows = []
for row in STAGE2D_SUMMARY.get("repeated_factor_sets", []):
    repeated_rows.append({
        "factor_set": row["factor_set"],
        "occurrence_count": row["occurrence_count"],
        "distinct_hashes_within_factor_set": row["distinct_hashes_within_factor_set"],
        "calls": row["occurring_at_calls"],
        "themes": row["themes_used_in"],
        "interpretation": "parameter/threshold exploration" if row["distinct_hashes_within_factor_set"] == row["occurrence_count"] else "possible exact duplicate collapse",
    })
repeated_probe_df = pd.DataFrame(repeated_rows)
display(repeated_probe_df.head(15))

repeat_concentration_rows = []
for top_n in [1, 3, 5]:
    covered = int(repeated_probe_df["occurrence_count"].head(top_n).sum()) if len(repeated_probe_df) else 0
    repeat_concentration_rows.append({
        "top_repeated_factor_sets": top_n,
        "covered_valid_call_occurrences": covered,
        "coverage_of_valid_calls": covered / STAGE2D_SUMMARY["total_valid_count"] if STAGE2D_SUMMARY["total_valid_count"] else None,
    })
repeat_concentration_df = pd.DataFrame(repeat_concentration_rows)
display(repeat_concentration_df)

high_repeated_df = repeated_probe_df[repeated_probe_df["occurrence_count"] >= 3]
exact_duplicate_like_df = repeated_probe_df[repeated_probe_df["distinct_hashes_within_factor_set"] < repeated_probe_df["occurrence_count"]]
repeat_checks = [
    check_row("repeated factor sets exist and are visible for D7", len(repeated_probe_df) == len(STAGE2D_SUMMARY.get("repeated_factor_sets", [])) and len(repeated_probe_df) > 0, len(repeated_probe_df)),
    check_row("all repeated factor sets have distinct hashes per occurrence", exact_duplicate_like_df.empty, exact_duplicate_like_df),
    check_row("occurrence_count>=3 sets are explicitly surfaced", len(high_repeated_df) >= 1, high_repeated_df[["occurrence_count", "distinct_hashes_within_factor_set", "calls", "interpretation"]].head(10)),
    check_row("no duplicate hypothesis hashes in Stage 2d valid calls", STAGE2D_SUMMARY["distinct_hash_count"] == STAGE2D_SUMMARY["total_valid_count"], {"distinct_hash_count": STAGE2D_SUMMARY["distinct_hash_count"], "total_valid_count": STAGE2D_SUMMARY["total_valid_count"]}),
    check_row("top-1/top-3/top-5 repeated factor-set concentration is visible", repeat_concentration_df["covered_valid_call_occurrences"].tolist() == [8, 19, 29], repeat_concentration_df),
]
display_check_table(repeat_checks, "Probe 4 — repeated factor-set forensic checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_4_repeated_factor_sets"] = True


,factor_set,occurrence_count,distinct_hashes_within_factor_set,calls,themes,interpretation
0,"[bb_upper_24_2, close, realized_vol_24h, retur...",8,8,"[17, 27, 102, 112, 147, 152, 162, 197]","[mean_reversion, mean_reversion, mean_reversio...",parameter/threshold exploration
1,"[bb_upper_24_2, close, realized_vol_24h, retur...",6,6,"[72, 87, 92, 142, 157, 192]","[mean_reversion, mean_reversion, mean_reversio...",parameter/threshold exploration
2,"[bb_upper_24_2, close, return_24h, rsi_14, sma...",5,5,"[32, 82, 97, 122, 127]","[mean_reversion, mean_reversion, mean_reversio...",parameter/threshold exploration
3,"[macd_hist, return_24h, rsi_14, volume_zscore_...",5,5,"[36, 41, 44, 49, 131]","[momentum, momentum, volume_divergence, volume...",parameter/threshold exploration
4,"[close, return_24h, rsi_14, sma_50, volume_zsc...",5,5,"[39, 149, 154, 159, 169]","[volume_divergence, volume_divergence, volume_...",parameter/threshold exploration
5,"[close, ema_26, macd_hist, return_24h, rsi_14]",5,5,"[51, 156, 171, 176, 186]","[momentum, momentum, momentum, momentum, momen...",parameter/threshold exploration
6,"[close, return_1h, return_24h, rsi_14, sma_50,...",4,4,"[9, 84, 94, 109]","[volume_divergence, volume_divergence, volume_...",parameter/threshold exploration
7,"[close, day_of_week, return_24h, rsi_14, sma_20]",4,4,"[35, 80, 185, 195]","[calendar_effect, calendar_effect, calendar_ef...",parameter/threshold exploration
8,"[bb_upper_24_2, close, return_24h, rsi_14, zsc...",4,4,"[37, 42, 47, 137]","[mean_reversion, mean_reversion, mean_reversio...",parameter/threshold exploration
9,"[close, return_1h, return_24h, rsi_14, sma_20,...",4,4,"[64, 69, 89, 199]","[volume_divergence, volume_divergence, volume_...",parameter/threshold exploration


,top_repeated_factor_sets,covered_valid_call_occurrences,coverage_of_valid_calls
0,1,8,0.040201
1,3,19,0.095477
2,5,29,0.145729


Probe 4 — repeated factor-set forensic checks


,check,status,detail
0,repeated factor sets exist and are visible for D7,PASS,32
1,all repeated factor sets have distinct hashes ...,PASS,"Empty DataFrame Columns: [factor_set, occurren..."
2,occurrence_count>=3 sets are explicitly surfaced,PASS,occurrence_count distinct_hashes_within_fa...
3,no duplicate hypothesis hashes in Stage 2d val...,PASS,"{'distinct_hash_count': 199, 'total_valid_coun..."
4,top-1/top-3/top-5 repeated factor-set concentr...,PASS,top_repeated_factor_sets covered_valid_cal...


### Probe 5 — Thin-Theme Momentum Bleed Confirmation

This probe compares Stage 2c and Stage 2d per-theme aggregates. It focuses on thin themes (`volatility_regime`, `volume_divergence`, `calendar_effect`) where the prompt has fewer native factors and where default momentum factors can dominate the model's fallback behavior.


In [54]:
def _theme_df(summary: dict, stage: str) -> pd.DataFrame:
    rows = []
    for row in summary["per_theme"]:
        valid = row["valid_count"] or 1
        rows.append({
            "stage": stage,
            "theme": row["theme"],
            "valid_count": row["valid_count"],
            "avg_overlap_count": row["avg_overlap_count"],
            "avg_overlap_ratio": row["avg_overlap_ratio"],
            "avg_out_of_theme_factors": row["avg_out_of_theme_factors"],
            "contains_momentum_default_count": row["contains_momentum_default_count"],
            "default_momentum_usage_ratio": row["contains_momentum_default_count"] / valid,
            "dominant_factors_top3": row["dominant_factors_top3"],
        })
    return pd.DataFrame(rows)

theme_2c_df = _theme_df(STAGE2C_SUMMARY_FOR_PROBES, "2c")
theme_2d_df = _theme_df(STAGE2D_SUMMARY, "2d")
theme_compare_probe_df = theme_2c_df.merge(theme_2d_df, on="theme", suffixes=("_2c", "_2d"))
theme_compare_probe_df["overlap_count_delta_2d_minus_2c"] = theme_compare_probe_df["avg_overlap_count_2d"] - theme_compare_probe_df["avg_overlap_count_2c"]
theme_compare_probe_df["overlap_ratio_delta_2d_minus_2c"] = theme_compare_probe_df["avg_overlap_ratio_2d"] - theme_compare_probe_df["avg_overlap_ratio_2c"]
theme_compare_probe_df["out_of_theme_delta_2d_minus_2c"] = theme_compare_probe_df["avg_out_of_theme_factors_2d"] - theme_compare_probe_df["avg_out_of_theme_factors_2c"]
theme_compare_probe_df["default_momentum_ratio_delta_2d_minus_2c"] = theme_compare_probe_df["default_momentum_usage_ratio_2d"] - theme_compare_probe_df["default_momentum_usage_ratio_2c"]
display(theme_compare_probe_df)

thin_themes = ["volatility_regime", "volume_divergence", "calendar_effect"]
thin_df = theme_compare_probe_df[theme_compare_probe_df["theme"].isin(thin_themes)]
bleed_checks = [
    check_row("thin themes have low Stage 2d overlap ratios", bool((thin_df["avg_overlap_ratio_2d"] < 0.25).all()), thin_df[["theme", "avg_overlap_ratio_2d"]]),
    check_row("thin themes have high Stage 2d out-of-theme factor counts", bool((thin_df["avg_out_of_theme_factors_2d"] >= 4.0).all()), thin_df[["theme", "avg_out_of_theme_factors_2d"]]),
    check_row("thin themes use default momentum factors in every valid Stage 2d call", bool((thin_df["default_momentum_usage_ratio_2d"] == 1.0).all()), thin_df[["theme", "default_momentum_usage_ratio_2d"]]),
    check_row("thin-theme overlap ratios are lower in Stage 2d than Stage 2c", bool((thin_df["overlap_ratio_delta_2d_minus_2c"] < 0).all()), thin_df[["theme", "overlap_ratio_delta_2d_minus_2c"]]),
]
display_check_table(bleed_checks, "Probe 5 — thin-theme momentum-bleed checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_5_theme_bleed"] = True


,stage_2c,theme,valid_count_2c,avg_overlap_count_2c,avg_overlap_ratio_2c,avg_out_of_theme_factors_2c,contains_momentum_default_count_2c,default_momentum_usage_ratio_2c,dominant_factors_top3_2c,stage_2d,...,avg_overlap_count_2d,avg_overlap_ratio_2d,avg_out_of_theme_factors_2d,contains_momentum_default_count_2d,default_momentum_usage_ratio_2d,dominant_factors_top3_2d,overlap_count_delta_2d_minus_2c,overlap_ratio_delta_2d_minus_2c,out_of_theme_delta_2d_minus_2c,default_momentum_ratio_delta_2d_minus_2c
0,2c,momentum,4,2.50,0.687500,1.00,4,1.00,"[macd_hist, rsi_14, return_1h]",2d,...,3.179487,0.662943,1.820513,39,1.0,"[macd_hist, rsi_14, return_24h]",0.679487,-0.024557,0.820513,0.00
1,2c,mean_reversion,4,2.75,0.562500,2.25,3,0.75,"[bb_upper_24_2, close, rsi_14]",2d,...,2.900000,0.481071,3.200000,40,1.0,"[close, bb_upper_24_2, rsi_14]",0.150000,-0.081429,0.950000,0.25
2,2c,volatility_regime,4,1.25,0.266667,3.50,2,0.50,"[close, realized_vol_24h, sma_20]",2d,...,1.150000,0.194821,4.825000,40,1.0,"[realized_vol_24h, close, return_24h]",-0.100000,-0.071845,1.325000,0.50
3,2c,volume_divergence,4,1.00,0.237500,3.50,4,1.00,"[volume_zscore_24h, close, return_24h]",2d,...,1.000000,0.192440,4.425000,40,1.0,"[volume_zscore_24h, rsi_14, return_24h]",0.000000,-0.045060,0.925000,0.00
4,2c,calendar_effect,4,1.50,0.350000,3.00,4,1.00,"[day_of_week, return_24h, return_1h]",2d,...,1.150000,0.208780,4.500000,40,1.0,"[day_of_week, rsi_14, return_24h]",-0.350000,-0.141220,1.500000,0.00


Probe 5 — thin-theme momentum-bleed checks


,check,status,detail
0,thin themes have low Stage 2d overlap ratios,PASS,theme avg_overlap_ratio_2d 2 ...
1,thin themes have high Stage 2d out-of-theme fa...,PASS,theme avg_out_of_theme_factors...
2,thin themes use default momentum factors in ev...,PASS,theme default_momentum_usage_r...
3,thin-theme overlap ratios are lower in Stage 2...,PASS,theme overlap_ratio_delta_2d_m...


### Probe 6 — Theme-Coherence Scoring Inputs For D7

This probe verifies the specific Stage 2c → Stage 2d shifts: `calendar_effect` overlap drops from `1.50` to `1.15`, while `momentum` overlap rises from `2.50` to approximately `3.18`. These are not Stage 2d gate failures; they are design inputs for D7 theme-coherence scoring.


In [55]:
calendar_row = theme_compare_probe_df[theme_compare_probe_df["theme"] == "calendar_effect"].iloc[0]
momentum_row = theme_compare_probe_df[theme_compare_probe_df["theme"] == "momentum"].iloc[0]
coherence_probe_df = pd.DataFrame([
    {
        "theme": "calendar_effect",
        "2c_avg_overlap_count": calendar_row["avg_overlap_count_2c"],
        "2d_avg_overlap_count": calendar_row["avg_overlap_count_2d"],
        "delta": calendar_row["overlap_count_delta_2d_minus_2c"],
        "interpretation": "calendar coherence weakened at N=40",
    },
    {
        "theme": "momentum",
        "2c_avg_overlap_count": momentum_row["avg_overlap_count_2c"],
        "2d_avg_overlap_count": momentum_row["avg_overlap_count_2d"],
        "delta": momentum_row["overlap_count_delta_2d_minus_2c"],
        "interpretation": "momentum adherence strengthened at N=40",
    },
])
display(coherence_probe_df)

theme_coherence_checks = [
    check_row("calendar_effect avg_overlap drops from 1.50 to 1.15", calendar_row["avg_overlap_count_2c"] == 1.5 and calendar_row["avg_overlap_count_2d"] == 1.15 and calendar_row["overlap_count_delta_2d_minus_2c"] < 0, calendar_row.to_dict()),
    check_row("momentum avg_overlap rises from 2.50 to ~3.18", momentum_row["avg_overlap_count_2c"] == 2.5 and abs(momentum_row["avg_overlap_count_2d"] - 3.1794871794871793) < 1e-12 and momentum_row["overlap_count_delta_2d_minus_2c"] > 0, momentum_row.to_dict()),
    check_row("D7 needs theme-coherence scoring inputs preserved", {"avg_overlap_ratio_2d", "avg_out_of_theme_factors_2d", "default_momentum_usage_ratio_2d"}.issubset(theme_compare_probe_df.columns), list(theme_compare_probe_df.columns)),
]
display_check_table(theme_coherence_checks, "Probe 6 — D7 theme-coherence input checks")
D6_STAGE2D_PROBE_ACCEPTANCE["probe_6_theme_coherence_inputs"] = True

probe_final_rows = [
    {"probe": "1. contract stability", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_1_contract_stability", False)},
    {"probe": "2. rsi_14 dominance", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_2_rsi_dominance", False)},
    {"probe": "3. unique factor-set trajectory", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_3_unique_fs_trajectory", False)},
    {"probe": "4. repeated factor-set forensic", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_4_repeated_factor_sets", False)},
    {"probe": "5. thin-theme momentum bleed", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_5_theme_bleed", False)},
    {"probe": "6. theme-coherence scoring inputs", "pass": D6_STAGE2D_PROBE_ACCEPTANCE.get("probe_6_theme_coherence_inputs", False)},
]
probe_final_df = pd.DataFrame(probe_final_rows)
probe_final_df["status"] = probe_final_df["pass"].map({True: "PASS", False: "FAIL"})
display(probe_final_df[["probe", "status"]])
assert probe_final_df["pass"].all(), probe_final_df[~probe_final_df["pass"]]

print("Stage 2d reviewer probes 1-6: PASS")
print("These probes confirm the reviewer observations and preserve them as D7 Critic design inputs, not Stage 2d execution-gate failures.")


,theme,2c_avg_overlap_count,2d_avg_overlap_count,delta,interpretation
0,calendar_effect,1.5,1.150000,-0.350000,calendar coherence weakened at N=40
1,momentum,2.5,3.179487,0.679487,momentum adherence strengthened at N=40


Probe 6 — D7 theme-coherence input checks


,check,status,detail
0,calendar_effect avg_overlap drops from 1.50 to...,PASS,"{'stage_2c': '2c', 'theme': 'calendar_effect',..."
1,momentum avg_overlap rises from 2.50 to ~3.18,PASS,"{'stage_2c': '2c', 'theme': 'momentum', 'valid..."
2,D7 needs theme-coherence scoring inputs preserved,PASS,"[stage_2c, theme, valid_count_2c, avg_overlap_..."


,probe,status
0,1. contract stability,PASS
1,2. rsi_14 dominance,PASS
2,3. unique factor-set trajectory,PASS
3,4. repeated factor-set forensic,PASS
4,5. thin-theme momentum bleed,PASS
5,6. theme-coherence scoring inputs,PASS


Stage 2d reviewer probes 1-6: PASS
These probes confirm the reviewer observations and preserve them as D7 Critic design inputs, not Stage 2d execution-gate failures.


## AW. D7 Stage 1 Scope and Contracts

D7 Stage 1 verifies critic plumbing, not critic intelligence.

The contract is deliberately narrow:

- D7a is a deterministic rule critic whose features and formulas can be independently recomputed.
- D7b is a stub backend with fixed scores and an unmistakable reasoning prefix.
- Inline D7 is an observer sidecar, not a control loop.
- `approved_examples`, lifecycle routing, proposer prompts/contexts, and D6 ledger behavior must not change.
- `run_critic()` is fail-open and never lets critic failures break the D6 batch.
- The reliability fuse exists as telemetry scaffold only; it is not enforced in Stage 1.

**Stage 1 proves loop discipline without introducing critic-model variability. Stage 2 must swap in the live critic backend without relaxing Stage 1 accounting, leakage, sidecar, or fail-open constraints.**


In [56]:
import contextlib
import inspect
import io
import json
import os
import sqlite3
import sys
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display

# Allow D7 cells to run after a kernel restart from either repo root or docs/test notebooks.
_D7_START_CWD = Path.cwd().resolve()
_D7_REPO_ROOT = next(
    (p for p in (_D7_START_CWD, *_D7_START_CWD.parents) if (p / "pyproject.toml").exists() and (p / "agents").exists()),
    None,
)
if _D7_REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {_D7_START_CWD}")
if str(_D7_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_D7_REPO_ROOT))
os.chdir(_D7_REPO_ROOT)

if "status" not in globals():
    def status(ok: bool) -> str:
        return "PASS" if bool(ok) else "FAIL"

if "check_row" not in globals():
    def check_row(check: str, ok: bool, detail="") -> dict:
        return {"check": check, "status": status(ok), "detail": detail}

if "display_check_table" not in globals():
    def display_check_table(rows: list[dict], title: str, require_all: bool = True) -> pd.DataFrame:
        print(title)
        df = pd.DataFrame(rows)
        display(df)
        if require_all:
            failed = df[df["status"] != "PASS"]
            assert failed.empty, failed
        return df

from factors.registry import get_registry
from strategies.dsl import StrategyDSL

from agents.critic.batch_context import (
    DEFAULT_MOMENTUM_FACTORS,
    THEME_ANCHOR_FACTORS,
    THEME_HINTS,
    BatchContext as CriticBatchContext,
)
from agents.critic.d7a_feature_extraction import (
    count_conditions,
    count_entry_groups,
    count_exit_groups,
    extract_factors,
    factor_set_tuple,
    get_description_length,
    get_max_hold_bars,
)
from agents.critic.d7a_rules import (
    complexity_appropriateness,
    compute_rule_flags,
    compute_supporting_measures,
    default_momentum_fallback,
    score_d7a,
    structural_novelty,
    theme_coherence,
)
from agents.critic.d7b_backend import D7B_SCORE_KEYS, D7bBackend
from agents.critic.d7b_stub import STUB_REASONING_PREFIX, StubD7bBackend
from agents.critic.orchestrator import (
    CRITIC_RELIABILITY_FUSE_ENFORCED,
    compute_reliability_stats,
    run_critic,
    should_fuse_halt,
)
from agents.critic.result import CriticResult

import agents.critic.orchestrator as d7_orch_mod
import agents.proposer.stage2d_batch as stage2d_mod
from agents.orchestrator.ingest import PENDING_BACKTEST
from agents.proposer.interface import BatchContext as ProposerBatchContext
from agents.proposer.interface import ProposerOutput
from agents.proposer.stub_backend import classify_raw_json
from agents.themes import THEMES

D7_ACCEPTANCE: dict[str, bool] = {}
D7_REGISTRY = get_registry()
D7_ALL_FACTORS = tuple(D7_REGISTRY.list_names())


def d7_make_dsl(
    *,
    name: str = "d7_acceptance_candidate",
    description: str = "D7 acceptance candidate.",
    entry: list[dict] | None = None,
    exit_: list[dict] | None = None,
    max_hold_bars: int | None = None,
) -> StrategyDSL:
    if entry is None:
        entry = [{"conditions": [{"factor": "sma_20", "op": ">", "value": 50000.0}]}]
    if exit_ is None:
        exit_ = [{"conditions": [{"factor": "sma_20", "op": "<", "value": 45000.0}]}]
    return StrategyDSL.model_validate({
        "name": name,
        "description": description,
        "entry": entry,
        "exit": exit_,
        "position_sizing": "full_equity",
        "max_hold_bars": max_hold_bars,
    })


def d7_make_ctx(
    prior_factor_sets: tuple[tuple[str, ...], ...] = (),
    prior_hashes: tuple[str, ...] = (),
    batch_position: int = 1,
) -> CriticBatchContext:
    return CriticBatchContext(
        prior_factor_sets=prior_factor_sets,
        prior_hashes=prior_hashes,
        batch_position=batch_position,
        theme_hints=THEME_HINTS,
        default_momentum_factors=DEFAULT_MOMENTUM_FACTORS,
        theme_anchor_factors=THEME_ANCHOR_FACTORS,
    )


class D7EmptyDSL:
    """Defensive helper-level object: schema cannot produce an empty factor set."""
    entry: list = []
    exit: list = []
    description: str = "empty factor set defensive object"
    max_hold_bars: int | None = None


class D7FailingBackend(D7bBackend):
    @property
    def mode(self):
        return "stub"

    def score(self, dsl, theme, batch_context):
        raise RuntimeError("injected D7b failure")


print("D7 notebook repo root:", _D7_REPO_ROOT)
print("Registered factors:", len(D7_ALL_FACTORS))
D7_ACCEPTANCE["scope_contract"] = True


D7 notebook repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Registered factors: 18


## AX. CriticResult Schema and Surface Audit

This section verifies the D7 Stage 1 artifact surface: critic modules exist, `CriticResult` round-trips through JSON-compatible dicts, null means unknown while zero means measured bad, `--with-critic` defaults off, and the reliability fuse is scaffolded but not enforced.


In [57]:
critic_files = [
    "agents/critic/__init__.py",
    "agents/critic/result.py",
    "agents/critic/batch_context.py",
    "agents/critic/d7a_feature_extraction.py",
    "agents/critic/d7a_rules.py",
    "agents/critic/d7b_backend.py",
    "agents/critic/d7b_stub.py",
    "agents/critic/orchestrator.py",
]
component_df = pd.DataFrame([
    {"artifact": path, "exists": Path(path).exists()} for path in critic_files
])
display(component_df)

surface_dsl = d7_make_dsl()
surface_ctx = d7_make_ctx()
surface_result = run_critic(surface_dsl, "momentum", surface_ctx, StubD7bBackend())
round_trip_result = CriticResult.from_dict(surface_result.to_dict())

none_scores = CriticResult(
    critic_version="d7_v1",
    critic_status="d7a_error",
    d7b_mode="stub",
    d7a_rule_scores=None,
    d7a_supporting_measures={},
    d7a_rule_flags=[],
    d7b_llm_scores={k: 0.5 for k in D7B_SCORE_KEYS},
    d7b_reasoning="stub",
    d7b_raw_response_path=None,
    d7b_cost_actual_usd=0.0,
    d7b_input_tokens=0,
    d7b_output_tokens=0,
    d7b_retry_count=0,
)
zero_scores = CriticResult(
    critic_version="d7_v1",
    critic_status="ok",
    d7b_mode="stub",
    d7a_rule_scores={
        "theme_coherence": 0.0,
        "structural_novelty": 0.0,
        "default_momentum_fallback": 0.0,
        "complexity_appropriateness": 0.0,
    },
    d7a_supporting_measures={},
    d7a_rule_flags=[],
    d7b_llm_scores={k: 0.5 for k in D7B_SCORE_KEYS},
    d7b_reasoning="stub",
    d7b_raw_response_path=None,
    d7b_cost_actual_usd=0.0,
    d7b_input_tokens=0,
    d7b_output_tokens=0,
    d7b_retry_count=0,
)

stage2d_src = inspect.getsource(stage2d_mod)
claude_text = Path("CLAUDE.md").read_text(encoding="utf-8")
result_src = inspect.getsource(CriticResult)

surface_checks = [
    check_row("all D7 critic module artifacts exist", bool(component_df["exists"].all())),
    check_row("CriticResult is a frozen dataclass-like schema", hasattr(CriticResult, "__dataclass_fields__")),
    check_row("CriticResult to_dict/from_dict round-trip is exact", round_trip_result == surface_result),
    check_row("None score block remains None", none_scores.to_dict()["d7a_rule_scores"] is None),
    check_row("zero score block remains measured zeros", all(v == 0.0 for v in zero_scores.to_dict()["d7a_rule_scores"].values())),
    check_row("D7b score keys are frozen", set(D7B_SCORE_KEYS) == {"semantic_plausibility", "semantic_theme_alignment", "structural_variant_risk"}, D7B_SCORE_KEYS),
    check_row("structural_variant_risk reverse-polarity warning is documented", "reverse-polarity" in result_src and "structural_variant_risk" in result_src),
    check_row("stage2d run_stage2d has with_critic default off", inspect.signature(stage2d_mod.run_stage2d).parameters["with_critic"].default is False),
    check_row("CLI exposes --with-critic explicit opt-in", "--with-critic" in stage2d_src),
    check_row("CLAUDE.md protects critic sidecar boundary", "critic_result" in claude_text and "with_critic=False" in claude_text and "approved_examples" in claude_text),
    check_row("critic reliability fuse scaffold is not enforced", CRITIC_RELIABILITY_FUSE_ENFORCED is False and should_fuse_halt(100, 0.95) is False),
]
display_check_table(surface_checks, "D7 Stage 1 schema and artifact surface checks")
D7_ACCEPTANCE["schema_surface"] = True


,artifact,exists
0,agents/critic/__init__.py,True
1,agents/critic/result.py,True
2,agents/critic/batch_context.py,True
3,agents/critic/d7a_feature_extraction.py,True
4,agents/critic/d7a_rules.py,True
5,agents/critic/d7b_backend.py,True
6,agents/critic/d7b_stub.py,True
7,agents/critic/orchestrator.py,True


D7 Stage 1 schema and artifact surface checks


,check,status,detail
0,all D7 critic module artifacts exist,PASS,
1,CriticResult is a frozen dataclass-like schema,PASS,
2,CriticResult to_dict/from_dict round-trip is e...,PASS,
3,None score block remains None,PASS,
4,zero score block remains measured zeros,PASS,
5,D7b score keys are frozen,PASS,"(semantic_plausibility, semantic_theme_alignme..."
6,structural_variant_risk reverse-polarity warni...,PASS,
7,stage2d run_stage2d has with_critic default off,PASS,
8,CLI exposes --with-critic explicit opt-in,PASS,
9,CLAUDE.md protects critic sidecar boundary,PASS,


## AY. D7a Feature Extraction Audit

D7a rules are only as good as their structural features. These constructed examples independently verify factor extraction, factor-vs-factor RHS extraction, condition/group counts, sorted factor-set tuples, max-hold reporting, and the defensive empty-factor-set path.


In [58]:
feature_cases = [
    {
        "case": "pure momentum",
        "dsl": d7_make_dsl(
            name="d7_pure_momentum",
            entry=[{"conditions": [
                {"factor": "rsi_14", "op": ">", "value": 30.0},
                {"factor": "return_1h", "op": ">", "value": 0.01},
            ]}],
            exit_=[{"conditions": [{"factor": "macd_hist", "op": "<", "value": 0.0}]}],
            max_hold_bars=24,
        ),
        "expected_factors": ["macd_hist", "return_1h", "rsi_14"],
        "expected_conditions": 3,
        "expected_entry_groups": 1,
        "expected_exit_groups": 1,
        "expected_hold": 24,
    },
    {
        "case": "factor-vs-factor RHS included",
        "dsl": d7_make_dsl(
            name="d7_factor_vs_factor",
            entry=[{"conditions": [{"factor": "sma_20", "op": ">", "value": "sma_50"}]}],
            exit_=[{"conditions": [{"factor": "rsi_14", "op": "<", "value": 40.0}]}],
        ),
        "expected_factors": ["rsi_14", "sma_20", "sma_50"],
        "expected_conditions": 2,
        "expected_entry_groups": 1,
        "expected_exit_groups": 1,
        "expected_hold": None,
    },
    {
        "case": "repeated LHS factor deduped",
        "dsl": d7_make_dsl(
            name="d7_repeated_factor",
            entry=[{"conditions": [{"factor": "rsi_14", "op": ">", "value": 30.0}]}],
            exit_=[{"conditions": [{"factor": "rsi_14", "op": "<", "value": 70.0}]}],
        ),
        "expected_factors": ["rsi_14"],
        "expected_conditions": 2,
        "expected_entry_groups": 1,
        "expected_exit_groups": 1,
        "expected_hold": None,
    },
    {
        "case": "defensive empty factor set",
        "dsl": D7EmptyDSL(),
        "expected_factors": [],
        "expected_conditions": 0,
        "expected_entry_groups": 0,
        "expected_exit_groups": 0,
        "expected_hold": None,
    },
]

feature_rows = []
for case in feature_cases:
    dsl_obj = case["dsl"]
    actual_factors = extract_factors(dsl_obj)
    actual_tuple = factor_set_tuple(dsl_obj)
    feature_rows.append({
        "case": case["case"],
        "factors": actual_factors,
        "expected_factors": case["expected_factors"],
        "factor_tuple": actual_tuple,
        "conditions": count_conditions(dsl_obj),
        "entry_groups": count_entry_groups(dsl_obj),
        "exit_groups": count_exit_groups(dsl_obj),
        "max_hold_bars": get_max_hold_bars(dsl_obj),
        "description_length": get_description_length(dsl_obj),
        "pass": (
            actual_factors == case["expected_factors"]
            and actual_tuple == tuple(case["expected_factors"])
            and count_conditions(dsl_obj) == case["expected_conditions"]
            and count_entry_groups(dsl_obj) == case["expected_entry_groups"]
            and count_exit_groups(dsl_obj) == case["expected_exit_groups"]
            and get_max_hold_bars(dsl_obj) == case["expected_hold"]
        ),
    })
feature_df = pd.DataFrame(feature_rows)
display(feature_df)

feature_checks = [
    check_row("all constructed feature extraction cases match expected values", bool(feature_df["pass"].all()), feature_df.loc[~feature_df["pass"]]),
    check_row("factor sets are deterministic sorted tuples", all(tuple(row["factors"]) == row["factor_tuple"] for _, row in feature_df.iterrows())),
    check_row("empty-factor defensive helper path returns empty tuple", feature_df.loc[feature_df["case"] == "defensive empty factor set", "factor_tuple"].iloc[0] == ()),
]
display_check_table(feature_checks, "D7a feature extraction primitive checks")
D7_ACCEPTANCE["feature_extraction"] = True


,case,factors,expected_factors,factor_tuple,conditions,entry_groups,exit_groups,max_hold_bars,description_length,pass
0,pure momentum,"[macd_hist, return_1h, rsi_14]","[macd_hist, return_1h, rsi_14]","(macd_hist, return_1h, rsi_14)",3,1,1,24.0,24,True
1,factor-vs-factor RHS included,"[rsi_14, sma_20, sma_50]","[rsi_14, sma_20, sma_50]","(rsi_14, sma_20, sma_50)",2,1,1,NaN,24,True
2,repeated LHS factor deduped,[rsi_14],[rsi_14],"(rsi_14,)",2,1,1,NaN,24,True
3,defensive empty factor set,[],[],(),0,0,0,NaN,33,True


D7a feature extraction primitive checks


,check,status,detail
0,all constructed feature extraction cases match...,PASS,"Empty DataFrame Columns: [case, factors, expec..."
1,factor sets are deterministic sorted tuples,PASS,
2,empty-factor defensive helper path returns emp...,PASS,


## AZ. D7a Rule Bit-Level Verification

This section recomputes the D7a formulas in notebook code and compares them to the implementation. The cases cover normal theme coherence, thin-theme momentum bleed, repeated factor-set novelty, complexity cliffs, supporting measures, flags, and the defensive empty-factor-set semantics.


In [59]:
def d7_expected_scores(dsl_obj: Any, theme: str, ctx: CriticBatchContext) -> dict:
    factors = set(extract_factors(dsl_obj))
    factor_tuple = factor_set_tuple(dsl_obj)
    hints = ctx.theme_hints.get(theme, frozenset())
    overlap = factors & hints
    theme_score = 0.0 if not factors else round(len(overlap) / len(factors), 4)
    novelty = None if not factor_tuple else round(1.0 / (1.0 + ctx.prior_factor_sets.count(factor_tuple)), 4)

    if not factors:
        momentum_score = 0.0
    elif theme == "momentum":
        momentum_score = 1.0
    else:
        momentum_count = len(factors & ctx.default_momentum_factors)
        if momentum_count == 0:
            momentum_score = 1.0
        elif momentum_count == 1:
            momentum_score = 0.8
        elif momentum_count == 2:
            momentum_score = 0.5
        else:
            momentum_score = round(max(0.0, 0.5 - 0.15 * (momentum_count - 2)), 4)

    n_factors = len(factors)
    n_conditions = count_conditions(dsl_obj)
    desc_len = get_description_length(dsl_obj)
    if n_conditions == 0 or n_factors == 0:
        complexity = 0.0
    elif n_conditions == 1:
        complexity = 0.7
    elif 2 <= n_conditions <= 4:
        complexity = 1.0
    else:
        complexity = max(0.0, 1.0 - 0.15 * (n_conditions - 4))
    if n_conditions > 0 and n_factors > 7:
        complexity *= 0.85
    if n_conditions > 0 and n_factors > 0 and desc_len > 500:
        complexity *= 0.9
    complexity = round(complexity, 4)

    return {
        "theme_coherence": theme_score,
        "structural_novelty": novelty,
        "default_momentum_fallback": momentum_score,
        "complexity_appropriateness": complexity,
    }

pure_momentum_dsl = feature_cases[0]["dsl"]
thin_bleed_dsl = d7_make_dsl(
    name="d7_thin_bleed",
    entry=[{"conditions": [
        {"factor": "rsi_14", "op": ">", "value": 30.0},
        {"factor": "return_1h", "op": ">", "value": 0.01},
        {"factor": "macd_hist", "op": ">", "value": 0.0},
    ]}],
    exit_=[{"conditions": [{"factor": "rsi_14", "op": "<", "value": 70.0}]}],
)
repeated_fs_dsl = d7_make_dsl()
heavy_dsl = d7_make_dsl(
    name="d7_heavy",
    entry=[{"conditions": [
        {"factor": "sma_20", "op": ">", "value": 50000.0},
        {"factor": "rsi_14", "op": ">", "value": 30.0},
        {"factor": "atr_14", "op": ">", "value": 1.0},
        {"factor": "macd_hist", "op": ">", "value": 0.0},
    ]}],
    exit_=[{"conditions": [
        {"factor": "return_1h", "op": "<", "value": 0.0},
        {"factor": "return_24h", "op": "<", "value": 0.0},
    ]}],
)

rule_cases = [
    {"case": "pure momentum full overlap", "dsl": pure_momentum_dsl, "theme": "momentum", "ctx": d7_make_ctx()},
    {"case": "thin theme momentum bleed", "dsl": thin_bleed_dsl, "theme": "volume_divergence", "ctx": d7_make_ctx()},
    {"case": "repeated factor set novelty", "dsl": repeated_fs_dsl, "theme": "momentum", "ctx": d7_make_ctx(prior_factor_sets=(("sma_20",), ("sma_20",), ("sma_20",)))},
    {"case": "six-condition complexity cliff", "dsl": heavy_dsl, "theme": "momentum", "ctx": d7_make_ctx()},
    {"case": "defensive empty factor set", "dsl": D7EmptyDSL(), "theme": "momentum", "ctx": d7_make_ctx()},
]

rule_rows = []
flag_rows = []
for case in rule_cases:
    actual_scores, actual_measures, actual_flags = score_d7a(case["dsl"], case["theme"], case["ctx"])
    expected_scores = d7_expected_scores(case["dsl"], case["theme"], case["ctx"])
    rule_rows.append({
        "case": case["case"],
        "theme": case["theme"],
        "actual_scores": actual_scores,
        "expected_scores": expected_scores,
        "scores_match": actual_scores == expected_scores,
        "measures": actual_measures,
        "flags": actual_flags,
    })

    factors = set(extract_factors(case["dsl"]))
    expected_prior = None if not factors else case["ctx"].prior_factor_sets.count(factor_set_tuple(case["dsl"]))
    flag_rows.append({
        "case": case["case"],
        "factor_set_prior_occurrences": actual_measures.get("factor_set_prior_occurrences"),
        "expected_prior_occurrences": expected_prior,
        "n_factors": actual_measures.get("n_factors"),
        "n_conditions": actual_measures.get("n_conditions"),
        "flags": actual_flags,
    })

rule_df = pd.DataFrame(rule_rows)
flag_df = pd.DataFrame(flag_rows)
display(rule_df)
display(flag_df)

prior_occurrence_matches = all(
    (pd.isna(actual) and pd.isna(expected)) or actual == expected
    for actual, expected in zip(
        flag_df["factor_set_prior_occurrences"],
        flag_df["expected_prior_occurrences"],
    )
)

rule_checks = [
    check_row("all D7a scores match independently recomputed formulas", bool(rule_df["scores_match"].all()), rule_df.loc[~rule_df["scores_match"]]),
    check_row("thin-theme momentum bleed flag fires", "thin_theme_momentum_bleed" in rule_df.loc[rule_df["case"] == "thin theme momentum bleed", "flags"].iloc[0]),
    check_row("theme-anchor-missing flag fires for thin non-anchor candidate", "theme_anchor_missing" in rule_df.loc[rule_df["case"] == "thin theme momentum bleed", "flags"].iloc[0]),
    check_row("top repeated factor-set flag fires", "factor_set_in_top3_repeated" in rule_df.loc[rule_df["case"] == "repeated factor set novelty", "flags"].iloc[0]),
    check_row("heavy condition flag fires", "n_conditions_heavy" in rule_df.loc[rule_df["case"] == "six-condition complexity cliff", "flags"].iloc[0]),
    check_row("empty factor-set defensive semantics are explicit", "empty_factor_set" in rule_df.loc[rule_df["case"] == "defensive empty factor set", "flags"].iloc[0] and rule_df.loc[rule_df["case"] == "defensive empty factor set", "actual_scores"].iloc[0]["structural_novelty"] is None),
    check_row("supporting measure prior-occurrence counts match expected", prior_occurrence_matches, flag_df),
]
display_check_table(rule_checks, "D7a rule and flag checks")
D7_ACCEPTANCE["d7a_rules"] = True


,case,theme,actual_scores,expected_scores,scores_match,measures,flags
0,pure momentum full overlap,momentum,"{'theme_coherence': 1.0, 'structural_novelty':...","{'theme_coherence': 1.0, 'structural_novelty':...",True,"{'factor_set_prior_occurrences': 0, 'n_factors...",[]
1,thin theme momentum bleed,volume_divergence,"{'theme_coherence': 0.0, 'structural_novelty':...","{'theme_coherence': 0.0, 'structural_novelty':...",True,"{'factor_set_prior_occurrences': 0, 'n_factors...","[thin_theme_momentum_bleed, theme_anchor_missing]"
2,repeated factor set novelty,momentum,"{'theme_coherence': 0.0, 'structural_novelty':...","{'theme_coherence': 0.0, 'structural_novelty':...",True,"{'factor_set_prior_occurrences': 3, 'n_factors...",[factor_set_in_top3_repeated]
3,six-condition complexity cliff,momentum,"{'theme_coherence': 0.6667, 'structural_novelt...","{'theme_coherence': 0.6667, 'structural_novelt...",True,"{'factor_set_prior_occurrences': 0, 'n_factors...",[n_conditions_heavy]
4,defensive empty factor set,momentum,"{'theme_coherence': 0.0, 'structural_novelty':...","{'theme_coherence': 0.0, 'structural_novelty':...",True,"{'factor_set_prior_occurrences': None, 'n_fact...",[empty_factor_set]


,case,factor_set_prior_occurrences,expected_prior_occurrences,n_factors,n_conditions,flags
0,pure momentum full overlap,0.0,0.0,3,3,[]
1,thin theme momentum bleed,0.0,0.0,3,4,"[thin_theme_momentum_bleed, theme_anchor_missing]"
2,repeated factor set novelty,3.0,3.0,1,2,[factor_set_in_top3_repeated]
3,six-condition complexity cliff,0.0,0.0,6,6,[n_conditions_heavy]
4,defensive empty factor set,NaN,NaN,0,0,[empty_factor_set]


D7a rule and flag checks


,check,status,detail
0,all D7a scores match independently recomputed ...,PASS,"Empty DataFrame Columns: [case, theme, actual_..."
1,thin-theme momentum bleed flag fires,PASS,
2,theme-anchor-missing flag fires for thin non-a...,PASS,
3,top repeated factor-set flag fires,PASS,
4,heavy condition flag fires,PASS,
5,empty factor-set defensive semantics are explicit,PASS,
6,supporting measure prior-occurrence counts mat...,PASS,case factor_set_...


### AZ1. Complexity Appropriateness Edge Table Closure

`complexity_appropriateness` is the D7a rule with the highest mirrored-bug risk because it has cliffs, floors, and multiplicative penalties. This cell pins the full 12-row edge table independently from the composite examples above.

Note: the current checked-in D7 rule test pins `(n_conditions=3, n_factors=8, desc_len=600)` at `0.765`, because the locked formula applies `0.85 × 0.90`. A review note cited `0.7225`; that is not the current implementation/test contract. This notebook surfaces the difference rather than hiding it.


In [60]:
import agents.critic.d7a_feature_extraction as d7_fe_mod

complexity_edge_rows = [
    {"n_conditions": 0, "n_factors": 3, "desc_len": 100, "expected": 0.0, "contract_note": "zero conditions floor"},
    {"n_conditions": 1, "n_factors": 3, "desc_len": 100, "expected": 0.7, "contract_note": "single-condition cliff"},
    {"n_conditions": 2, "n_factors": 3, "desc_len": 100, "expected": 1.0, "contract_note": "ideal low complexity"},
    {"n_conditions": 4, "n_factors": 4, "desc_len": 100, "expected": 1.0, "contract_note": "upper ideal condition count"},
    {"n_conditions": 5, "n_factors": 4, "desc_len": 100, "expected": 0.85, "contract_note": "first condition penalty"},
    {"n_conditions": 6, "n_factors": 4, "desc_len": 100, "expected": 0.7, "contract_note": "second condition penalty"},
    {"n_conditions": 10, "n_factors": 4, "desc_len": 100, "expected": 0.1, "contract_note": "deep cliff"},
    {"n_conditions": 20, "n_factors": 4, "desc_len": 100, "expected": 0.0, "contract_note": "zero floor"},
    {"n_conditions": 3, "n_factors": 0, "desc_len": 100, "expected": 0.0, "contract_note": "zero factor floor"},
    {"n_conditions": 3, "n_factors": 8, "desc_len": 100, "expected": 0.85, "contract_note": "many-factor penalty"},
    {"n_conditions": 3, "n_factors": 4, "desc_len": 600, "expected": 0.9, "contract_note": "long-description penalty"},
    {"n_conditions": 3, "n_factors": 8, "desc_len": 600, "expected": 0.765, "review_cited": 0.7225, "contract_note": "stacked many-factor x long-description penalty"},
]

edge_carrier_dsl = d7_make_dsl()
original_extract = d7_fe_mod.extract_factors
original_count = d7_fe_mod.count_conditions
original_desc = d7_fe_mod.get_description_length
try:
    for row in complexity_edge_rows:
        fake_factors = [f"f{i}" for i in range(row["n_factors"])]
        d7_fe_mod.extract_factors = lambda _dsl, fake_factors=fake_factors: fake_factors
        d7_fe_mod.count_conditions = lambda _dsl, n=row["n_conditions"]: n
        d7_fe_mod.get_description_length = lambda _dsl, d=row["desc_len"]: d
        row["actual"] = complexity_appropriateness(edge_carrier_dsl)
        row["abs_diff"] = abs(row["actual"] - row["expected"])
        row["pass"] = row["abs_diff"] <= 1e-12
finally:
    d7_fe_mod.extract_factors = original_extract
    d7_fe_mod.count_conditions = original_count
    d7_fe_mod.get_description_length = original_desc

complexity_edge_df = pd.DataFrame(complexity_edge_rows)
display(complexity_edge_df)

complexity_edge_checks = [
    check_row("all 12 complexity edge-table rows match locked contract", bool(complexity_edge_df["pass"].all()), complexity_edge_df.loc[~complexity_edge_df["pass"]]),
    check_row("deep cliff row 10 conditions / 4 factors is pinned at 0.1", float(complexity_edge_df.query("n_conditions == 10 and n_factors == 4")["actual"].iloc[0]) == 0.1),
    check_row("zero floor row 20 conditions / 4 factors is pinned at 0.0", float(complexity_edge_df.query("n_conditions == 20 and n_factors == 4")["actual"].iloc[0]) == 0.0),
    check_row("long-description penalty row is pinned at 0.9", float(complexity_edge_df.query("n_conditions == 3 and n_factors == 4 and desc_len == 600")["actual"].iloc[0]) == 0.9),
    check_row("stacked penalty row follows current locked 0.765 contract", float(complexity_edge_df.query("n_conditions == 3 and n_factors == 8 and desc_len == 600")["actual"].iloc[0]) == 0.765, "review note cited 0.7225; current tests pin 0.765"),
]
display_check_table(complexity_edge_checks, "D7a complexity edge-table closure checks")
D7_ACCEPTANCE["complexity_edge_table"] = True


,n_conditions,n_factors,desc_len,expected,contract_note,actual,abs_diff,pass,review_cited
0,0,3,100,0.000,zero conditions floor,0.000,0.0,True,NaN
1,1,3,100,0.700,single-condition cliff,0.700,0.0,True,NaN
2,2,3,100,1.000,ideal low complexity,1.000,0.0,True,NaN
3,4,4,100,1.000,upper ideal condition count,1.000,0.0,True,NaN
4,5,4,100,0.850,first condition penalty,0.850,0.0,True,NaN
5,6,4,100,0.700,second condition penalty,0.700,0.0,True,NaN
6,10,4,100,0.100,deep cliff,0.100,0.0,True,NaN
7,20,4,100,0.000,zero floor,0.000,0.0,True,NaN
8,3,0,100,0.000,zero factor floor,0.000,0.0,True,NaN
9,3,8,100,0.850,many-factor penalty,0.850,0.0,True,NaN


D7a complexity edge-table closure checks


,check,status,detail
0,all 12 complexity edge-table rows match locked...,PASS,"Empty DataFrame Columns: [n_conditions, n_fact..."
1,deep cliff row 10 conditions / 4 factors is pi...,PASS,
2,zero floor row 20 conditions / 4 factors is pi...,PASS,
3,long-description penalty row is pinned at 0.9,PASS,
4,stacked penalty row follows current locked 0.7...,PASS,review note cited 0.7225; current tests pin 0.765


### AZ2. Expanded D7a Flag Coverage

The earlier rule table verifies the main flags, but Stage 1 sign-off needs the full fragile surface: description near-limit, OR-anchor behavior for calendar and volatility themes, and multi-flag coexistence. The synthetic empty-factor object remains a helper-level defensive proof only; production proposer output cannot reach it because the D2 schema requires at least one condition.


In [61]:
def d7_flags_with_desc_override(dsl_obj, theme: str, ctx: CriticBatchContext, desc_len: int | None = None) -> list[str]:
    if desc_len is None:
        return compute_rule_flags(dsl_obj, theme, ctx)
    original = d7_fe_mod.get_description_length
    try:
        d7_fe_mod.get_description_length = lambda _dsl, d=desc_len: d
        return compute_rule_flags(dsl_obj, theme, ctx)
    finally:
        d7_fe_mod.get_description_length = original

calendar_anchor_missing_dsl = d7_make_dsl(
    name="d7_calendar_missing",
    entry=[{"conditions": [{"factor": "rsi_14", "op": ">", "value": 30.0}]}],
    exit_=[{"conditions": [{"factor": "rsi_14", "op": "<", "value": 70.0}]}],
)
calendar_anchor_present_dsl = d7_make_dsl(
    name="d7_calendar_present",
    entry=[{"conditions": [{"factor": "day_of_week", "op": "==", "value": 1.0}]}],
    exit_=[{"conditions": [{"factor": "hour_of_day", "op": "==", "value": 12.0}]}],
)
vol_anchor_missing_dsl = d7_make_dsl(
    name="d7_vol_missing",
    entry=[{"conditions": [{"factor": "rsi_14", "op": ">", "value": 30.0}]}],
    exit_=[{"conditions": [{"factor": "macd_hist", "op": "<", "value": 0.0}]}],
)
vol_anchor_present_dsl = d7_make_dsl(
    name="d7_vol_present",
    entry=[{"conditions": [{"factor": "atr_14", "op": ">", "value": 1.0}]}],
    exit_=[{"conditions": [{"factor": "realized_vol_24h", "op": "<", "value": 0.02}]}],
)
near_limit_dsl = d7_make_dsl(name="d7_desc_near_limit")
multi_flag_dsl = d7_make_dsl(
    name="d7_multi_flag",
    entry=[{"conditions": [
        {"factor": "rsi_14", "op": ">", "value": 30.0},
        {"factor": "return_1h", "op": ">", "value": 0.01},
    ]}],
    exit_=[{"conditions": [{"factor": "rsi_14", "op": "<", "value": 70.0}]}],
)
multi_flag_fs = factor_set_tuple(multi_flag_dsl)

flag_case_rows = [
    {"case": "description_length_near_limit", "flags": d7_flags_with_desc_override(near_limit_dsl, "momentum", d7_make_ctx(), desc_len=420), "expected_present": {"description_length_near_limit"}, "expected_absent": set()},
    {"case": "calendar_effect anchor missing", "flags": d7_flags_with_desc_override(calendar_anchor_missing_dsl, "calendar_effect", d7_make_ctx()), "expected_present": {"theme_anchor_missing"}, "expected_absent": set()},
    {"case": "calendar_effect OR anchor present", "flags": d7_flags_with_desc_override(calendar_anchor_present_dsl, "calendar_effect", d7_make_ctx()), "expected_present": set(), "expected_absent": {"theme_anchor_missing"}},
    {"case": "volatility_regime anchor missing", "flags": d7_flags_with_desc_override(vol_anchor_missing_dsl, "volatility_regime", d7_make_ctx()), "expected_present": {"theme_anchor_missing", "thin_theme_momentum_bleed"}, "expected_absent": set()},
    {"case": "volatility_regime OR anchor present", "flags": d7_flags_with_desc_override(vol_anchor_present_dsl, "volatility_regime", d7_make_ctx()), "expected_present": set(), "expected_absent": {"theme_anchor_missing"}},
    {"case": "multi-flag coexistence", "flags": d7_flags_with_desc_override(multi_flag_dsl, "volume_divergence", d7_make_ctx(prior_factor_sets=(multi_flag_fs, multi_flag_fs)), desc_len=420), "expected_present": {"thin_theme_momentum_bleed", "theme_anchor_missing", "description_length_near_limit", "factor_set_in_top3_repeated"}, "expected_absent": set()},
]

for row in flag_case_rows:
    flags = set(row["flags"])
    row["present_ok"] = row["expected_present"].issubset(flags)
    row["absent_ok"] = flags.isdisjoint(row["expected_absent"])
    row["pass"] = row["present_ok"] and row["absent_ok"]
    row["expected_present"] = sorted(row["expected_present"])
    row["expected_absent"] = sorted(row["expected_absent"])

flag_coverage_df = pd.DataFrame(flag_case_rows)
display(flag_coverage_df)

flag_coverage_checks = [
    check_row("all expanded flag cases match expected presence/absence", bool(flag_coverage_df["pass"].all()), flag_coverage_df.loc[~flag_coverage_df["pass"]]),
    check_row("description near-limit flag is explicitly covered", "description_length_near_limit" in flag_coverage_df.loc[flag_coverage_df["case"] == "description_length_near_limit", "flags"].iloc[0]),
    check_row("calendar OR-anchor missing and present paths are both covered", bool(flag_coverage_df[flag_coverage_df["case"].str.contains("calendar_effect")]["pass"].all())),
    check_row("volatility OR-anchor missing and present paths are both covered", bool(flag_coverage_df[flag_coverage_df["case"].str.contains("volatility_regime")]["pass"].all())),
    check_row("multi-flag coexistence is explicit", len(flag_coverage_df.loc[flag_coverage_df["case"] == "multi-flag coexistence", "flags"].iloc[0]) >= 4),
]
display_check_table(flag_coverage_checks, "D7a expanded flag coverage checks")
D7_ACCEPTANCE["expanded_flag_coverage"] = True


,case,flags,expected_present,expected_absent,present_ok,absent_ok,pass
0,description_length_near_limit,[description_length_near_limit],[description_length_near_limit],[],True,True,True
1,calendar_effect anchor missing,[theme_anchor_missing],[theme_anchor_missing],[],True,True,True
2,calendar_effect OR anchor present,[],[],[theme_anchor_missing],True,True,True
3,volatility_regime anchor missing,"[thin_theme_momentum_bleed, theme_anchor_missing]","[theme_anchor_missing, thin_theme_momentum_bleed]",[],True,True,True
4,volatility_regime OR anchor present,[],[],[theme_anchor_missing],True,True,True
5,multi-flag coexistence,"[thin_theme_momentum_bleed, factor_set_in_top3...","[description_length_near_limit, factor_set_in_...",[],True,True,True


D7a expanded flag coverage checks


,check,status,detail
0,all expanded flag cases match expected presenc...,PASS,"Empty DataFrame Columns: [case, flags, expecte..."
1,description near-limit flag is explicitly covered,PASS,
2,calendar OR-anchor missing and present paths a...,PASS,
3,volatility OR-anchor missing and present paths...,PASS,
4,multi-flag coexistence is explicit,PASS,


## BA. D7b Stub Audit

Stage 1 must not accidentally call a live critic. The stub backend is deterministic, fixed at 0.5 on all D7b axes, carries zero/null cost telemetry, and emits a magic reasoning prefix that downstream consumers can defensively assert.


In [62]:
stub_backend = StubD7bBackend()
stub_dsl = d7_make_dsl()
stub_ctx = d7_make_ctx()
stub_scores_1, stub_reasoning_1, stub_meta_1 = stub_backend.score(stub_dsl, "momentum", stub_ctx)
stub_scores_2, stub_reasoning_2, stub_meta_2 = stub_backend.score(stub_dsl, "momentum", stub_ctx)
stub_result = run_critic(stub_dsl, "momentum", stub_ctx, stub_backend)
stub_100_results = [stub_backend.score(stub_dsl, "momentum", stub_ctx) for _ in range(100)]
stub_src = inspect.getsource(StubD7bBackend)

stub_summary = pd.DataFrame([
    {"item": "mode", "value": stub_backend.mode},
    {"item": "scores", "value": stub_scores_1},
    {"item": "reasoning_prefix", "value": stub_reasoning_1[:40]},
    {"item": "metadata", "value": stub_meta_1},
    {"item": "critic_status via run_critic", "value": stub_result.critic_status},
])
display(stub_summary)

stub_checks = [
    check_row("stub backend mode is stub", stub_backend.mode == "stub"),
    check_row("stub score output is deterministic", (stub_scores_1, stub_reasoning_1, stub_meta_1) == (stub_scores_2, stub_reasoning_2, stub_meta_2)),
    check_row("stub remains deterministic across 100 calls", all(result == stub_100_results[0] for result in stub_100_results)),
    check_row("all D7b score keys are present", set(stub_scores_1) == set(D7B_SCORE_KEYS), stub_scores_1),
    check_row("all D7b stub scores equal 0.5", all(v == 0.5 for v in stub_scores_1.values()), stub_scores_1),
    check_row("reasoning carries magic stub prefix", stub_reasoning_1.startswith(STUB_REASONING_PREFIX), stub_reasoning_1[:80]),
    check_row("stub cost/token telemetry is zero or null", stub_meta_1 == {"raw_response_path": None, "cost_actual_usd": 0.0, "input_tokens": 0, "output_tokens": 0, "retry_count": 0}, stub_meta_1),
    check_row("stub source contains no live-model dependency", "anthropic" not in stub_src.lower() and "sonnet" not in stub_src.lower().replace("stage 2 replaces this with a live sonnet critic", "")),
]
display_check_table(stub_checks, "D7b stub checks")
D7_ACCEPTANCE["d7b_stub"] = True


,item,value
0,mode,stub
1,scores,"{'semantic_plausibility': 0.5, 'semantic_theme..."
2,reasoning_prefix,"[STUB REASONING — D7b is in stub mode, n"
3,metadata,"{'raw_response_path': None, 'cost_actual_usd':..."
4,critic_status via run_critic,ok


D7b stub checks


,check,status,detail
0,stub backend mode is stub,PASS,
1,stub score output is deterministic,PASS,
2,stub remains deterministic across 100 calls,PASS,
3,all D7b score keys are present,PASS,"{'semantic_plausibility': 0.5, 'semantic_theme..."
4,all D7b stub scores equal 0.5,PASS,"{'semantic_plausibility': 0.5, 'semantic_theme..."
5,reasoning carries magic stub prefix,PASS,"[STUB REASONING — D7b is in stub mode, no live..."
6,stub cost/token telemetry is zero or null,PASS,"{'raw_response_path': None, 'cost_actual_usd':..."
7,stub source contains no live-model dependency,PASS,


## BB. Orchestrator Fail-Open and Fuse Scaffold Audit

`run_critic()` is a fail-open sidecar. D7a errors, D7b errors, and both-error cases must return `CriticResult` records instead of raising. The reliability fuse is counted but remains non-enforcing in Stage 1.


In [63]:
failopen_dsl = d7_make_dsl()
failopen_ctx = d7_make_ctx()

ok_result = run_critic(failopen_dsl, "momentum", failopen_ctx, StubD7bBackend())
d7b_error_result = run_critic(failopen_dsl, "momentum", failopen_ctx, D7FailingBackend())

original_score_d7a = d7_orch_mod.score_d7a
try:
    def _d7a_raises(*args, **kwargs):
        raise ValueError("injected D7a rule failure")
    d7_orch_mod.score_d7a = _d7a_raises
    d7a_error_result = d7_orch_mod.run_critic(failopen_dsl, "momentum", failopen_ctx, StubD7bBackend())
    both_error_result = d7_orch_mod.run_critic(failopen_dsl, "momentum", failopen_ctx, D7FailingBackend())
finally:
    d7_orch_mod.score_d7a = original_score_d7a

failopen_rows = [
    {
        "case": "ok",
        "status": ok_result.critic_status,
        "d7a_scores_present": ok_result.d7a_rule_scores is not None,
        "d7b_scores_present": ok_result.d7b_llm_scores is not None,
    },
    {
        "case": "D7a raises",
        "status": d7a_error_result.critic_status,
        "d7a_scores_present": d7a_error_result.d7a_rule_scores is not None,
        "d7b_scores_present": d7a_error_result.d7b_llm_scores is not None,
    },
    {
        "case": "D7b raises",
        "status": d7b_error_result.critic_status,
        "d7a_scores_present": d7b_error_result.d7a_rule_scores is not None,
        "d7b_scores_present": d7b_error_result.d7b_llm_scores is not None,
    },
    {
        "case": "both raise",
        "status": both_error_result.critic_status,
        "d7a_scores_present": both_error_result.d7a_rule_scores is not None,
        "d7b_scores_present": both_error_result.d7b_llm_scores is not None,
    },
]
failopen_df = pd.DataFrame(failopen_rows)
display(failopen_df)

reliability_stats = compute_reliability_stats(
    critic_ok_count=15,
    critic_d7a_error_count=3,
    critic_d7b_error_count=1,
    critic_both_error_count=1,
)
display(pd.DataFrame([reliability_stats]))

failopen_checks = [
    check_row("ok path returns CriticResult status ok", ok_result.critic_status == "ok"),
    check_row("D7a exception is fail-open d7a_error", d7a_error_result.critic_status == "d7a_error" and d7a_error_result.d7a_rule_scores is None and d7a_error_result.d7b_llm_scores is not None),
    check_row("D7b exception is fail-open d7b_error", d7b_error_result.critic_status == "d7b_error" and d7b_error_result.d7a_rule_scores is not None and d7b_error_result.d7b_llm_scores is None),
    check_row("both exceptions are fail-open both_error", both_error_result.critic_status == "both_error" and both_error_result.d7a_rule_scores is None and both_error_result.d7b_llm_scores is None),
    check_row("reliability stats count failures but do not enforce fuse", reliability_stats["critic_failure_count"] == 5 and reliability_stats["fuse_enforced"] is False),
    check_row("fuse scaffold never halts in Stage 1", should_fuse_halt(100, 0.95) is False),
]
display_check_table(failopen_checks, "D7 orchestrator fail-open and fuse checks")
D7_ACCEPTANCE["fail_open_fuse"] = True


,case,status,d7a_scores_present,d7b_scores_present
0,ok,ok,True,True
1,D7a raises,d7a_error,False,True
2,D7b raises,d7b_error,True,False
3,both raise,both_error,False,False


,critic_total_count,critic_ok_count,critic_d7a_error_count,critic_d7b_error_count,critic_both_error_count,critic_failure_count,critic_failure_rate,fuse_enforced,tier1_min_k,tier1_failure_rate_threshold,tier2_min_k,tier2_failure_rate_threshold
0,20,15,3,1,1,5,0.25,False,20,0.5,50,0.2


D7 orchestrator fail-open and fuse checks


,check,status,detail
0,ok path returns CriticResult status ok,PASS,
1,D7a exception is fail-open d7a_error,PASS,
2,D7b exception is fail-open d7b_error,PASS,
3,both exceptions are fail-open both_error,PASS,
4,reliability stats count failures but do not en...,PASS,
5,fuse scaffold never halts in Stage 1,PASS,


### BB1. `run_critic()` 200-Input Never-Raises Fuzz Audit

The hand-written fail-open cases prove the named branches. This fuzz cell exercises a broader valid-DSL input surface: factor counts, condition counts, descriptions, themes, batch positions, and prior factor-set histories. The contract is not score quality; it is that `run_critic()` always returns a `CriticResult` and never bubbles exceptions.


In [64]:
import random

fuzz_rng = random.Random(42)
fuzz_backend = StubD7bBackend()
fuzz_themes = list(THEMES)[:5]
fuzz_results = []
fuzz_errors = []

for i in range(200):
    try:
        n_factors = fuzz_rng.randint(1, min(4, len(D7_ALL_FACTORS)))
        factors = fuzz_rng.sample(list(D7_ALL_FACTORS), n_factors)
        n_entry_conditions = fuzz_rng.randint(1, min(4, n_factors))
        entry_conditions = [
            {"factor": f, "op": fuzz_rng.choice(["<", "<=", ">", ">=", "=="]), "value": float(fuzz_rng.randint(1, 100000))}
            for f in factors[:n_entry_conditions]
        ]
        exit_conditions = [{"factor": factors[0], "op": fuzz_rng.choice(["<", "<=", ">", ">=", "=="]), "value": float(fuzz_rng.randint(1, 100000))}]
        dsl = d7_make_dsl(
            name=f"d7_fuzz_{i}",
            description="x" * fuzz_rng.randint(1, 300),
            entry=[{"conditions": entry_conditions}],
            exit_=[{"conditions": exit_conditions}],
            max_hold_bars=fuzz_rng.choice([None, fuzz_rng.randint(1, 720)]),
        )
        prior_sets = tuple(
            tuple(sorted(fuzz_rng.sample(list(D7_ALL_FACTORS), fuzz_rng.randint(1, min(3, len(D7_ALL_FACTORS))))))
            for _ in range(fuzz_rng.randint(0, 10))
        )
        ctx = d7_make_ctx(prior_factor_sets=prior_sets, batch_position=fuzz_rng.randint(1, 200))
        theme = fuzz_rng.choice(fuzz_themes)
        result = run_critic(dsl, theme, ctx, fuzz_backend)
        fuzz_results.append({"i": i, "theme": theme, "n_factors": len(extract_factors(dsl)), "n_conditions": count_conditions(dsl), "prior_factor_sets": len(prior_sets), "status": result.critic_status, "is_critic_result": isinstance(result, CriticResult)})
    except Exception as exc:
        fuzz_errors.append({"i": i, "error_type": type(exc).__name__, "error": str(exc)})

fuzz_df = pd.DataFrame(fuzz_results)
fuzz_error_df = pd.DataFrame(fuzz_errors)
display(fuzz_df.head(10))
print("Fuzz status distribution:")
display(fuzz_df["status"].value_counts().rename_axis("critic_status").reset_index(name="count") if len(fuzz_df) else fuzz_error_df)

fuzz_checks = [
    check_row("200 fuzz inputs completed without exception", len(fuzz_errors) == 0, fuzz_error_df),
    check_row("all fuzz calls returned CriticResult", len(fuzz_df) == 200 and bool(fuzz_df["is_critic_result"].all()), fuzz_df.tail()),
    check_row("all fuzz statuses are known critic statuses", set(fuzz_df["status"]).issubset({"ok", "d7a_error", "d7b_error", "both_error"}), fuzz_df["status"].value_counts().to_dict()),
]
display_check_table(fuzz_checks, "D7 run_critic never-raises fuzz checks")
D7_ACCEPTANCE["never_raises_fuzz"] = True


,i,theme,n_factors,n_conditions,prior_factor_sets,status,is_critic_result
0,0,momentum,1,2,9,ok,True
1,1,volatility_regime,3,4,1,ok,True
2,2,volatility_regime,1,2,5,ok,True
3,3,mean_reversion,3,4,9,ok,True
4,4,volatility_regime,1,2,2,ok,True
5,5,volatility_regime,1,2,10,ok,True
6,6,volume_divergence,1,2,3,ok,True
7,7,volume_divergence,1,2,8,ok,True
8,8,calendar_effect,1,2,2,ok,True
9,9,volatility_regime,1,2,8,ok,True


Fuzz status distribution:


,critic_status,count
0,ok,200


D7 run_critic never-raises fuzz checks


,check,status,detail
0,200 fuzz inputs completed without exception,PASS,Empty DataFrame Columns: [] Index: []
1,all fuzz calls returned CriticResult,PASS,i theme n_factors n_cond...
2,all fuzz statuses are known critic statuses,PASS,{'ok': 200}


## BC. D6 Regression Audit (`--with-critic` Off)

This dry-run uses a deterministic injected proposer backend and a temporary ledger/payload directory. With critic disabled, the Stage 2d summary must retain the signed-off D6 schema: no top-level critic fields and no per-call `critic_result` keys.


In [65]:
class D7SmallVariedBackend:
    """Small deterministic proposer backend for D7 sidecar integration audit."""

    _THEME_FACTOR = {
        "momentum": "return_24h",
        "mean_reversion": "zscore_48",
        "volatility_regime": "atr_14",
        "volume_divergence": "volume_zscore_24h",
        "calendar_effect": "day_of_week",
    }

    def __init__(self, registry):
        self._registry = registry
        self._n = 0
        self.contexts: list[ProposerBatchContext] = []

    def generate(self, context: ProposerBatchContext) -> ProposerOutput:
        self.contexts.append(context)
        self._n += 1
        theme = THEMES[(context.position - 1) % stage2d_mod.THEME_CYCLE_LEN]
        factor = self._THEME_FACTOR[theme]
        entry_op = "==" if factor == "day_of_week" else ">"
        exit_op = "==" if factor == "day_of_week" else "<"
        entry_value = float((self._n % 7) + 1) if factor == "day_of_week" else round(0.001 * self._n, 6)
        exit_value = float(((self._n + 1) % 7) + 1) if factor == "day_of_week" else 0.0
        dsl_dict = {
            "name": f"d7_sidecar_{self._n}",
            "description": f"D7 sidecar integration candidate {self._n} for {theme} using {factor}.",
            "entry": [{"conditions": [{"factor": factor, "op": entry_op, "value": entry_value}]}],
            "exit": [{"conditions": [{"factor": factor, "op": exit_op, "value": exit_value}]}],
            "position_sizing": "full_equity",
            "max_hold_bars": None,
        }
        raw_json = json.dumps(dsl_dict)
        candidate = classify_raw_json(raw_json, registry=self._registry)
        return ProposerOutput(
            candidates=(candidate,),
            cost_estimate_usd=0.01,
            cost_actual_usd=0.01,
            backend_name="d7_acceptance_small",
            telemetry={"input_tokens": 500 + self._n, "output_tokens": 100 + self._n},
        )


def d7_run_small_stage2d(*, with_critic: bool, label: str):
    root = D7_INTEGRATION_ROOT / label
    root.mkdir(parents=True, exist_ok=True)
    ledger_path = root / "ledger.db"
    payload_dir = root / "payloads"
    backend = D7SmallVariedBackend(D7_REGISTRY)
    original_batch_size = stage2d_mod.STAGE2D_BATCH_SIZE
    stage2d_mod.STAGE2D_BATCH_SIZE = 10
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            summary = stage2d_mod.run_stage2d(
                dry_run=True,
                with_critic=with_critic,
                _backend=backend,
                _ledger_path=ledger_path,
                _payload_dir=payload_dir,
            )
    finally:
        stage2d_mod.STAGE2D_BATCH_SIZE = original_batch_size
    return summary, backend.contexts, ledger_path, payload_dir, buffer.getvalue()

D7_INTEGRATION_TMPDIR = tempfile.TemporaryDirectory()
D7_INTEGRATION_ROOT = Path(D7_INTEGRATION_TMPDIR.name)
D7_SUMMARY_OFF, D7_CONTEXTS_OFF, D7_LEDGER_OFF, D7_PAYLOAD_OFF, D7_LOG_OFF = d7_run_small_stage2d(with_critic=False, label="critic_off")

regression_receipt = pd.DataFrame([{
    "mode": "critic_off",
    "batch_id": D7_SUMMARY_OFF["batch_id"],
    "calls": len(D7_SUMMARY_OFF["calls"]),
    "hypotheses_attempted": D7_SUMMARY_OFF["hypotheses_attempted"],
    "parse_rate": D7_SUMMARY_OFF["parse_rate"],
    "total_actual_cost_usd": D7_SUMMARY_OFF["total_actual_cost_usd"],
    "critic_enabled_key_present": "critic_enabled" in D7_SUMMARY_OFF,
    "any_call_has_critic_result": any("critic_result" in c for c in D7_SUMMARY_OFF["calls"]),
}])
display(regression_receipt)

regression_checks = [
    check_row("critic-off dry-run completed 10 calls", len(D7_SUMMARY_OFF["calls"]) == 10),
    check_row("critic-off summary has no critic_enabled key", "critic_enabled" not in D7_SUMMARY_OFF),
    check_row("critic-off summary has no d7b_mode key", "d7b_mode" not in D7_SUMMARY_OFF),
    check_row("critic-off summary has no critic_reliability key", "critic_reliability" not in D7_SUMMARY_OFF),
    check_row("critic-off calls have no critic_result key", not any("critic_result" in c for c in D7_SUMMARY_OFF["calls"])),
    check_row("critic-off lifecycle invariant remains true", D7_SUMMARY_OFF["lifecycle_invariant_ok"] is True),
]
display_check_table(regression_checks, "D7 critic-off D6 regression checks")
D7_ACCEPTANCE["d6_regression_off"] = True


,mode,batch_id,calls,hypotheses_attempted,parse_rate,total_actual_cost_usd,critic_enabled_key_present,any_call_has_critic_result
0,critic_off,b7126dc4-ee44-4ecf-a7c4-210421d7606a,10,10,1.0,0.1,False,False


D7 critic-off D6 regression checks


,check,status,detail
0,critic-off dry-run completed 10 calls,PASS,
1,critic-off summary has no critic_enabled key,PASS,
2,critic-off summary has no d7b_mode key,PASS,
3,critic-off summary has no critic_reliability key,PASS,
4,critic-off calls have no critic_result key,PASS,
5,critic-off lifecycle invariant remains true,PASS,


## BD. D7 Integration Audit (`--with-critic` On)

With critic enabled, D6 behavior must remain unchanged. The allowed differences are only the D7 sidecar fields: top-level `critic_enabled`, `d7b_mode`, `critic_reliability`, and per-call `critic_result` annotations for `pending_backtest` candidates.


In [66]:
D7_SUMMARY_ON, D7_CONTEXTS_ON, D7_LEDGER_ON, D7_PAYLOAD_ON, D7_LOG_ON = d7_run_small_stage2d(with_critic=True, label="critic_on")


def d7_call_trace(summary: dict) -> list[dict]:
    keep = [
        "position",
        "theme",
        "lifecycle_state",
        "valid_status",
        "hypothesis_hash",
        "approved_examples_count_in_prompt_before_call",
        "approved_example_names_before_call",
        "approved_example_logic_notes",
        "factors_used",
        "overlap_count",
        "overlap_ratio",
        "out_of_theme_factor_count",
        "contains_default_momentum_factor",
        "default_momentum_factors_used",
        "actual_cost_usd",
        "estimated_cost_usd",
        "input_tokens",
        "output_tokens",
    ]
    return [{k: c.get(k) for k in keep} for c in summary["calls"]]


def d7_context_trace(contexts: list[ProposerBatchContext]) -> list[dict]:
    return [
        {
            "position": c.position,
            "batch_size": c.batch_size,
            "allowed_factors": c.allowed_factors,
            "allowed_operators": c.allowed_operators,
            "theme_slot": c.theme_slot,
            "budget_remaining": c.budget_remaining,
            "batch_metadata": c.batch_metadata,
        }
        for c in contexts
    ]


def d7_ledger_df(path: Path) -> pd.DataFrame:
    with sqlite3.connect(path) as conn:
        return pd.read_sql_query(
            """
            SELECT api_call_kind, status, estimated_cost, actual_cost
            FROM ledger
            ORDER BY created_at_utc
            """,
            conn,
        )

trace_off = d7_call_trace(D7_SUMMARY_OFF)
trace_on = d7_call_trace(D7_SUMMARY_ON)
context_trace_off = d7_context_trace(D7_CONTEXTS_OFF)
context_trace_on = d7_context_trace(D7_CONTEXTS_ON)
ledger_off_df = d7_ledger_df(D7_LEDGER_OFF)
ledger_on_df = d7_ledger_df(D7_LEDGER_ON)

pending_on = [c for c in D7_SUMMARY_ON["calls"] if c.get("lifecycle_state") == PENDING_BACKTEST]
critic_results_on = [c.get("critic_result") for c in pending_on]

onoff_diff_df = pd.DataFrame([
    {"field_or_behavior": "candidate lifecycle/hash/factor trace", "critic_off": "recorded", "critic_on": "recorded", "expected_relation": "identical", "pass": trace_off == trace_on},
    {"field_or_behavior": "approved_examples names/logic", "critic_off": "recorded", "critic_on": "recorded", "expected_relation": "identical", "pass": [c["approved_example_names_before_call"] for c in trace_off] == [c["approved_example_names_before_call"] for c in trace_on]},
    {"field_or_behavior": "proposer BatchContext excluding batch_id", "critic_off": "captured", "critic_on": "captured", "expected_relation": "identical", "pass": context_trace_off == context_trace_on},
    {"field_or_behavior": "ledger status/cost rows", "critic_off": len(ledger_off_df), "critic_on": len(ledger_on_df), "expected_relation": "identical status/cost shape", "pass": ledger_off_df.equals(ledger_on_df)},
    {"field_or_behavior": "top-level critic fields", "critic_off": "absent", "critic_on": "present", "expected_relation": "differs only when enabled", "pass": all(k not in D7_SUMMARY_OFF for k in ["critic_enabled", "d7b_mode", "critic_reliability"]) and all(k in D7_SUMMARY_ON for k in ["critic_enabled", "d7b_mode", "critic_reliability"])},
    {"field_or_behavior": "per-call critic_result", "critic_off": "absent", "critic_on": "non-null for pending", "expected_relation": "annotation only", "pass": not any("critic_result" in c for c in D7_SUMMARY_OFF["calls"]) and all(cr is not None for cr in critic_results_on)},
])
display(onoff_diff_df)

display(pd.DataFrame([
    {
        "mode": "critic_on",
        "batch_id": D7_SUMMARY_ON["batch_id"],
        "calls": len(D7_SUMMARY_ON["calls"]),
        "critic_enabled": D7_SUMMARY_ON.get("critic_enabled"),
        "d7b_mode": D7_SUMMARY_ON.get("d7b_mode"),
        "critic_reliability": D7_SUMMARY_ON.get("critic_reliability"),
    }
]))

integration_checks = [
    check_row("critic-on dry-run completed 10 calls", len(D7_SUMMARY_ON["calls"]) == 10),
    check_row("critic-on summary enables stub critic", D7_SUMMARY_ON.get("critic_enabled") is True and D7_SUMMARY_ON.get("d7b_mode") == "stub"),
    check_row("D6 call trace is identical with critic on/off", trace_off == trace_on),
    check_row("approved_examples trace is identical with critic on/off", [c["approved_example_names_before_call"] for c in trace_off] == [c["approved_example_names_before_call"] for c in trace_on]),
    check_row("proposer BatchContext is identical with critic on/off", context_trace_off == context_trace_on),
    check_row("ledger status/cost rows are identical with critic on/off", ledger_off_df.equals(ledger_on_df)),
    check_row("all pending candidates receive sidecar CriticResult", len(pending_on) > 0 and all(cr is not None for cr in critic_results_on)),
    check_row("all sidecar results are ok stub results", all(cr["critic_status"] == "ok" and cr["d7b_mode"] == "stub" for cr in critic_results_on)),
    check_row("critic reliability reports no failures and fuse off", D7_SUMMARY_ON["critic_reliability"]["critic_failure_rate"] == 0.0 and D7_SUMMARY_ON["critic_reliability"]["fuse_enforced"] is False),
    check_row("D6 lifecycle invariant remains true with critic enabled", D7_SUMMARY_ON["lifecycle_invariant_ok"] is True),
]
display_check_table(integration_checks, "D7 with-critic sidecar integration checks")
D7_ACCEPTANCE["integration_sidecar_on"] = True


,field_or_behavior,critic_off,critic_on,expected_relation,pass
0,candidate lifecycle/hash/factor trace,recorded,recorded,identical,True
1,approved_examples names/logic,recorded,recorded,identical,True
2,proposer BatchContext excluding batch_id,captured,captured,identical,True
3,ledger status/cost rows,10,10,identical status/cost shape,True
4,top-level critic fields,absent,present,differs only when enabled,True
5,per-call critic_result,absent,non-null for pending,annotation only,True


,mode,batch_id,calls,critic_enabled,d7b_mode,critic_reliability
0,critic_on,1be84ee7-5aad-4c44-a3f3-b20c22cdb85c,10,True,stub,"{'critic_total_count': 10, 'critic_ok_count': ..."


D7 with-critic sidecar integration checks


,check,status,detail
0,critic-on dry-run completed 10 calls,PASS,
1,critic-on summary enables stub critic,PASS,
2,D6 call trace is identical with critic on/off,PASS,
3,approved_examples trace is identical with crit...,PASS,
4,proposer BatchContext is identical with critic...,PASS,
5,ledger status/cost rows are identical with cri...,PASS,
6,all pending candidates receive sidecar CriticR...,PASS,
7,all sidecar results are ok stub results,PASS,
8,critic reliability reports no failures and fus...,PASS,
9,D6 lifecycle invariant remains true with criti...,PASS,


### BD1. Baseline Closure: Golden Artifact Search And Canonical D6 Surface Comparison

The repo does not currently carry a checked-in pre-D7 golden summary artifact. Because full `stage2d_summary.json` files include dynamic fields such as `batch_id`, timestamps, and duration, raw full-file byte identity is not a meaningful notebook assertion.

To close the regression gap without inventing a stale golden file, this cell does two things:

1. Searches for an existing checked-in golden/baseline artifact and surfaces the result.
2. Builds a deterministic canonical JSON surface from D6 behavioral fields only, then proves that surface is byte-identical with critic off vs critic on. The only allowed differences remain the D7 annotation fields.


In [67]:
baseline_artifact_candidates = [
    path for path in sorted(Path(".").glob("**/*"))
    if path.is_file()
    and any(token in path.name.lower() for token in ["golden", "baseline", "pre_d7", "pred7", "byte_identical"])
    and ".git" not in path.parts
    and "__pycache__" not in path.parts
    and path.suffix.lower() in {".json", ".txt", ".md"}
]

pre_d7_expected_keys = {
    "config", "batch_id", "dry_run", "batch_status", "hypotheses_attempted", "unissued_slots", "truncated", "truncated_at_call", "early_stop_reason", "early_stop_detail", "lifecycle_counts", "lifecycle_invariant_ok", "parse_rate", "cardinality_distribution", "cardinality_violation_count", "total_estimated_cost_usd", "total_actual_cost_usd", "cost_ratio", "cumulative_monthly_spend_usd", "batch_duration_seconds", "factor_usage", "error_breakdown", "per_theme", "total_valid_count", "distinct_hash_count", "valid_with_empty_factor_set_count", "valid_with_empty_factor_set_calls", "distinct_factor_set_count", "unique_factor_set_ratio", "repeated_factor_sets", "block_trends", "interim_snapshots", "anomaly_flags", "approved_examples_trace", "calls",
}

critic_off_keys = set(D7_SUMMARY_OFF.keys())
critic_on_stripped = {k: v for k, v in D7_SUMMARY_ON.items() if k not in {"critic_enabled", "d7b_mode", "critic_reliability"}}

TOP_LEVEL_D6_COMPARE_KEYS = ["dry_run", "batch_status", "hypotheses_attempted", "unissued_slots", "truncated", "truncated_at_call", "early_stop_reason", "early_stop_detail", "lifecycle_counts", "lifecycle_invariant_ok", "parse_rate", "cardinality_distribution", "cardinality_violation_count", "total_estimated_cost_usd", "total_actual_cost_usd", "cost_ratio", "factor_usage", "error_breakdown", "per_theme", "total_valid_count", "distinct_hash_count", "valid_with_empty_factor_set_count", "valid_with_empty_factor_set_calls", "distinct_factor_set_count", "unique_factor_set_ratio", "repeated_factor_sets", "anomaly_flags", "approved_examples_trace"]
CALL_D6_COMPARE_KEYS = ["position", "theme", "truncated_by_cap", "lifecycle_state", "valid_status", "hypothesis_hash", "estimated_cost_usd", "actual_cost_usd", "approved_examples_count_in_prompt_before_call", "approved_example_names_before_call", "approved_example_logic_notes", "cardinality", "retry_count", "error_info", "factors_used", "overlap_count", "overlap_ratio", "out_of_theme_factor_count", "contains_default_momentum_factor", "default_momentum_factors_used", "input_tokens", "output_tokens"]

def d7_d6_behavior_surface(summary: dict) -> dict:
    return {
        "top_level": {k: summary.get(k) for k in TOP_LEVEL_D6_COMPARE_KEYS},
        "calls": [{k: call.get(k) for k in CALL_D6_COMPARE_KEYS} for call in summary["calls"]],
    }

surface_off = d7_d6_behavior_surface(D7_SUMMARY_OFF)
surface_on = d7_d6_behavior_surface(D7_SUMMARY_ON)
surface_off_json = json.dumps(surface_off, sort_keys=True, separators=(",", ":"), default=str)
surface_on_json = json.dumps(surface_on, sort_keys=True, separators=(",", ":"), default=str)

baseline_closure_df = pd.DataFrame([
    {"item": "checked-in baseline/golden artifacts found", "value": len(baseline_artifact_candidates), "detail": [str(p) for p in baseline_artifact_candidates[:10]]},
    {"item": "critic-off top-level schema extra keys", "value": sorted(critic_off_keys - pre_d7_expected_keys), "detail": "must be empty"},
    {"item": "critic-off top-level schema missing keys", "value": sorted(pre_d7_expected_keys - critic_off_keys), "detail": "must be empty"},
    {"item": "canonical D6 surface byte length off", "value": len(surface_off_json), "detail": "dynamic envelope removed"},
    {"item": "canonical D6 surface byte length on", "value": len(surface_on_json), "detail": "dynamic envelope removed"},
])
display(baseline_closure_df)

baseline_closure_checks = [
    check_row("no checked-in pre-D7 golden summary artifact is present", len(baseline_artifact_candidates) == 0, [str(p) for p in baseline_artifact_candidates[:10]]),
    check_row("critic-off summary exactly matches pre-D7 top-level schema", critic_off_keys == pre_d7_expected_keys, {"extra": sorted(critic_off_keys - pre_d7_expected_keys), "missing": sorted(pre_d7_expected_keys - critic_off_keys)}),
    check_row("critic-on stripped top-level D6 keys include the pre-D7 schema", set(critic_on_stripped.keys()) >= pre_d7_expected_keys),
    check_row("canonical D6 behavior surface is byte-identical off vs on", surface_off_json == surface_on_json),
    check_row("allowed D7 top-level annotation fields appear only when critic is on", all(k not in D7_SUMMARY_OFF for k in ["critic_enabled", "d7b_mode", "critic_reliability"]) and all(k in D7_SUMMARY_ON for k in ["critic_enabled", "d7b_mode", "critic_reliability"])),
]
display_check_table(baseline_closure_checks, "D7 D6-baseline closure checks")
D7_ACCEPTANCE["baseline_closure"] = True


,item,value,detail
0,checked-in baseline/golden artifacts found,0,[]
1,critic-off top-level schema extra keys,[],must be empty
2,critic-off top-level schema missing keys,[],must be empty
3,canonical D6 surface byte length off,14078,dynamic envelope removed
4,canonical D6 surface byte length on,14078,dynamic envelope removed


D7 D6-baseline closure checks


,check,status,detail
0,no checked-in pre-D7 golden summary artifact i...,PASS,[]
1,critic-off summary exactly matches pre-D7 top-...,PASS,"{'extra': [], 'missing': []}"
2,critic-on stripped top-level D6 keys include t...,PASS,
3,canonical D6 behavior surface is byte-identica...,PASS,
4,allowed D7 top-level annotation fields appear ...,PASS,


## BE. D7 Stage 1 Verdict

D7 Stage 1 is accepted only if the critic surface is schema-stable, deterministic D7a rules are independently reproducible, D7b is obviously stubbed, the orchestrator fails open, and D6 behavior is unchanged when critic is off or differs only by sidecar annotations when critic is on.

Open questions intentionally deferred to Stage 2:

- live D7b prompt wording and leakage audit
- semantic score stability methodology
- critic reliability fuse thresholds
- whether `structural_variant_risk` correlates with real repeated-factor patterns under live D7b
- how D8 should consume D7a/D7b scores without mixing normal-polarity axes and the reverse-polarity `structural_variant_risk`


In [68]:
d7_gate_rows = [
    {"gate": "A. Scope contract framed", "pass": D7_ACCEPTANCE.get("scope_contract", False)},
    {"gate": "B. Schema and artifact surface", "pass": D7_ACCEPTANCE.get("schema_surface", False)},
    {"gate": "C. Feature extraction primitives", "pass": D7_ACCEPTANCE.get("feature_extraction", False)},
    {"gate": "D. D7a rules and flags", "pass": D7_ACCEPTANCE.get("d7a_rules", False)},
    {"gate": "E. Complexity edge table pinned", "pass": D7_ACCEPTANCE.get("complexity_edge_table", False)},
    {"gate": "F. Expanded flag coverage", "pass": D7_ACCEPTANCE.get("expanded_flag_coverage", False)},
    {"gate": "G. D7b stub deterministic/non-live", "pass": D7_ACCEPTANCE.get("d7b_stub", False)},
    {"gate": "H. run_critic fail-open and fuse scaffold", "pass": D7_ACCEPTANCE.get("fail_open_fuse", False)},
    {"gate": "I. 200-input never-raises fuzz", "pass": D7_ACCEPTANCE.get("never_raises_fuzz", False)},
    {"gate": "J. D6 regression with critic off", "pass": D7_ACCEPTANCE.get("d6_regression_off", False)},
    {"gate": "K. D7 sidecar annotations only with critic on", "pass": D7_ACCEPTANCE.get("integration_sidecar_on", False)},
    {"gate": "L. D6 baseline closure", "pass": D7_ACCEPTANCE.get("baseline_closure", False)},
]
d7_verdict_df = pd.DataFrame(d7_gate_rows)
d7_verdict_df["status"] = d7_verdict_df["pass"].map(lambda ok: "PASS" if ok else "FAIL")
display(d7_verdict_df[["gate", "status"]])

assert d7_verdict_df["pass"].all(), d7_verdict_df[~d7_verdict_df["pass"]]
print("D7 Stage 1 verdict: PASS. The critic is accepted as a deterministic fail-open sidecar that preserves D6 baseline behavior.")


,gate,status
0,A. Scope contract framed,PASS
1,B. Schema and artifact surface,PASS
2,C. Feature extraction primitives,PASS
3,D. D7a rules and flags,PASS
4,E. Complexity edge table pinned,PASS
5,F. Expanded flag coverage,PASS
6,G. D7b stub deterministic/non-live,PASS
7,H. run_critic fail-open and fuse scaffold,PASS
8,I. 200-input never-raises fuzz,PASS
9,J. D6 regression with critic off,PASS


D7 Stage 1 verdict: PASS. The critic is accepted as a deterministic fail-open sidecar that preserves D6 baseline behavior.
